# Wine Peer
## Advanced Machine Learning — Final Project

---

### What this project does

Wine Peer is a food-to-wine recommendation system. The user photographs their meal — that is the only input. The system identifies the dish, maps its flavor profile into a wine flavor space using Word2Vec embeddings, and returns three wine recommendations, each with a genuine tasting note from a Vivino user and a Vivino approval rating.

**Full pipeline — one photo to three wine recommendations:**

```
Food photo
  ↓
ResNet-50 CNN — identifies the dish from 101 Food-101 classes
  ↓
Food flavor table — maps the dish to three sets of taste keywords
  ↓
Word2Vec (fine-tuned on 824k Vivino reviews)
  — embeds keywords into a 300-d flavor space
  — computes cosine similarity against 15 grape-variety centroids
  ↓
Three grape recommendations  (Safe Bet · Bold Move · Hidden Gem)
  ↓
BiLSTM encoder — selects a representative Vivino tasting note per grape (median confidence on test set)
df_wine — selects the highest-rated wine bottle per grape
  ↓
Recommendation card: grape variety · wine name · drinker quote · approval %
```

---

### Example output

| | |
|---|---|
| **Input** | Photo of pizza margherita |
| **CNN output** | Pizza (94% confidence) |
| **Flavor keywords** | savory · cheesy · tomato · herbaceous · rich · fatty |

| Pairing | Grape | Wine | Drinker Quote | Vivino |
|---|---|---|---|---|
| **Safe Bet** — echoes the food's dominant flavors | Sangiovese | Chianti Classico Riserva 2019 | *"Deep cherry and dried herbs — wraps around the tomato sauce like it was made for it."* | 92% |
| **Bold Move** — cuts through richness, refreshes the palate | Chardonnay | Chablis Premier Cru 2021 | *"Bone-dry mineral acidity cuts straight through the richness. Resets every bite."* | 89% |
| **Hidden Gem** — crowd-pleasing, won't clash | Pinot Grigio | Santa Margherita Alto Adige 2022 | *"Light, clean and gently fruited. Gets out of the way and lets the pizza do the talking."* | 86% |

---

### Models and components

| Component | Dataset | Task |
|---|---|---|
| CNN from scratch | Food-101 (101k images) | 101-class food classification (baseline) |
| CNN ResNet-50 | Food-101 | 101-class food classification (transfer learning) |
| LSTM baseline | WineSensed (824k reviews) | 15-class grape variety classification |
| BiLSTM + Bahdanau attention | WineSensed | 15-class grape classification; also used for tasting note retrieval |
| DistilBERT *(+3 pts bonus)* | WineSensed | 15-class grape classification (Transformer comparison) |
| Word2Vec (fine-tuned) | Google News (base) + WineSensed | Flavor embedding — maps food keywords and grape reviews into a shared 300-d vector space |
| Joint model *(+10 pts bonus)* | Food-101 + WineSensed | Learned food→wine projection (ResNet embeddings → grape space) |

---

### Pairing logic

Three pairing intents correspond to three different flavor strategies:

| Intent key | Card label | Strategy | Food flavor keywords (pizza example) |
|---|---|---|---|
| `classic` | **Safe Bet** | Echo and amplify the food's dominant flavors | `savory`, `cheesy`, `tomato`, `herbaceous` |
| `contrast` | **Bold Move** | Cut through richness and refresh the palate | `mineral`, `crisp`, `acidic`, `citrus` |
| `safe_bet` | **Hidden Gem** | Approachable, crowd-pleasing, won't clash | `light`, `fruity`, `soft`, `clean` |

Keywords are stored in `data/food_flavor_table.json` — 101 Food-101 classes × 3 keyword sets. At inference, each set is embedded via Word2Vec and matched to the nearest of 15 grape-variety centroids by cosine similarity. No rules are hardcoded; the match is driven entirely by the learned vector space.

---

### Why grape variety (not wine type)?

Classifying by grape variety rather than broad type (Red / White / Rosé) makes the task harder and the output more useful:

- The model must learn the **fine-grained flavor vocabulary** specific to each grape — *"cassis and cedar"* for Cabernet Sauvignon vs *"strawberry and forest floor"* for Pinot Noir.
- The output is more informative: *"you would enjoy a Sangiovese"* is actionable at a wine shop; *"you would enjoy a Red"* is not.
- Word2Vec places grape varieties in a **continuous flavor space**: Syrah and Primitivo cluster near each other; Riesling sits far from Malbec. This geometry is what makes cosine similarity a meaningful pairing signal.

**The 15 grape classes** cover ~85% of all WineSensed reviews:

| Reds | Whites |
|---|---|
| Cabernet Sauvignon · Merlot · Pinot Noir · Shiraz/Syrah · Malbec · Sangiovese · Tempranillo · Barbera · Primitivo · Cabernet Franc · Tinta Roriz | Chardonnay · Riesling · Pinot Grigio · Monastrell |

---

### Datasets

| Dataset | Size | Source | Used for |
|---|---|---|---|
| Food-101 | 101,000 images · 101 classes | `torchvision.datasets.Food101` | CNN training and evaluation |
| WineSensed (NeurIPS 2023) | 824,000 reviews | `Dakhoo/L2T-NeurIPS-2023` on Hugging Face | LSTM/BiLSTM training · Word2Vec fine-tuning · tasting note retrieval |
| Food flavor table | 101 entries × 3 keyword sets | `data/food_flavor_table.json` (curated) | Word2Vec pairing bridge |

---

### Development phases

| Phase | Environment | Sections | GPU required |
|---|---|---|---|
| **Phase 1** | Local / VS Code | 1 – 5, 7, 11 | No |
| **Phase 2** | Google Colab | 6, 8, 9 | **Yes** |
| **Phase 3** | Local / VS Code | 10, 12, 13 | No |
| **Phase 4** | Google Colab / HF Spaces | 14, 15 | Yes (training) |

---

### Notebook structure

| Section | Status | Content |
|---|---|---|
| 1 | ✅ Done | Environment setup — packages, imports, storage helpers |
| 2 | ✅ Done | Raw data loading — Food-101, WineSensed |
| 3 | ✅ Done | Data cleaning — WineSensed, top-15 grape selection, food flavor table |
| 4 | ✅ Done | Train / val / test split — both datasets, stratified, before EDA |
| 5 | ✅ Done | EDA — image and text datasets |
| 6 | ✅ Done | Image preprocessing and DataLoaders |
| 7 | ✅ Done | Text preprocessing — tokenisation, vocab, padding, GloVe, class weights, Word2Vec (7.6), grape centroids (7.7), DataLoaders |
| 8 | ✅ Done | CNN from scratch — architecture, training, evaluation |
| 9 | ✅ Done | CNN ResNet-50 — two-phase fine-tuning, evaluation, comparison |
| 10 | ✅ Done | CNN explainability — Grad-CAM |
| 11 | ✅ Done | RNN/LSTM — LSTM baseline, BiLSTM + attention, DistilBERT *(+3 pts)*, 3-model comparison |
| 12 | ✅ Done | Save all models — 5 models, Drive sync |
| 13 | ✅ Done | Business integration — `recommend()` pipeline, 20-example table |
| 14 | ⬜ To do | Joint model — learned food→wine projection *(+10 pts)* |
| 15 | ⬜ To do | Deployment — Gradio app |
| 16 | ⬜ To do | Business framing, ethics, team contributions |

---


## Section 1 — Environment Setup

This section prepares the execution environment before any data is loaded or models are trained. It must be run in full at the start of every Colab session.

**Four steps:**
1. **Environment check** — confirm Colab + CUDA availability before installing packages
2. **pip installs** — install and version-pin all required libraries (PyTorch, gensim, transformers, etc.)
3. **Library imports** — import all packages, set the random seed for reproducibility, detect hardware
4. **Storage helpers** — define `save_checkpoint()` / `load_checkpoint()` / `save_figure()` that transparently handle the Colab VM → Google Drive save flow

> **Always re-run Section 1 completely when starting a fresh Colab session.** Skipping any cell will cause import errors or missing helpers in later sections.


In [1]:

import sys, torch

IN_COLAB = "google.colab" in sys.modules

# ── CUDA health check ─────────────────────────────────────────────────────────
# torch.cuda.is_available() can return True yet the GPU kernel image may not
# match the installed PyTorch build (AcceleratorError: no kernel image for device).
# We probe with a small operation and fall back to CPU if it fails.
_cuda_ok = False
if torch.cuda.is_available():
    try:
        torch.zeros(1).cuda()
        _cuda_ok = True
    except Exception as _e:
        print(f"⚠  CUDA reported available but failed health check: {_e}")
        print("   Falling back to CPU. For GPU support reinstall PyTorch with the")
        print("   correct CUDA version: Runtime → Factory reset → reinstall torch.")

DEVICE = torch.device("cuda" if _cuda_ok else "cpu")

print(f"{'Environment':<20} {'Google Colab' if IN_COLAB else 'Local'}")
print(f"{'Device':<20} {DEVICE}")
print(f"{'CUDA available':<20} {torch.cuda.is_available()}  (healthy: {_cuda_ok})")

if not IN_COLAB:
    print("\n⚠  Not running in Google Colab — GPU sections (5, 7, 8, 13) require Colab.")
elif not _cuda_ok:
    print("\n⚠  CUDA not healthy — enable the T4 GPU runtime: Runtime → Change runtime type → T4 GPU.")
else:
    print("\n✓ Environment check passed — Colab + CUDA confirmed.")


Environment          Google Colab
Device               cuda
CUDA available       True  (healthy: True)

✓ Environment check passed — Colab + CUDA confirmed.


### 1.1 — Package installation

Installs all required libraries and pins them to specific versions to guarantee reproducibility across Colab sessions and local environments.

> ⚠️ **Expected behaviour — the kernel will crash and restart when you run this cell.**
> This is intentional and required, not an error.
> Python cannot hot-reload an upgraded package into a running process, so the cell deliberately triggers a kernel restart after any install or upgrade.
> **After the restart, continue from Section 1.2 — do not re-run Section 1.1.**
> If all packages are already at the correct versions, no restart occurs and the cell exits in under 5 seconds.

**Design decisions:**
- **Version pinning** (`gensim==4.3.3`, `transformers>=4.40`) prevents silent breakage from upstream API changes — gensim's Word2Vec API changed significantly between versions 3 and 4.
- **Child-process probe for transformers** — a broken transformers install can crash the kernel (segfault), not just raise `ImportError`. We test it in a subprocess so a failure is caught cleanly.
- **Automatic kernel restart** — if any package was installed or upgraded, the kernel restarts immediately. Python cannot hot-reload packages into a running process; skipping the restart causes silent import of the old version.
- **Environment detection** — PyTorch is installed with the CUDA 11.8 wheel on Colab and the CPU wheel locally, saving several hundred MB on local machines.


In [1]:
import subprocess, sys, importlib.util, importlib.metadata

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

def installed_version(dist_name):
    try:
        return importlib.metadata.version(dist_name)
    except importlib.metadata.PackageNotFoundError:
        return None

# ── Detect environment ────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

_anything_installed = False   # only True if something actually changed on disk

# ── Install PyTorch with correct backend ────────────────────────────────
if is_installed("torch"):
    import torch
    print(f"  OK  torch {torch.__version__}")
else:
    print("  INSTALLING  torch ...")
    if IN_COLAB:
        pip_install("torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cu118")
    else:
        pip_install("torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cpu")
    print("  DONE  torch")
    _anything_installed = True

# ── Pinned packages — only install/upgrade when the version is wrong ──────────
# Python can't reload packages into a running process; any real install requires
# a kernel restart. We avoid spurious restarts by checking the version first.
_PINNED = {
    "gensim":   ("gensim",   "4.3.3"),
    "datasets": ("datasets", "2.20.0"),
}

for _import_name, (_dist_name, _target_ver) in _PINNED.items():
    _cur = installed_version(_dist_name)
    if _cur == _target_ver:
        print(f"  OK  {_dist_name} {_cur}")
    else:
        print(f"  INSTALLING  {_dist_name}=={_target_ver}  (current: {_cur}) ...")
        pip_install(f"{_dist_name}=={_target_ver}")
        print(f"  DONE  {_dist_name}=={_target_ver}")
        _anything_installed = True

# ── Early restart if pinned packages changed ────────────────────────────────
# Restart now so the transformers probe always runs on a stable, clean kernel.
if _anything_installed:
    print("\nPinned packages changed — restarting kernel before continuing ...")
    if IN_COLAB:
        import os, signal; os.kill(os.getpid(), signal.SIGKILL)
    else:
        import IPython; ip = IPython.get_ipython()
        if ip: ip.ask_exit()

# ── Install remaining packages ────────────────────────────────────────────
PACKAGES = {
    "nltk":       "nltk",
    "torchcam":   "torchcam",
    "wordcloud":  "wordcloud",
    "matplotlib": "matplotlib",
    "seaborn":    "seaborn",
    "pandas":     "pandas",
    "sklearn":    "scikit-learn",
    "PIL":        "Pillow",
    "ipywidgets": "ipywidgets",
}

for pkg, install_name in PACKAGES.items():
    if is_installed(pkg):
        print(f"  OK  {pkg}")
    else:
        print(f"  INSTALLING  {pkg} ...")
        pip_install(install_name)
        print(f"  DONE  {pkg}")
        _anything_installed = True

# ── Transformers ecosystem (DistilBERT bonus, Section 11.10–11.12) ────────────
# Probe in a *child process* so a broken pre-installed transformers
# (which can crash/segfault, not just raise ImportError) cannot kill this kernel.
_tf_ok = subprocess.call(
    [sys.executable, "-c",
     "from transformers import DistilBertTokenizerFast, DistilBertModel"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
) == 0

if _tf_ok:
    _tf_ver = installed_version("transformers") or "unknown"
    print(f"  OK  transformers {_tf_ver}")
else:
    print("  INSTALLING  transformers ecosystem (import probe failed) ...")
    pip_install(
        "transformers>=4.40.0,<5.0",
        "tokenizers>=0.19,<1.0",
        "huggingface-hub>=0.23,<1.0",
        "safetensors>=0.4",
    )
    print("  DONE  transformers ecosystem")
    _anything_installed = True

# ── Auto-restart kernel if anything was installed ─────────────────────────────
if _anything_installed:
    print("\nPackages installed/upgraded — restarting kernel automatically ...")
    if IN_COLAB:
        import os, signal
        os.kill(os.getpid(), signal.SIGKILL)
    else:
        import IPython
        ip = IPython.get_ipython()
        if ip is not None:
            ip.ask_exit()
        else:
            print("  \u26a0  Could not auto-restart. Please restart the kernel manually.")
else:
    print("\nAll packages already at required versions — no restart needed.")
    print("Continue to the next cell.")


Environment: Google Colab
  OK  torch 2.10.0+cu128
  OK  gensim 4.3.3
  OK  datasets 2.20.0
  OK  nltk
  OK  torchcam
  OK  wordcloud
  OK  matplotlib
  OK  seaborn
  OK  pandas
  OK  sklearn
  OK  PIL
  OK  ipywidgets
  OK  transformers 5.0.0

All packages already at required versions — no restart needed.
Continue to the next cell.


### 1.2 — Library imports and global configuration

Imports all packages used across the notebook and sets three global constants that every subsequent section depends on:

| Constant | Value | Purpose |
|---|---|---|
| `SEED` | `42` | Fixed random seed — passed to `random`, `numpy`, `torch`, and all `train_test_split` calls |
| `DEVICE` | `cuda` or `cpu` | Auto-detected — all tensors and models are moved here |
| `IN_COLAB` | `True` / `False` | Controls Drive paths, display behavior, and install logic |

**Reproducibility:** `SEED = 42` is applied to `random.seed()`, `np.random.seed()`, `torch.manual_seed()`, and `torch.cuda.manual_seed_all()`. This ensures that weight initialisation, data shuffling, and train/val/test splits produce identical results across runs.

**Project directories** (`weights/`, `figures/`, `deployment/`, `data/`) are created here if they do not exist. On Colab these are temporary VM folders; the permanent copies live on Google Drive (set up in Section 1.3).


In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import random
import importlib
from pathlib import Path
from collections import Counter

# Suppress HF Hub symlinks warning on Windows (fallback to copies, still works)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# ── Numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from wordcloud import WordCloud
from PIL import Image

# ── PyTorch core ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Computer vision ───────────────────────────────────────────────────────────
import torchvision.transforms as T
import torchvision.models as models
import torchvision.datasets as tv_datasets

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize

# ── Word2Vec (Google News pre-trained + fine-tune on WineSensed reviews) ──────
import gensim
import gensim.downloader as gensim_api
from gensim.models import Word2Vec, KeyedVectors

# ── Transformers ────────────────────────────────────────────────────────────────────────────────
from transformers import DistilBertTokenizerFast, DistilBertModel

# ── Sklearn utilities ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

# ── Environment ───────────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Project directories ───────────────────────────────────────────────────────
BASE_DIR = Path(".")
WEIGHTS  = BASE_DIR / "weights"
FIGURES  = BASE_DIR / "figures"
DEPLOY   = BASE_DIR / "deployment"
DATA_DIR = BASE_DIR / "data"

for d in [WEIGHTS, FIGURES, DEPLOY, DATA_DIR]:
    d.mkdir(exist_ok=True)

# ── NLTK data ─────────────────────────────────────────────────────────────────
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

# ── Library versions ──────────────────────────────────────────────────────────
libs = [
    "torch", "torchvision", "numpy", "pandas",
    "matplotlib", "seaborn", "PIL", "sklearn",
    "nltk", "wordcloud", "gensim",
    "transformers",
]

print(f"{'Library':<20} {'Version'}")
print("-" * 35)
for lib in libs:
    try:
        mod = importlib.import_module(lib)
        print(f"{lib:<20} {getattr(mod, '__version__', 'n/a')}")
    except ImportError:
        print(f"{lib:<20} NOT INSTALLED")

print()
print(f"{'Environment':<20} {'Google Colab' if IN_COLAB else 'Local'}")
print(f"{'Device':<20} {DEVICE}")
print(f"{'CUDA available':<20} {torch.cuda.is_available()}")
print(f"{'Directories':<20} {[str(d) for d in [WEIGHTS, FIGURES, DEPLOY, DATA_DIR]]}")
print("\nSection 1 complete.")


Library              Version
-----------------------------------
torch                2.10.0+cu128
torchvision          0.25.0+cu128
numpy                1.26.4
pandas               2.2.2
matplotlib           3.10.0
seaborn              0.13.2
PIL                  11.3.0
sklearn              1.6.1
nltk                 3.9.1
wordcloud            1.9.6
gensim               4.3.3
transformers         5.0.0

Environment          Google Colab
Device               cuda
CUDA available       True
Directories          ['weights', 'figures', 'deployment', 'data']

Section 1 complete.


### 1.3 — Storage setup: Colab VM → Google Drive save flow

**The problem:** Google Colab's `/content/` filesystem is ephemeral — it is wiped when the runtime disconnects or times out. Training a ResNet-50 for 15 epochs takes ~30 minutes on a T4; losing that without a backup would mean retraining from scratch.

**The solution:** a three-step save helper that writes to both the fast VM disk and Google Drive:

```
Step 1 → Save to /content/weights/           (fast, temporary — wiped on disconnect)
Step 2 → Copy  to /MyDrive/wine-dine/weights/  (permanent — survives all disconnects)
Step 3 → Verify the Drive copy is non-empty    (catches silent network write failures)
```

If Step 2 or 3 fails, a clear warning is printed and the local copy is preserved.

**Helper functions defined in the code cell below:**

| Function | Purpose |
|---|---|
| `save_checkpoint(state, filename)` | Save any dict (model weights, optimiser state, epoch, metrics) as a `.pt` file using the 3-step flow |
| `load_checkpoint(filename, device)` | Load a `.pt` file — tries the Colab VM first (fast), falls back to Drive if not found |
| `save_figure(fig, filename)` | Save a matplotlib figure to `figures/` locally and Drive |

On a local machine (VS Code / CPU), all saves go directly to `./weights/` and `./figures/` — no Drive is mounted.

> **Run this cell before any training cell.** After a Colab disconnect, re-run all of Section 1, then load the most recent checkpoint with `load_checkpoint()` to resume.


In [3]:
# ── 1.3  Storage Setup ────────────────────────────────────────────────────────
import shutil

# ── Paths ──────────────────────────────────────────────────────────────────────
# LOCAL_WEIGHTS / LOCAL_FIGURES always point to the local folder on this machine.
# WEIGHTS_DIR  / FIGURES_DIR   point to Drive on Colab, local folder otherwise.
LOCAL_WEIGHTS = WEIGHTS   # Path object defined in Section 1.2
LOCAL_FIGURES = FIGURES   # Path object defined in Section 1.2

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR   = '/content/drive/MyDrive/wine-dine'
    WEIGHTS_DIR = f'{DRIVE_DIR}/weights'
    FIGURES_DIR = f'{DRIVE_DIR}/figures'
    for d in [DRIVE_DIR, WEIGHTS_DIR, FIGURES_DIR]:
        os.makedirs(d, exist_ok=True)
    print("✓ Google Drive mounted")
    print(f"  Permanent weights → {WEIGHTS_DIR}")
    print(f"  Permanent figures → {FIGURES_DIR}")
else:
    WEIGHTS_DIR = str(WEIGHTS)
    FIGURES_DIR = str(FIGURES)
    print("✓ Local mode — no Drive needed")
    print(f"  Weights → {WEIGHTS_DIR}")
    print(f"  Figures → {FIGURES_DIR}")


# ─────────────────────────────────────────────────────────────────────────────
# save_checkpoint  — safe 3-step model save
# ─────────────────────────────────────────────────────────────────────────────
def save_checkpoint(state, filename):
    """
    Safely save a PyTorch model checkpoint.

    On Colab  (3 steps):
      1. Save to /content/weights/<filename>   ← fast write, temporary (wiped on disconnect)
      2. Copy  to Google Drive/<filename>       ← permanent, survives disconnect
      3. Verify the Drive copy is not empty     ← catches silent failures

    Locally:
      Saves directly to ./weights/<filename>

    Usage:
        save_checkpoint({
            "epoch":           epoch,
            "model_state":     model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss":        val_loss,
        }, "my_model_ep01.pt")

        save_checkpoint(model.state_dict(), "my_model_best.pt")
    """
    local_path = os.path.join(str(LOCAL_WEIGHTS), filename)

    # ── Step 1: save locally (always fast and reliable) ───────────────────────
    torch.save(state, local_path)

    if IN_COLAB:
        drive_path = os.path.join(WEIGHTS_DIR, filename)

        # ── Step 2: copy to Drive ─────────────────────────────────────────────
        try:
            shutil.copy2(local_path, drive_path)
        except Exception as e:
            print(f"  ⚠️  Drive copy FAILED for '{filename}': {e}")
            print(f"  ⚠️  Colab VM copy is still safe at: {local_path}  (temporary — will be lost on disconnect!)")
            return

        # ── Step 3: verify Drive copy is not empty ────────────────────────────
        if os.path.getsize(drive_path) == 0:
            print(f"  ⚠️  Drive copy of '{filename}' is empty — something went wrong.")
            print(f"  ⚠️  Colab VM copy is still safe at: {local_path}  (temporary — will be lost on disconnect!)")
            return

        print(f"  ✓ Saved on Colab VM (temporary):  {local_path}")
        print(f"  ✓ Copied to Drive   (permanent):  {drive_path}")
    else:
        print(f"  ✓ Checkpoint saved (local machine): {local_path}")


# ─────────────────────────────────────────────────────────────────────────────
# load_checkpoint  — safe load (Colab VM first, Drive fallback)
# ─────────────────────────────────────────────────────────────────────────────
def load_checkpoint(filename, device=None):
    """
    Safely load a saved checkpoint.

    Tries Colab VM first (works within the same Colab session — fastest).
    If not found on VM, tries Google Drive (works after starting a new session).

    Usage:
        # Full checkpoint (epoch + optimizer + val_loss):
        ckpt = load_checkpoint("my_model_ep11.pt")
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])

        # Best-model checkpoint (state dict only):
        model.load_state_dict(load_checkpoint("my_model_best.pt"))
    """
    if device is None:
        device = DEVICE

    local_path = os.path.join(str(LOCAL_WEIGHTS), filename)

    if os.path.exists(local_path):
        print(f"  → Loading from Colab VM (temporary): {local_path}")
        return torch.load(local_path, map_location=device, weights_only=False)

    if IN_COLAB:
        drive_path = os.path.join(WEIGHTS_DIR, filename)
        if os.path.exists(drive_path):
            print(f"  → Loading from Drive (permanent): {drive_path}")
            return torch.load(drive_path, map_location=device, weights_only=False)
        raise FileNotFoundError(
            f"\n❌ Checkpoint '{filename}' not found!\n"
            f"   Checked Colab VM: {local_path}\n"
            f"   Checked Drive:    {drive_path}\n"
            f"   → Did you train the model? If resuming, make sure Drive is mounted (re-run Section 1.3)."
        )

    raise FileNotFoundError(
        f"\n❌ Checkpoint '{filename}' not found at: {local_path}\n"
        f"   → Did you run the training cell?"
    )


# ─────────────────────────────────────────────────────────────────────────────
# save_figure  — safe matplotlib figure save
# ─────────────────────────────────────────────────────────────────────────────
def save_figure(fig, filename, dpi=100):
    """
    Safely save a matplotlib figure.

    On Colab  (3 steps):
      1. Save to /content/figures/<filename>   ← fast write, temporary (wiped on disconnect)
      2. Copy  to Google Drive/<filename>       ← permanent, survives disconnect
      3. Verify the Drive copy is not empty

    Locally:
      Saves directly to ./figures/<filename>

    Usage:
        fig, ax = plt.subplots()
        ax.plot(...)
        save_figure(fig, "my_plot.png")
        plt.show()
    """
    local_path = os.path.join(str(LOCAL_FIGURES), filename)

    # ── Step 1: save locally ──────────────────────────────────────────────────
    fig.savefig(local_path, dpi=dpi, bbox_inches="tight")

    if IN_COLAB:
        drive_path = os.path.join(FIGURES_DIR, filename)

        # ── Step 2: copy to Drive ─────────────────────────────────────────────
        try:
            shutil.copy2(local_path, drive_path)
        except Exception as e:
            print(f"  ⚠️  Drive copy FAILED for '{filename}': {e}")
            print(f"  ⚠️  Colab VM copy is still safe at: {local_path}  (temporary — will be lost on disconnect!)")
            return

        # ── Step 3: verify ────────────────────────────────────────────────────
        if os.path.getsize(drive_path) == 0:
            print(f"  ⚠️  Drive copy of '{filename}' is empty — something went wrong.")
            print(f"  ⚠️  Colab VM copy is still safe at: {local_path}  (temporary — will be lost on disconnect!)")
            return

        print(f"  ✓ Figure saved on Colab VM (temporary):  {local_path}")
        print(f"  ✓ Figure copied to Drive   (permanent):  {drive_path}")
    else:
        print(f"  ✓ Figure saved (local machine): {local_path}")


print("✓ Storage helpers ready:  save_checkpoint | load_checkpoint | save_figure")


# ─────────────────────────────────────────────────────────────────────────────
# restore_weights_from_drive  — bulk copy Drive → VM at session start
# ─────────────────────────────────────────────────────────────────────────────
def restore_weights_from_drive():
    """
    Copy all .pt / .pth files from Google Drive back to the Colab VM.

    Call this once at the start of a new Colab session (after Drive is mounted)
    to restore the temporary /content/weights/ folder from the permanent Drive
    copy.  Has no effect when running locally.

    Usage:
        restore_weights_from_drive()   # at top of new session
    """
    if not IN_COLAB:
        print("  (restore_weights_from_drive: local mode — nothing to restore)")
        return

    drive_weights = WEIGHTS_DIR          # e.g. /content/drive/MyDrive/wine-dine/weights
    vm_weights    = str(LOCAL_WEIGHTS)   # /content/weights

    if not os.path.isdir(drive_weights):
        print(f"  ⚠️  Drive weights folder not found: {drive_weights}")
        print("  → Has Drive been mounted? Re-run Section 1.3.")
        return

    files = [f for f in os.listdir(drive_weights) if f.endswith((".pt", ".pth"))]
    if not files:
        print(f"  ℹ️  No .pt / .pth files found in {drive_weights} — nothing to restore.")
        return

    copied, skipped = 0, 0
    for fname in sorted(files):
        src  = os.path.join(drive_weights, fname)
        dest = os.path.join(vm_weights,    fname)
        if os.path.exists(dest) and os.path.getsize(dest) == os.path.getsize(src):
            skipped += 1
            continue                     # already up-to-date on VM
        shutil.copy2(src, dest)
        print(f"  ✓ Restored: {fname}  ({os.path.getsize(dest) / 1e6:.1f} MB)")
        copied += 1

    print(f"\nrestore_weights_from_drive: {copied} copied, {skipped} already up-to-date.")


# ─────────────────────────────────────────────────────────────────────────────
# download_weights_to_local  — trigger browser downloads of all .pt files
# ─────────────────────────────────────────────────────────────────────────────
def download_weights_to_local():
    """Convenience wrapper — prints reminder to use Section 1.4 cell."""
    if not IN_COLAB:
        print("  (Local mode — weights are already on disk)")
        return
    print("  Use the Section 1.4 cell to create a shareable zip on Drive")
    print("  and then pull it to your local PC with gdown.")


# ── Auto-restore on Colab at session start ────────────────────────────────────
restore_weights_from_drive()


Mounted at /content/drive
✓ Google Drive mounted
  Permanent weights → /content/drive/MyDrive/wine-dine/weights
  Permanent figures → /content/drive/MyDrive/wine-dine/figures
✓ Storage helpers ready:  save_checkpoint | load_checkpoint | save_figure
  ✓ Restored: bilstm_best.pt  (12.6 MB)
  ✓ Restored: bilstm_ckpt_ep01.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep02.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep03.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep04.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep05.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep06.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep07.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep08.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep09.pt  (37.7 MB)
  ✓ Restored: bilstm_ckpt_ep10.pt  (37.7 MB)
  ✓ Restored: cnn_resnet50_best.pt  (95.2 MB)
  ✓ Restored: cnn_scratch_best.pt  (23.2 MB)
  ✓ Restored: cnn_scratch_ckpt_ep01.pt  (69.5 MB)
  ✓ Restored: cnn_scratch_ckpt_ep02.pt  (69.5 MB)
  ✓ Restored: cnn_scratch_ckpt_ep03.pt  (69.5 MB)
  ✓ Restored: cnn_sc

### 1.4 — Download trained weights to your local machine

After training in Colab, this cell lets you retrieve all saved weights and figures to your local machine in one step — without manually navigating Google Drive.

**Two-step workflow:**

| Step | Where | Action |
|---|---|---|
| **Step A** | Colab | Run cell — zips all `weights/` and `figures/` files, uploads the zip to Drive, prints a `DRIVE_FILE_ID` |
| **Step B** | Local (VS Code) | Run the same cell — uses `gdown` to pull the zip from Drive by that ID and extracts it into `wine-dine/weights/` |

The cell auto-detects whether it is running on Colab or locally, so the same code cell handles both directions.

> **When to use:** after finishing a training phase in Colab (e.g., after Section 9 — ResNet-50), run this cell on Colab to package the weights, then switch to VS Code and run it again to download them. This keeps the local repository in sync with the latest Colab outputs.


In [5]:
# ── 1.4  Weights ↔ Google Drive ↔ Local PC ────────────────────────────────────
#
# ON COLAB:  zip weights → share on Drive → print the file ID
# ON LOCAL:  pull that zip from Drive via gdown → extract to ./weights/
# ──────────────────────────────────────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════════════════════
# ▶ PASTE YOUR DRIVE FILE ID HERE (printed when you run this cell on Colab):
WEIGHTS_ZIP_ID = ""   # e.g. "1aBcDeFgHiJkLmNoPqRsTuVwXyZ"
# ══════════════════════════════════════════════════════════════════════════════

if IN_COLAB:
    # ── COLAB SIDE: zip + share + print ID ────────────────────────────────────
    import zipfile, glob

    # Step 1: ensure weights are on VM
    print("Step 1 — restoring weights from Google Drive to Colab VM ...")
    restore_weights_from_drive()

    # Step 2: create zip
    found = sorted(glob.glob(os.path.join(str(LOCAL_WEIGHTS), "*.pt")))
    found += sorted(glob.glob(os.path.join(str(LOCAL_WEIGHTS), "*.pth")))

    if not found:
        print("\n⚠️  No .pt / .pth files found — train the model first!")
    else:
        zip_name = "wine-dine-weights.zip"
        zip_path = os.path.join(DRIVE_DIR, zip_name)

        print(f"\nStep 2 — creating {zip_name} on Google Drive ...")
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fpath in found:
                fname = os.path.basename(fpath)
                size_mb = os.path.getsize(fpath) / 1e6
                print(f"  + {fname}  ({size_mb:.1f} MB)")
                zf.write(fpath, arcname=fname)

        zip_size = os.path.getsize(zip_path) / 1e6
        print(f"\n✓ Zip created: {zip_path}  ({zip_size:.1f} MB)")

        # Step 3: make it shareable and get file ID
        print("\nStep 3 — making zip shareable (anyone with link) ...")
        from google.colab import drive as _drive
        from googleapiclient.discovery import build
        from google.auth import default

        creds, _ = default()
        service = build('drive', 'v3', credentials=creds)

        # Find the file on Drive
        query = f"name='{zip_name}' and trashed=false"
        results = service.files().list(q=query, fields="files(id,name)").execute()
        files = results.get('files', [])

        if files:
            file_id = files[0]['id']
            # Make it accessible to anyone with the link
            service.permissions().create(
                fileId=file_id,
                body={'type': 'anyone', 'role': 'reader'},
            ).execute()

            print(f"\n{'='*70}")
            print(f"  ✓ DONE!  Copy this ID into WEIGHTS_ZIP_ID in this cell:")
            print(f"")
            print(f'    WEIGHTS_ZIP_ID = "{file_id}"')
            print(f"")
            print(f"  Then run this same cell on your LOCAL machine to download.")
            print(f"{'='*70}")
        else:
            print("  ⚠️  Could not find zip on Drive — try downloading manually.")

else:
    # ── LOCAL SIDE: pull zip from Drive using gdown ───────────────────────────
    if not WEIGHTS_ZIP_ID:
        print("⚠️  WEIGHTS_ZIP_ID is empty!")
        print("   → Run this cell on Colab first to get the ID.")
        print("   → Then paste it into WEIGHTS_ZIP_ID above and re-run locally.")
    else:
        import subprocess, sys, zipfile

        # Install gdown if needed
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown", "-q"])
            import gdown

        zip_dest = str(WEIGHTS / "wine-dine-weights.zip")
        url = f"https://drive.google.com/uc?id={WEIGHTS_ZIP_ID}"

        print(f"Downloading weights from Google Drive ...")
        print(f"  ID:   {WEIGHTS_ZIP_ID}")
        print(f"  Dest: {WEIGHTS}/\n")

        gdown.download(url, zip_dest, quiet=False)

        # Extract
        print(f"\nExtracting to {WEIGHTS} ...")
        with zipfile.ZipFile(zip_dest, 'r') as zf:
            zf.extractall(str(WEIGHTS))
            names = zf.namelist()

        # Clean up zip
        os.remove(zip_dest)

        print(f"\n✓ Done! {len(names)} file(s) extracted to {WEIGHTS}/")
        for n in sorted(names):
            size_mb = os.path.getsize(str(WEIGHTS / n)) / 1e6
            print(f"  • {n}  ({size_mb:.1f} MB)")

## Section 2 — Raw Data Loading

This section loads both datasets in their **original, uncleaned form**. No rows are dropped, no columns are filtered, and no transformations are applied. The goal is to confirm that the data arrived correctly and to establish a baseline picture of what we are working with before any cleaning decisions are made.

**Why load raw before cleaning?**  
If you clean and load in the same step, you cannot inspect what was removed. Keeping loading and cleaning as separate steps (Sections 2 and 3) means every dropped row is an explicit, documented decision — not a silent side effect of the loader.

**Two data sources:**

| Sub-section | Dataset | Format | Size |
|---|---|---|---|
| 2.1 | Food-101 | JPEG images, folder-per-class | 101,000 images across 101 classes |
| 2.2 | WineSensed (NeurIPS 2023) | JSONL (one review per line) | ~824,000 rows, 357 MB |

---

### 2.1 — Load Food-101

Food-101 is loaded via `torchvision.datasets.Food101`. On the first run it downloads and extracts the dataset to `data/food-101/` (~5 GB). Subsequent runs load from disk instantly from the cache.

**Dataset structure:** 1,000 images per class — 750 in the official train split and 250 in the test split — for a total of 75,750 training and 25,250 test images. The class distribution is perfectly balanced by design, which means no class weighting is needed for the image model.

**Validation checks performed:**
- Train split has exactly 75,750 images, test split has 25,250
- Number of unique class labels equals 101
- Spot-check: first 10 integer labels are in the valid range [0, 100]

> The official Food-101 train/test split (75k/25k) does **not** match the required 70/15/15 ratio. We will pool both splits in Section 4 and create a fresh stratified split.


In [ ]:
# ── 2.1  Load Food-101 via torchvision ───────────────────────────────────────
# torchvision.datasets.Food101 downloads to DATA_DIR/food-101/ on first run.
# Subsequent runs load from the local cache — no network needed.
FOOD101_ROOT = DATA_DIR  # downloads into DATA_DIR/food-101/

print("Loading Food-101 (train split) …")
ds_train = tv_datasets.Food101(root=FOOD101_ROOT, split="train", download=True)
print("Loading Food-101 (test split) …")
ds_test  = tv_datasets.Food101(root=FOOD101_ROOT, split="test",  download=True)
print("Done.\n")

# ── Inspect structure ─────────────────────────────────────────────────────────
train_size  = len(ds_train)
test_size   = len(ds_test)
class_names = ds_train.classes          # list of 101 food names
n_classes   = len(class_names)

print(f"{'Split':<15} {'Rows':>8}")
print("-" * 25)
print(f"{'train':<15} {train_size:>8,}")
print(f"{'test':<15} {test_size:>8,}")
print(f"{'total':<15} {train_size + test_size:>8,}")
print()
print(f"Classes : {n_classes}")
print(f"Example labels: {class_names[:5]} … {class_names[-5:]}")

# ── Validation ────────────────────────────────────────────────────────────────
assert train_size == 75_750, f"Expected 75,750 train rows, got {train_size}"
assert test_size  == 25_250, f"Expected 25,250 test rows, got {test_size}"
assert n_classes  == 101,    f"Expected 101 classes, got {n_classes}"

# Spot-check: first 10 labels are valid integers in [0, 100]
sample_labels = [ds_train[i][1] for i in range(10)]
assert all(0 <= l < 101 for l in sample_labels), "Invalid labels found in train split"

print("\n✓ Section 2.1 validation passed — Food-101 loaded correctly.")


Loading Food-101 (train split) …


 11%|█▏        | 565M/5.00G [00:33<04:06, 18.0MB/s]  

### 2.2 — Load WineSensed reviews (raw)

WineSensed is the NeurIPS 2023 dataset of 824,000 real Vivino user tasting notes. It is downloaded from Hugging Face Hub as a single JSONL file (`all_dataset.jsonl`, ~357 MB) and cached to `data/` — subsequent runs skip the download entirely.

**Why this dataset?**  
WineSensed is one of the few large-scale wine review datasets with structured metadata (grape variety, vintage, Vivino rating, region) attached to every review. The `grape` column gives us a natural multi-class label for the BiLSTM. The `rating` column enables the recommendation engine to surface the highest-rated wine of each grape variety. The review text itself provides the corpus for Word2Vec fine-tuning.

**Raw columns loaded:**

| Column | Content | Notes |
|---|---|---|
| `review` | Vivino tasting note written by a real user | Renamed to `review_text` in Section 3 |
| `wine` | Wine name and vintage (e.g. *"Château Margaux 2015"*) | Used to display the recommended bottle |
| `grape` | Comma-separated grape varieties, primary grape listed first | Primary grape extracted as classification label in Section 3.2 |
| `rating` | Vivino average rating, scale 0–5 | Rescaled to 0–100% approval in Section 3 |
| `country` / `region` | Wine origin | Context only |

**Raw data quality — expected issues at this stage:**
- Missing values in `review`, `grape`, `rating`
- Non-English reviews (Vivino is a global platform)
- Duplicate reviews (same tasting note submitted multiple times)
- Very short reviews (< 10 words) that carry no useful signal
- Multi-grape entries (e.g. `"Cabernet Sauvignon, Merlot"`) requiring primary-grape extraction

All of the above are handled in Section 3. This cell loads without filtering so that Section 3 can report exactly how many rows each cleaning step removes.

**Licence:** CC BY-NC-ND 4.0 — non-commercial research use only.


In [7]:

# ── 2.2  Load WineSensed Reviews (raw) ───────────────────────────────────────
# Dataset : Dakhoo/L2T-NeurIPS-2023 — WineSensed (NeurIPS 2023)
# Licence : CC BY-NC-ND 4.0 — non-commercial research use
#
# The dataset's loading script is broken (references all.tar.gz which does not
# exist — the repo only has all_dataset.jsonl). We download the JSONL directly.
#
# Authentication: a Hugging Face token suppresses the vault warning and ensures
# access to gated datasets. The token is read from:
#   1. Colab Secrets (preferred, never logged) — add your HF token in
#      Runtime → Manage secrets → add key "HF_TOKEN"
#   2. HF_TOKEN environment variable (local or CI)
#   3. Unauthenticated fallback (still works for public datasets, but warns)
#
# No cleaning here. Missing values, non-English reviews, and duplicates are
# all expected — they will be handled in Section 3.

import os
import shutil
from huggingface_hub import hf_hub_download, login

# ── Resolve HF token ─────────────────────────────────────────────────────────
_hf_token = None

if IN_COLAB:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        if _hf_token:
            print("✓ HF_TOKEN loaded from Colab Secrets.")
        else:
            print("⚠  HF_TOKEN not found in Colab Secrets.")
            print("   Add it via Runtime → Manage secrets → key: HF_TOKEN")
    except Exception:
        pass

if not _hf_token:
    _hf_token = os.environ.get("HF_TOKEN")
    if _hf_token:
        print("✓ HF_TOKEN loaded from environment variable.")

if _hf_token:
    login(token=_hf_token, add_to_git_credential=False)
else:
    print("⚠  Proceeding without HF token — public datasets still work, "
          "but the vault warning above is expected.")

# ── Download ──────────────────────────────────────────────────────────────────
_jsonl_path = DATA_DIR / "all_dataset.jsonl"

if not _jsonl_path.exists():
    print("\nDownloading all_dataset.jsonl from Hugging Face (~357 MB) …")
    _downloaded = hf_hub_download(
        repo_id   = "Dakhoo/L2T-NeurIPS-2023",
        filename  = "data/all/all_dataset.jsonl",
        repo_type = "dataset",
        local_dir = str(DATA_DIR),
        token     = _hf_token,
    )
    if str(_downloaded) != str(_jsonl_path):
        shutil.copy(_downloaded, str(_jsonl_path))
    print(f"Saved to {_jsonl_path}")
else:
    print(f"\nUsing cached file: {_jsonl_path}")

print("\nReading JSONL …")
df_wine = pd.read_json(_jsonl_path, lines=True)
print(f"Raw shape : {df_wine.shape}")

# ── Normalise column names ────────────────────────────────────────────────────
df_wine.columns = [c.strip().lower().replace(" ", "_") for c in df_wine.columns]
print(f"Columns   : {df_wine.columns.tolist()}")

# ── Standardise review column → review_text ───────────────────────────────────
for _cand in ["review_text", "review", "reviews"]:
    if _cand in df_wine.columns:
        if _cand != "review_text":
            df_wine = df_wine.rename(columns={_cand: "review_text"})
        break

# ── wine_label = wine name + vintage year ─────────────────────────────────────
if "wine" in df_wine.columns and "year" in df_wine.columns:
    df_wine["wine_label"] = (
        df_wine["wine"].fillna("").astype(str).str.strip()
        + " "
        + df_wine["year"].fillna("").astype(str).str.strip()
    ).str.strip()
elif "wine" in df_wine.columns:
    df_wine["wine_label"] = df_wine["wine"].astype(str).str.strip()
else:
    df_wine["wine_label"] = ""

# ── rating_pct : Vivino avg rating (0–5) → user approval % ──────────────────
if "rating" in df_wine.columns:
    df_wine["rating_pct"] = (
        pd.to_numeric(df_wine["rating"], errors="coerce")
        .clip(0, 5)
        .div(5.0)
        .mul(100)
        .round(0)
        .astype("Int64")
    )

# ── Preview ───────────────────────────────────────────────────────────────────
_PREVIEW = ["wine_label", "review_text", "rating_pct", "grape", "country", "region"]
_preview_cols = [c for c in _PREVIEW if c in df_wine.columns]

pd.set_option("display.max_colwidth", 90)
display(df_wine[_preview_cols].head(5))
pd.reset_option("display.max_colwidth")


In [8]:

# ── 2.2  Raw data shape check ─────────────────────────────────────────────────
# Confirms the JSONL was parsed correctly. Missing values are expected at this
# stage — they will be cleaned in Section 3. Don't be alarmed by high null %.

print(f"{'Raw rows':<25}: {len(df_wine):,}")
print(f"{'Columns':<25}: {df_wine.shape[1]}")
print()

# ── Null counts for the four columns we will need ────────────────────────────
print(f"{'Column':<20} {'Non-null':>10}  {'Null':>8}  {'Null %':>7}")
print("-" * 48)
for col in ["review_text", "grape", "wine_label", "rating_pct"]:
    if col in df_wine.columns:
        non_null = df_wine[col].notna().sum()
        null     = df_wine[col].isna().sum()
        pct      = null / len(df_wine) * 100
        print(f"  {col:<18} {non_null:>10,}  {null:>8,}  {pct:>6.1f}%")

print(f"\n✓ Section 2.2 complete — {len(df_wine):,} raw rows loaded.")
print("  Missing values and non-English reviews will be removed in Section 3.")


## Section 3 — Data Cleaning

**Best practice:** always clean data *before* EDA and *before* model training.
If you clean after plotting, your charts describe data the model never sees — that is misleading.
Every statistic in Section 4 (review length, grape distribution, word clouds) should reflect
the data the models will actually train on.

Food-101 image data needs no cleaning — torchvision downloads a curated, balanced dataset with
no missing labels or corrupted files. Only the WineSensed text data is cleaned here.

### Why each cleaning step matters

| Step | What is removed | Why |
|---|---|---|
| Missing essentials | Rows with no review, grape, wine name, or rating | All four are required — missing any one makes the row useless for training or output |
| Zero rating | Wines rated 0.0 | Vivino encodes "not yet rated" as 0.0 — it is not a genuine score; showing 0% on the recommendation card would be wrong |
| Short reviews | Reviews shorter than 5 words | Too little text for the BiLSTM to learn from; also adds noise to Word2Vec vocabulary |
| Non-English | Reviews with < 80% ASCII characters | Google News Word2Vec is English-only — non-Latin-script reviews produce near-zero vectors |
| Duplicates | Identical review text | The same Vivino review can be attached to multiple wine vintages; duplicates bias training class distributions |


### 3.1 — Clean WineSensed



In [9]:

# ── 3.1  Clean WineSensed Reviews — setup ────────────────────────────────────
_raw_count = len(df_wine)
_step      = {}

# Helper: detect empty / null / placeholder values
def _is_empty(s):
    return (
        s.isna()
        | (s.astype(str).str.strip() == "")
        | (s.astype(str).str.lower().str.strip() == "none")
    )

print(f"Starting row count: {_raw_count:,}")


In [10]:

# ── Step 1: Drop rows missing any essential column ────────────────────────────
# review_text : BiLSTM training text + review retrieval output
# grape       : classification label + Word2Vec grape embedding target
# wine_label  : bottle name shown on the recommendation card
# rating_pct  : approval % shown on the recommendation card
_n    = len(df_wine)
_drop = (
    _is_empty(df_wine["review_text"])
    | _is_empty(df_wine["grape"])
    | _is_empty(df_wine["wine_label"])
    | df_wine["rating_pct"].isna()
)
df_wine = df_wine[~_drop].copy()
_step["1  missing essentials"] = _n - len(df_wine)

print(f"Step 1 — dropped {_step['1  missing essentials']:,} rows (missing essentials)  |  remaining: {len(df_wine):,}")


In [11]:

# ── Step 2: Drop zero-rated wines ─────────────────────────────────────────────
# Vivino encodes "not yet rated" as 0.0 — not a genuine score.
# Showing 0% approval on the recommendation card would be wrong.
_n      = len(df_wine)
df_wine = df_wine[df_wine["rating_pct"] > 0].copy()
_step["2  zero rating (0%)"] = _n - len(df_wine)

print(f"Step 2 — dropped {_step['2  zero rating (0%)']:,} rows (zero rating)  |  remaining: {len(df_wine):,}")


In [12]:

# ── Step 3: Drop very short reviews (< 5 words) ───────────────────────────────
# Reviews shorter than 5 words carry almost no semantic signal for the BiLSTM
# and add noise to the Word2Vec fine-tuning vocabulary.
_n      = len(df_wine)
df_wine = df_wine[df_wine["review_text"].str.split().str.len() >= 5].copy()
_step["3  review < 5 words"] = _n - len(df_wine)

print(f"Step 3 — dropped {_step['3  review < 5 words']:,} rows (review < 5 words)  |  remaining: {len(df_wine):,}")


In [13]:

# ── Step 4: Drop non-English reviews ──────────────────────────────────────────
# Google News Word2Vec is English-only. A French or Italian review will have
# near-zero vectors for most tokens — noise, not signal.
#
# Heuristic: keep rows where >= 80% of characters are plain ASCII (codes 0-127).
# This reliably filters CJK, Cyrillic, Arabic, Greek, etc. while keeping English
# and close Latin-script languages that share vocabulary with English anyway.
_n           = len(df_wine)
_ascii_ratio = df_wine["review_text"].apply(
    lambda t: sum(ord(c) < 128 for c in str(t)) / max(len(str(t)), 1)
)
df_wine = df_wine[_ascii_ratio >= 0.80].copy()
_step["4  non-English (ASCII < 80%)"] = _n - len(df_wine)

print(f"Step 4 — dropped {_step['4  non-English (ASCII < 80%)']:,} rows (non-English)  |  remaining: {len(df_wine):,}")


In [14]:

# ── 3.1  Summary ──────────────────────────────────────────────────────────────
print(f"{'Cleaning step':<42} {'Dropped':>8}  {'%':>5}")
print("─" * 58)
for step, dropped in _step.items():
    pct = dropped / _raw_count * 100
    print(f"  Step {step:<37} {dropped:>8,}  {pct:>4.1f}%")
print("─" * 58)
total_dropped = _raw_count - len(df_wine)
print(f"  {'Total dropped':<40} {total_dropped:>8,}  {total_dropped / _raw_count * 100:>4.1f}%")
print(f"  {'Clean rows remaining':<40} {len(df_wine):>8,}")
print(f"\n✓ Section 3.1 complete — df_wine is clean and ready for EDA and model training.")


### 3.2 — Select top 15 grape varieties as classification labels

Rather than grouping wines into broad types (Red / White / Rosé), we use the **primary grape variety** directly as the BiLSTM classification target. This forces the model to learn the fine-grained flavor language specific to each grape — *"cassis and cedar"* for Cabernet Sauvignon vs *"strawberry and silk"* for Pinot Noir.

We select the **top 15 grapes by review count** from the `grape` column (first entry in the comma-separated list). These 15 varieties typically cover ~85% of all reviews. Rows whose primary grape is not in the top 15 are dropped.

**Why this runs after cleaning:** grape frequency counts are computed on clean data only. Running this on raw data would inflate some counts with rows that will be dropped later (duplicates, non-English, etc.) and could distort which 15 grapes are selected.

**Expected result:** New `grape_class` column with one of 15 grape names. Distribution printed.

**Validation checks:**
- Exactly 15 unique values in `grape_class`
- Distribution printed (check for severe imbalance)
- No nulls in `grape_class`



In [15]:

# ── 3.2  Select top N grapes — Step 1: Extract primary grape ─────────────────
# Primary grape = first entry in the comma-separated `grape` column.

TOP_N_GRAPES = 15

df_wine["primary_grape"] = (
    df_wine["grape"]
    .fillna("")
    .str.split(",")
    .str[0]
    .str.strip()
)

print(f"Unique primary grapes found : {df_wine['primary_grape'].nunique():,}")
print(f"Sample values : {df_wine['primary_grape'].value_counts().head(5).to_dict()}")


In [16]:

# ── 3.2  Step 2: Find top N grapes by review count ────────────────────────────
grape_counts = df_wine["primary_grape"].value_counts()
top_grapes   = grape_counts.head(TOP_N_GRAPES).index.tolist()

print(f"Top {TOP_N_GRAPES} grapes selected (out of {df_wine['primary_grape'].nunique()} unique):\n")
print(", ".join(top_grapes))


In [17]:

# ── 3.2  Step 3: Filter df_wine to top N grapes & assign grape_class ──────────
df_wine["grape_class"] = df_wine["primary_grape"].where(
    df_wine["primary_grape"].isin(top_grapes)
)
df_wine_mapped = df_wine.dropna(subset=["grape_class"]).copy()

coverage = len(df_wine_mapped) / len(df_wine) * 100
print(f"Rows kept    : {len(df_wine_mapped):>10,}  ({coverage:.1f}% of reviews)")
print(f"Rows dropped : {len(df_wine) - len(df_wine_mapped):>10,}  "
      f"(primary grape not in top {TOP_N_GRAPES})")


In [18]:

# ── 3.2  Step 5: Distribution table & expose GRAPE_CLASSES ───────────────────
dist = df_wine_mapped["grape_class"].value_counts()

print(f"{'Grape variety':<28} {'Count':>8}  {'%':>6}")
print("-" * 46)
for grape, count in dist.items():
    pct = count / len(df_wine_mapped) * 100
    bar = "█" * int(pct / 2)
    print(f"{grape:<28} {count:>8,}  {pct:>5.1f}%  {bar}")

# Expose the ordered class list for later sections
GRAPE_CLASSES = dist.index.tolist()   # ordered by frequency

print(f"\n✓ Section 3.2 complete — grape_class column added ({TOP_N_GRAPES} classes).")


### 3.2.1 — Food-101 Image Class Balance Check

Before building the flavor table it is worth confirming that the image dataset has no class-imbalance problem. An imbalanced image dataset would bias the CNN toward over-represented classes, and that error would propagate all the way into wine recommendations.

**Food-101 spec (by design):** 1,000 images per class — 750 train + 250 test × 101 classes.

**Validation checks:**
- Min and max image counts per class are equal in both splits
- All 101 classes are present in both splits


In [19]:

# ── 3.2.1  Food-101 Class Balance Check ──────────────────────────────────────
# ds_train._labels and ds_test._labels are lists of integer class indices
# loaded from the Food-101 metadata JSON files — no image decoding needed.
from collections import Counter

train_counts = Counter(ds_train._labels)
test_counts  = Counter(ds_test._labels)

train_vals = list(train_counts.values())
test_vals  = list(test_counts.values())

print("Food-101 class balance")
print("=" * 52)
print(f"{'Split':<8}  {'Classes':>8}  {'Min imgs':>10}  {'Max imgs':>10}  {'Mean':>7}")
print("-" * 52)
print(f"{'train':<8}  {len(train_counts):>8}  "
      f"{min(train_vals):>10}  {max(train_vals):>10}  "
      f"{sum(train_vals)/len(train_vals):>7.1f}")
print(f"{'test':<8}  {len(test_counts):>8}  "
      f"{min(test_vals):>10}  {max(test_vals):>10}  "
      f"{sum(test_vals)/len(test_vals):>7.1f}")

# Balance verdict
print()
for split_name, vals in [("train", train_vals), ("test", test_vals)]:
    if min(vals) == max(vals):
        print(f"✓ {split_name.capitalize()} split — perfectly balanced "
              f"({min(vals)} images per class across all {len(vals)} classes).")
    else:
        ratio = max(vals) / min(vals)
        print(f"⚠️  {split_name.capitalize()} split — imbalanced  "
              f"(ratio {ratio:.2f}x, min={min(vals)}, max={max(vals)}).")

# Class coverage
missing_train = set(range(101)) - set(train_counts.keys())
missing_test  = set(range(101)) - set(test_counts.keys())
if not missing_train and not missing_test:
    print("✓ All 101 classes are present in both splits.")
else:
    if missing_train:
        print(f"⚠️  Missing from train: "
              f"{[ds_train.classes[i] for i in sorted(missing_train)]}")
    if missing_test:
        print(f"⚠️  Missing from test : "
              f"{[ds_train.classes[i] for i in sorted(missing_test)]}")

print("\n✓ Section 3.2.1 complete.")


### 3.3 — Food flavor table

The food flavor table is the bridge between the CNN's food label and the wine pairing system.  
It is stored in `data/food_flavor_table.json` — 101 entries, one per Food-101 class.

Each entry has three keyword sets in **plain food/taste language** (not wine vocabulary). Word2Vec translates them into grape space via cosine similarity.

| Key | Card label | Purpose | Example for `pizza` |
|---|---|---|---|
| `classic` | **Safe Bet** | Dominant flavors — points toward grapes with matching character | `tomato, cheesy, savory, herby, baked, meaty` |
| `contrast` | **Bold Move** | Heavy/rich qualities — points toward grapes that cut through | `fatty, greasy, heavy, starchy, salty` |
| `safe_bet` | **Hidden Gem** | Crowd-pleasing, neutral — approachable, easy-drinking grapes | `light, crisp, mild, clean, easy` |

**Validation checks run below:**
- All 101 Food-101 class names present as keys
- Each entry has exactly 3 keys: `classic`, `contrast`, `safe_bet`
- All keyword lists are non-empty
- Sample entries printed for inspection


In [ ]:
# ── 3.3  Load food flavor table ─────────────────────────────────────────────────────────
import json
from pathlib import Path

_flavor_candidates = [
    Path(DRIVE_DIR) / "data" / "food_flavor_table.json",  # Colab (Drive)
    BASE_DIR / "data" / "food_flavor_table.json",          # local
]
_FLAVOR_PATH = next((p for p in _flavor_candidates if p.exists()), None)
if _FLAVOR_PATH is None:
    raise FileNotFoundError(
        "food_flavor_table.json not found.\n"
        f"  Colab : {_flavor_candidates[0]}\n"
        f"  Local : {_flavor_candidates[1]}"
    )

with open(_FLAVOR_PATH, encoding="utf-8") as _f:
    food_flavor_table = {k: v for k, v in json.load(_f).items()
                         if not k.startswith("_")}

assert len(food_flavor_table) == 101
assert all(set(v) == {"classic", "contrast", "safe_bet"}
           for v in food_flavor_table.values())
print(f"✓ food_flavor_table — {len(food_flavor_table)} dishes  ← {_FLAVOR_PATH}")


## Section 4 — Train / Val / Test Split (Both Datasets)

**Both datasets are split here, before any EDA.** Performing EDA on the full dataset would leak test-set information into architecture decisions — `MAX_SEQ_LEN` would be set using test-review lengths, class-distribution charts would show test-class proportions, and word clouds would include test-set vocabulary. All of these constitute subtle but real data leakage.

> ⛔ **Test sets are frozen immediately after this section and must not be inspected until the final evaluation cell of each model.**

| Sub-section | Dataset | Method | Outputs |
|---|---|---|---|
| **4.1** | WineSensed (text) | Stratified `train_test_split` on `df_wine_mapped` | `df_train`, `df_val`, `df_test` |
| **4.2** | Food-101 (images) | Pool all 101k images, stratified `train_test_split` on indices | `_idx_train`, `_idx_val`, `_idx_test` |


### 4.1 — WineSensed (text reviews)

| Split | Variable | Size | Purpose |
|---|---|---|---|
| **train** (70%) | `df_train` | ~70% of rows | Model training, vocabulary building, Word2Vec fine-tuning |
| **val** (15%) | `df_val` | ~15% of rows | Early stopping, hyperparameter tuning |
| **test** (15%) | `df_test` | ~15% of rows | Final evaluation only — touched once per model |

Stratified by `grape_class` — every split preserves the same class proportions as the full dataset. A per-grape proportion table is printed to verify stratification.


In [20]:

# ── 4.1  Train / Val / Test split — WineSensed ───────────────────────────────
# Split BEFORE EDA (Section 5) so all statistics — review length percentiles,
# word clouds, class distributions — are computed on training data only.
# Computing them on the full dataset would let test-set information silently
# influence MAX_SEQ_LEN and other decisions baked into the model.
#
# 70 / 15 / 15 stratified by grape_class so every split preserves the same
# class proportions as the full cleaned dataset.
from sklearn.model_selection import train_test_split

GRAPE_TO_IDX = {g: i for i, g in enumerate(GRAPE_CLASSES)}

_df     = df_wine_mapped.copy()
_labels = _df["grape_class"].map(GRAPE_TO_IDX)

df_train, _df_tmp = train_test_split(
    _df, test_size=0.30, stratify=_labels, random_state=SEED
)
df_val, df_test = train_test_split(
    _df_tmp, test_size=0.50,
    stratify=_df_tmp["grape_class"].map(GRAPE_TO_IDX),
    random_state=SEED
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# ── Split sizes ───────────────────────────────────────────────────────────────
print(f"{'Split':<8} {'Rows':>8}  {'%':>5}")
print("-" * 25)
for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"{name:<8} {len(df):>8,}  {len(df)/len(df_wine_mapped)*100:>4.1f}%")
print(f"{'total':<8} {len(df_train)+len(df_val)+len(df_test):>8,}")

# ── Stratification check — class proportions should be ~70 / 15 / 15 ─────────
print("\nStratification check — each grape should be ~70 / 15 / 15:")
print(f"{'Grape':<25} {'train%':>7}  {'val%':>6}  {'test%':>6}")
print("-" * 50)
for grape in GRAPE_CLASSES:
    tr = (df_train["grape_class"] == grape).sum()
    v  = (df_val["grape_class"]   == grape).sum()
    te = (df_test["grape_class"]  == grape).sum()
    n  = tr + v + te
    print(f"{grape:<25} {tr/n*100:>6.1f}%  {v/n*100:>5.1f}%  {te/n*100:>5.1f}%")

print(f"\n⛔ df_test frozen — do not inspect until final evaluation (Sections 12 & 14).")
print(f"   EDA in Section 5 uses df_train only.")
print(f"\n✓ Section 4 complete.")


### 4.2 — Food-101 (images)

Food-101's built-in 75k / 25k split does not match the required 70 / 15 / 15 ratios. We **pool all 101,000 images** and create a fresh stratified 3-way split using the same fixed `SEED`.

| Split | Index array | Samples | % | Per-class |
|---|---|---|---|---|
| **train** | `_idx_train` | 70,700 | 70% | ~700 |
| **val** | `_idx_val` | 15,150 | 15% | ~150 |
| **test** | `_idx_test` | 15,150 | 15% | ~150 |

Only **index arrays** are produced here — transforms and DataLoaders require GPU and are built in Section 6 on Colab. The index arrays are deterministic (fixed `SEED`) so Section 6 will produce exactly the same split.

In [21]:

# ── 4.2  Train / Val / Test split — Food-101 (index arrays only) ─────────────
# Transforms and DataLoaders require GPU (Colab, Section 6).
# Here we only compute the index arrays so the split is determined BEFORE EDA
# and before any image statistics are computed.
#
# Food-101 has exactly 1,000 images per class across both official splits
# (750 train + 250 test). We pool all 101,000 and split 70/15/15 with SEED.

_base_train_meta = tv_datasets.Food101(root=FOOD101_ROOT, split="train", download=False,
                                        transform=None)
_base_test_meta  = tv_datasets.Food101(root=FOOD101_ROOT, split="test",  download=False,
                                        transform=None)

_n_food_tr  = len(_base_train_meta)                                      # 75,750
_all_labels = _base_train_meta._labels + _base_test_meta._labels         # 101,000 labels
_all_idx    = list(range(len(_all_labels)))                              # 0 .. 100,999

# Step 1 — 70% train vs 30% temp  (train_test_split imported in Section 4.1)
_idx_train, _idx_tmp = train_test_split(
    _all_idx, test_size=0.30, stratify=_all_labels, random_state=SEED
)
# Step 2 — split 30% temp evenly → val (15%) and test (15%)
_lbl_tmp = [_all_labels[i] for i in _idx_tmp]
_idx_val, _idx_test = train_test_split(
    _idx_tmp, test_size=0.50, stratify=_lbl_tmp, random_state=SEED
)

# ── Verify split sizes ────────────────────────────────────────────────────────
_total = len(_all_labels)
print(f"{'Split':<8} {'Samples':>8}  {'%':>6}  {'Per-class (avg)':>16}")
print("-" * 46)
for _name, _idx in [("train", _idx_train), ("val", _idx_val), ("test", _idx_test)]:
    _n = len(_idx)
    print(f"{_name:<8} {_n:>8,}  {_n/_total*100:>5.1f}%  {_n/101:>15.0f}")
print(f"{'total':<8} {_total:>8,}  100.0%")

# ── Verify stratification — per-class sample counts should be uniform ─────────
from collections import Counter as _Counter
_train_lbl_counts = _Counter(_all_labels[i] for i in _idx_train)
_val_lbl_counts   = _Counter(_all_labels[i] for i in _idx_val)
_test_lbl_counts  = _Counter(_all_labels[i] for i in _idx_test)

_train_per_class = sorted(_train_lbl_counts.values())
_val_per_class   = sorted(_val_lbl_counts.values())
_test_per_class  = sorted(_test_lbl_counts.values())

print(f"\nPer-class sample range:")
print(f"  train : min={min(_train_per_class)}  max={max(_train_per_class)}  (expected ~700)")
print(f"  val   : min={min(_val_per_class)}  max={max(_val_per_class)}  (expected ~150)")
print(f"  test  : min={min(_test_per_class)}  max={max(_test_per_class)}  (expected ~150)")

print(f"\n⛔ Image test split frozen — _idx_test must not be used until final evaluation (Sections 9 & 10).")
print(f"\n✓ Section 4.2 complete — index arrays ready: _idx_train ({len(_idx_train):,})  "
      f"_idx_val ({len(_idx_val):,})  _idx_test ({len(_idx_test):,})")


### 4.3 — Test Set Freeze

**Both test sets are now frozen. Do not inspect, plot, or evaluate on them until the final model evaluation cells.**

| Dataset | Train (70%) | Val (15%) | Test (15%) | Final evaluation |
|---|---|---|---|---|
| WineSensed (text) | `df_train` | `df_val` | `df_test` | Sections 8 & 12 |
| Food-101 (images) | `_idx_train` (70,700) | `_idx_val` (15,150) | `_idx_test` (15,150) | Sections 9 & 10 |


## Section 5 — Exploratory Data Analysis

EDA runs on **clean, split data only** (after Sections 3–4). Every chart here describes exactly what the models will train on. Using pre-split training data prevents test-set statistics from influencing architecture choices.

| Sub-section | Dataset | Purpose |
|---|---|---|
| **5.1** | Food-101 (images) | Sample grid, class distribution, image dimensions, channel statistics |
| **5.2** | WineSensed (text) | Review length → `MAX_SEQ_LEN`, grape distribution, word clouds, rating distribution |

`MAX_SEQ_LEN` is set at the end of 5.2 and reused throughout Sections 7–8.


### 5.1 — Food-101 Image EDA

Food-101 is a fixed, curated benchmark with 101 food categories and 1,000 images per class.
The goals here are to confirm class balance, characterise native image dimensions, and verify
that ImageNet normalisation statistics are appropriate for this dataset before applying them in Section 6.


In [22]:

# ── 5.1  Image EDA ────────────────────────────────────────────────────────────
# All cells below use _idx_train — the 70,700-image training slice from Section 4.2.
# Indices 0.._n_food_tr-1 map to ds_train; indices _n_food_tr.. map to ds_test.

def _food101_img_path(idx):
    """Return PIL image file path for a combined pool index from Section 4.2."""
    return ds_train._image_files[idx] if idx < _n_food_tr else ds_test._image_files[idx - _n_food_tr]

print(f"EDA uses _idx_train: {len(_idx_train):,} images  "
      f"(70% of pooled {len(_idx_train)+len(_idx_val)+len(_idx_test):,} Food-101 images)")

# Build class → training-index lookup once (reused in grid + class-dist cell)
_cls_to_train_idx = {}
for _i in _idx_train:
    _cls_to_train_idx.setdefault(_all_labels[_i], []).append(_i)

# ── Sample image grid (16 classes, 1 image each) ─────────────────────────────
_grid_cls_idxs = random.sample(list(_cls_to_train_idx.keys()), 16)
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle("Food-101 — sample images (1 per class, our training split)", fontsize=13)

for ax, cls_idx in zip(axes.flat, _grid_cls_idxs):
    pool_i = random.choice(_cls_to_train_idx[cls_idx])
    img    = Image.open(_food101_img_path(pool_i)).convert("RGB")
    ax.imshow(img)
    ax.set_title(ds_train.classes[cls_idx].replace("_", " "), fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES / "eda_image_grid.png", dpi=100)
plt.show()



### 5.1.1 — Class Distribution

Food-101 is a perfectly balanced benchmark: each of the 101 classes has exactly 750 training images (75,750 total).
The bar chart below confirms this — no class weighting is needed for the image model.


In [23]:

# ── 5.1.1  Class distribution ─────────────────────────────────────────────────
# Uses _cls_to_train_idx built in the cell above (only our training split).
counts_per_cls = {cls: len(idxs) for cls, idxs in _cls_to_train_idx.items()}
counts_sorted  = sorted(counts_per_cls.values())

fig, ax = plt.subplots(figsize=(18, 4))
ax.bar(range(101), counts_sorted, color="steelblue", width=1.0)
ax.axhline(700, color="red", linestyle="--", linewidth=1.5,
           label="Expected ~700 / class  (70% × 1,000)")
ax.set_xlabel("Class index (sorted by count)")
ax.set_ylabel("Number of training images")
ax.set_title("Food-101 — class distribution in our training split (70 %, all 101 classes)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "eda_image_class_dist.png", dpi=100)
plt.show()

print(f"Class counts — min: {min(counts_sorted)}  "
      f"max: {max(counts_sorted)}  "
      f"mean: {sum(counts_sorted)/len(counts_sorted):.1f}  "
      f"all equal: {min(counts_sorted) == max(counts_sorted)}")
print("→ Balanced dataset: no class weighting needed for the image model.")



### 5.1.2 — Image Dimensions & Aspect Ratio

Food-101 images are variable-sized JPEGs scraped from the web.
The histograms below show native sizes and aspect ratios **before** any resizing.
The aspect ratio distribution confirms most images are near-square (W/H ≈ 1),
which justifies the `CenterCrop(224)` strategy in Section 6 — minimal content loss versus padding.
All images will be resized to **224 × 224** in Section 6.


In [24]:

# ── 5.1.2  Image dimensions & aspect ratio ────────────────────────────────────
_dim_n      = 500
_dim_sample = random.sample(_idx_train, _dim_n)
_widths, _heights, _aspects = [], [], []

for _i in _dim_sample:
    with Image.open(_food101_img_path(_i)) as _im:
        _w, _h = _im.size
    _widths.append(_w)
    _heights.append(_h)
    _aspects.append(_w / _h)

_widths_s  = sorted(_widths)
_heights_s = sorted(_heights)
_asp_s     = sorted(_aspects)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(_widths,  bins=40, color="steelblue", edgecolor="none")
axes[0].set_title(f"Image widths (sample of {_dim_n})")
axes[0].set_xlabel("Width (pixels)")
axes[0].set_ylabel("Count")

axes[1].hist(_heights, bins=40, color="steelblue", edgecolor="none")
axes[1].set_title(f"Image heights (sample of {_dim_n})")
axes[1].set_xlabel("Height (pixels)")
axes[1].set_ylabel("Count")

axes[2].hist(_aspects, bins=40, color="teal", edgecolor="none")
axes[2].axvline(1.0, color="red", linestyle="--", linewidth=1.5, label="Square (1 : 1)")
axes[2].set_title(f"Aspect ratio W/H (sample of {_dim_n})")
axes[2].set_xlabel("Aspect ratio (W / H)")
axes[2].set_ylabel("Count")
axes[2].legend()

plt.suptitle("Food-101 — raw image dimensions & aspect ratios (our training split)", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "eda_image_dims.png", dpi=100)
plt.show()

print(f"Width   — min: {_widths_s[0]:<6}  median: {_widths_s[_dim_n // 2]:<6}  max: {_widths_s[-1]}")
print(f"Height  — min: {_heights_s[0]:<6}  median: {_heights_s[_dim_n // 2]:<6}  max: {_heights_s[-1]}")
print(f"Aspect  — min: {_asp_s[0]:.3f}  median: {_asp_s[_dim_n // 2]:.3f}  max: {_asp_s[-1]:.3f}")
print(f"→ Most images are near-square — CenterCrop(224) is appropriate (Section 6).")



### 5.1.3 — Channel Statistics (RGB)

We estimate per-channel mean and standard deviation on a random sample to confirm that
**ImageNet normalisation** (mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]) is appropriate for Food-101.
Close agreement means the pre-trained backbone's internal statistics remain valid without re-normalisation.


In [25]:

# ── 5.1.3  Channel statistics (RGB mean & std) ────────────────────────────────
_ch_n      = 1_000
_ch_sample = random.sample(_idx_train, _ch_n)
_px        = []

for _i in _ch_sample:
    _img = Image.open(_food101_img_path(_i)).convert("RGB").resize((64, 64))
    _px.append(np.array(_img, dtype=np.float32) / 255.0)   # (64, 64, 3)

_px  = np.stack(_px)          # (N, 64, 64, 3)
_mu  = _px.mean(axis=(0, 1, 2))
_sig = _px.std(axis=(0, 1, 2))

_channels      = ["R", "G", "B"]
_imagenet_mean = [0.485, 0.456, 0.406]
_imagenet_std  = [0.229, 0.224, 0.225]
_x = np.arange(3)
_w = 0.32

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(_x - _w / 2, _mu,            _w, label="Food-101",  color="steelblue")
ax1.bar(_x + _w / 2, _imagenet_mean, _w, label="ImageNet",  color="orange", alpha=0.85)
ax1.set_xticks(_x); ax1.set_xticklabels(_channels)
ax1.set_title("Per-channel mean")
ax1.set_ylabel("Mean pixel value (0–1)")
ax1.legend()

ax2.bar(_x - _w / 2, _sig,           _w, label="Food-101",  color="steelblue")
ax2.bar(_x + _w / 2, _imagenet_std,  _w, label="ImageNet",  color="orange", alpha=0.85)
ax2.set_xticks(_x); ax2.set_xticklabels(_channels)
ax2.set_title("Per-channel std")
ax2.set_ylabel("Std dev (0–1)")
ax2.legend()

plt.suptitle(f"Channel statistics — Food-101 vs. ImageNet (sample of {_ch_n} training images)", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "eda_channel_stats.png", dpi=100)
plt.show()

print(f"Food-101 mean : R={_mu[0]:.4f}  G={_mu[1]:.4f}  B={_mu[2]:.4f}")
print(f"Food-101 std  : R={_sig[0]:.4f}  G={_sig[1]:.4f}  B={_sig[2]:.4f}")
print(f"ImageNet  mean: R=0.4850       G=0.4560       B=0.4060")
print(f"ImageNet  std : R=0.2290       G=0.2240       B=0.2250")
print(f"\n→ Dataset stats are close to ImageNet norms — ImageNet normalisation is appropriate.")
print(f"\n✓ Section 5.1 complete.")


### 5.2 — WineSensed Text EDA

All statistics below are computed on `df_train` only. The key output of this sub-section is
`MAX_SEQ_LEN` — the sequence length cap derived from the 95th-percentile review length —
which is used to build the vocabulary and DataLoaders in Section 7.


In [26]:

# ── 5.2  Text EDA ─────────────────────────────────────────────────────────────
# Uses df_train only — split was done in Section 4.1 (before EDA) to prevent
# test-set statistics from influencing MAX_SEQ_LEN or any other model decision.

print(f"Training samples : {len(df_train):,}")
print(f"Val samples      : {len(df_val):,}")
print(f"Test samples     : {len(df_test):,}  (frozen until final evaluation)")
print(f"Grape classes    : {len(GRAPE_CLASSES)}")
print(f"Text column      : 'review_text'")



### 5.2.1 — Review Length Distribution

Knowing how long training reviews are determines `MAX_SEQ_LEN`, which must be set **before** building the vocabulary and DataLoaders in Section 7.
We choose the 95th percentile so that only 5 % of training reviews are truncated, keeping sequence length manageable.


In [27]:

# ── 5.2.1  Review length distribution ────────────────────────────────────────
lengths = df_train["review_text"].str.split().str.len()

p50  = int(lengths.quantile(0.50))
p90  = int(lengths.quantile(0.90))
p95  = int(lengths.quantile(0.95))
p99  = int(lengths.quantile(0.99))

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(lengths.clip(upper=300), bins=80, color="steelblue", edgecolor="none")
for pct, val, col in [(50, p50, "green"), (90, p90, "orange"), (95, p95, "red"), (99, p99, "purple")]:
    ax.axvline(val, color=col, linestyle="--", label=f"p{pct} = {val} words")
ax.set_xlabel("Review length (words)")
ax.set_ylabel("Count")
ax.set_title("WineSensed — review length distribution (training set, clipped at 300 words)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "eda_review_lengths.png", dpi=100)
plt.show()

print(f"p50 = {p50} words | p90 = {p90} | p95 = {p95} | p99 = {p99}")
pct_truncated = (lengths > p95).mean() * 100
print(f"Choosing MAX_SEQ_LEN = p95 = {p95}  →  {pct_truncated:.1f}% of reviews truncated")

# ── Define MAX_SEQ_LEN — reused in Section 7 ─────────────────────────────────
MAX_SEQ_LEN = p95
print(f"\nMAX_SEQ_LEN = {MAX_SEQ_LEN}  (reused in Section 7 DataLoaders)")



### 5.2.2 — Class / Target Distribution

The bar chart shows how many training reviews fall in each of the 15 grape-variety classes.
An imbalance ratio > 3× will trigger **weighted CrossEntropyLoss** in the training sections.


In [28]:

# ── 5.2.2  Grape class / target distribution ─────────────────────────────────
grape_dist = df_train["grape_class"].value_counts()

fig, ax = plt.subplots(figsize=(13, 5))
grape_dist.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("WineSensed — grape class distribution (training set, all 15 classes)")
ax.set_xlabel("Grape variety")
ax.set_ylabel("Number of reviews")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(FIGURES / "eda_grape_dist.png", dpi=100)
plt.show()

imbalance_ratio = grape_dist.max() / grape_dist.min()
print(f"Class counts — min: {grape_dist.min():,}  max: {grape_dist.max():,}")
print(f"Imbalance ratio (max / min): {imbalance_ratio:.1f}×")
if imbalance_ratio > 3:
    print("  ⚠  Significant imbalance — weighted CrossEntropyLoss will be used in Sections 11/12.")
else:
    print("  ✓ Imbalance acceptable.")



### 5.2.3 — Top-N Terms per Class (Word Clouds)

One word cloud per grape variety, generated from the **training set only**.
Common English stop-words plus generic wine terms (*wine*, *bottle*, *glass*, *finish*) are removed
so that the clouds reveal grape-specific flavour vocabulary rather than shared jargon.


In [29]:

# ── 5.2.3  Word clouds per grape variety (all 15 classes) ────────────────────
from nltk.corpus import stopwords

_stopwords = set(stopwords.words("english")) | {
    "wine", "drink", "bottle", "glass", "finish", "palate", "nose", "taste"
}

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle("WineSensed — word clouds per grape variety (training set)", fontsize=14)

for ax, grape in zip(axes.flat, GRAPE_CLASSES):
    text = " ".join(df_train[df_train["grape_class"] == grape]["review_text"])
    wc = WordCloud(
        width=300, height=200, background_color="white",
        stopwords=_stopwords, max_words=60, collocations=False,
    ).generate(text)
    ax.imshow(wc.to_image(), interpolation="bilinear")
    ax.set_title(grape, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES / "eda_wordclouds.png", dpi=100)
plt.show()



### 5.2.4 — Representative Samples per Class

Three reviews are displayed for each of the 15 grape varieties (from the training set, truncated to 150 characters).
This gives an intuition for the vocabulary richness and how distinguishable the classes are at a glance.


In [31]:

# ── 5.2.4  Representative samples per class (all 15 grapes) ──────────────────
_n_samples = 3
for grape in GRAPE_CLASSES:
    _subset  = df_train[df_train["grape_class"] == grape]["review_text"]
    _samples = _subset.sample(min(_n_samples, len(_subset)), random_state=SEED)
    print(f"── {grape} ({len(_subset):,} reviews) ──")
    for rev in _samples:
        print(f"  • {rev[:150]}")
    print()

print("✓ Section 5.2 complete.")



### 5.2.5 — Rating Distribution per Grape Class

`rating_pct` (Vivino average rating rescaled to 0–100 %) lets us check whether
some grape varieties are systematically rated higher than others.
This directly motivates the business integration layer: a recommendation engine
can combine the CNN's food-match score with the wine's grape-class approval rating.


In [32]:

# ── 5.2.5  Rating distribution per grape class ────────────────────────────────
# Uses df_train only.
if "rating_pct" not in df_train.columns:
    print("⚠  rating_pct column not found — skipping rating EDA.")
else:
    _medians = df_train.groupby("grape_class")["rating_pct"].median()
    _order   = _medians.sort_values(ascending=False).index.tolist()

    # Build sorted list of arrays for matplotlib boxplot (no 'order=' kwarg needed)
    _data_sorted = [
        df_train.loc[df_train["grape_class"] == g, "rating_pct"].dropna().values
        for g in _order
    ]

    fig, ax = plt.subplots(figsize=(14, 5))
    bp = ax.boxplot(
        _data_sorted,
        labels=_order,
        patch_artist=True,
        medianprops=dict(color="red", linewidth=2),
        flierprops=dict(marker=".", markersize=2, alpha=0.3, color="grey"),
    )
    for patch in bp["boxes"]:
        patch.set(facecolor="#aec6e8", edgecolor="steelblue")
    for w in bp["whiskers"] + bp["caps"]:
        w.set(color="steelblue")

    ax.set_title("WineSensed — approval rating (%) by grape variety (training set)")
    ax.set_xlabel("Grape variety (sorted by median rating)")
    ax.set_ylabel("Approval rating (%)")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.savefig(FIGURES / "eda_rating_per_grape.png", dpi=100)
    plt.show()

    print("Median approval rating per grape (sorted):")
    for _g, _m in _medians.sort_values(ascending=False).items():
        print(f"  {_g:<30} {_m:.1f}%")
    print(f"\nRange: {_medians.min():.1f}% – {_medians.max():.1f}%  "
          f"(spread = {_medians.max() - _medians.min():.1f} pp)")



### 5.2.6 — Review Length per Grape Class

Are some grape varieties described in longer sentences than others?
Significant variation would suggest that a single global `MAX_SEQ_LEN` might under-serve
verbose classes — or confirm that one length fits all.


In [33]:

# ── 5.2.6  Review length per grape class ─────────────────────────────────────
_df_len            = df_train.copy()
_df_len["n_words"] = _df_len["review_text"].str.split().str.len()

_len_medians = _df_len.groupby("grape_class")["n_words"].median()
_order_len   = _len_medians.sort_values(ascending=False).index.tolist()

# Build sorted list of data arrays for matplotlib boxplot
_len_data_sorted = [
    _df_len.loc[_df_len["grape_class"] == g, "n_words"].dropna().values
    for g in _order_len
]

fig, ax = plt.subplots(figsize=(14, 5))
bp2 = ax.boxplot(
    _len_data_sorted,
    labels=_order_len,
    patch_artist=True,
    medianprops=dict(color="red", linewidth=2),
    flierprops=dict(marker=".", markersize=2, alpha=0.3, color="grey"),
)
for patch in bp2["boxes"]:
    patch.set(facecolor="#aec6e8", edgecolor="steelblue")
for w in bp2["whiskers"] + bp2["caps"]:
    w.set(color="steelblue")

ax.axhline(MAX_SEQ_LEN, color="purple", linestyle="--", linewidth=1.5,
           label=f"MAX_SEQ_LEN = {MAX_SEQ_LEN}")
ax.set_title("WineSensed — review length (words) by grape variety (training set)")
ax.set_xlabel("Grape variety (sorted by median length)")
ax.set_ylabel("Review length (words)")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(FIGURES / "eda_length_per_grape.png", dpi=100)
plt.show()

print(f"Median review length per grape — range: "
      f"{_len_medians.min():.0f} – {_len_medians.max():.0f} words  "

      f"(spread = {_len_medians.max() - _len_medians.min():.0f} words)")

print(f"MAX_SEQ_LEN ({MAX_SEQ_LEN}) covers the majority of reviews across all classes.")
print(f"\n✓ Section 5.2 complete.")

## Section 6 — Image Preprocessing and DataLoaders

**Requires GPU — run in Google Colab.**

The 70 / 15 / 15 index arrays (\_idx_train\, \_idx_val\, \_idx_test\) were computed in Section 4.2 before EDA. This section applies transforms to those indices and builds the DataLoaders.

| Sub-section | What it does |
|---|---|
| **6.1** | Resize strategy — 256-then-crop to 224 × 224 |
| **6.2** | ImageNet normalisation statistics and why they apply to Food-101 |
| **6.3** | Data augmentation (training only) — each transform justified |
| **6.4** | Wrap index arrays into \_Food101Combined\ Dataset objects |
| **6.5** | Build \DataLoader\ objects with batching and GPU pinned memory |


### 6.1 — Resize

All images are resized to **224 × 224 pixels**. The pipeline first upsizes the shorter edge to 256 (`Resize(256)`) and then takes a 224-crop — random during training, centred during validation/test. This two-step approach is standard for ImageNet-pretrained models: it preserves more scene content than a direct 224-resize while keeping the input size constant, and matches the preprocessing used when ResNet-50 was originally trained.

> The EDA (Section 5.1.2) confirmed that Food-101 images are predominantly ≥ 256 px on both sides, so upscaling artefacts are negligible.

In [34]:

# ── 6.1  Resize demo ──────────────────────────────────────────────────────────
# Show a sample Food-101 image before and after the resize/crop pipeline.

_demo_img_a, _demo_cls_a = ds_train[0]   # raw PIL image, no transform applied
_resize_only = T.Compose([T.Resize(256), T.CenterCrop(224)])
_resized_img_a = _resize_only(_demo_img_a)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(_demo_img_a)
axes[0].set_title(f"Original  {_demo_img_a.size[0]}×{_demo_img_a.size[1]} px")
axes[0].axis("off")
axes[1].imshow(_resized_img_a)
axes[1].set_title("After Resize(256) → CenterCrop(224)\n224×224 px")
axes[1].axis("off")
fig.suptitle(f"6.1 — Resize  (class: {ds_train.classes[_demo_cls_a]})", fontsize=13)
plt.tight_layout()
plt.savefig(str(FIGURES / "6a_resize_demo.png"), dpi=100, bbox_inches="tight")
plt.show()

print(f"Original size : {_demo_img_a.size}")
print(f"After resize  : {_resized_img_a.size}")


### 6.2 — Normalisation

Pixel values are normalised using **ImageNet statistics**:

| Channel | Mean | Std |
|---|---|---|
| Red | 0.485 | 0.229 |
| Green | 0.456 | 0.224 |
| Blue | 0.406 | 0.225 |

**Why these values:** ResNet-50 was pre-trained on ImageNet with these exact statistics baked into its first-layer weights. Applying the same normalisation maps our inputs into the feature space the backbone expects, making transfer learning valid. The EDA (Section 5.1.3) confirmed that Food-101's per-channel means are within ≤ 0.02 of ImageNet norms.

In [35]:

# ── 6.2  ImageNet normalisation constants ─────────────────────────────────────
# ResNet-50 was pre-trained on ImageNet with these per-channel statistics.
# Applying the same normalisation at inference maps inputs into the expected
# feature space of the frozen backbone.

IMAGENET_MEAN = [0.485, 0.456, 0.406]   # per-channel pixel mean
IMAGENET_STD  = [0.229, 0.224, 0.225]   # per-channel pixel std

print(f"{'Channel':<10} {'Mean':>8}  {'Std':>8}")
print("-" * 30)
for ch, m, s in zip(["Red", "Green", "Blue"], IMAGENET_MEAN, IMAGENET_STD):
    print(f"{ch:<10} {m:>8.3f}  {s:>8.3f}")

# Verify on a single sample image (note: single-image stats ≠ dataset mean/std)
_norm_t = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                     T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
_demo_img_b, _ = ds_train[5]
_normed = _norm_t(_demo_img_b)
print(f"\nNormalized tensor shape         : {tuple(_normed.shape)}")
print(f"Per-channel mean (single image) : {_normed.mean(dim=[1,2]).numpy().round(3)}")
print(f"  (→ approaches 0 in aggregate across full dataset)")
print(f"\n✓ 6.2 — IMAGENET_MEAN and IMAGENET_STD defined.")


### 6.3 — Data Augmentation choices

Augmentation is applied to the **training set only**. Val and test receive only deterministic resize + centre-crop + normalise to ensure reproducible evaluation.

| Transform | Justification |
|---|---|
| `Resize(256)` → `RandomCrop(224)` | Stochastic crop simulates different dish framings and aspect ratios across photographers |
| `RandomHorizontalFlip` | Food photos are horizontally symmetric — a pizza looks identical when flipped |
| `RandomRotation(15°)` | Dishes are photographed at varying angles; ±15° covers realistic variation without distorting recognisable textures |
| `ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05)` | Restaurant lighting and white-balance vary widely; small `hue=0.05` avoids changing food colour identity |
| `Normalize(ImageNet mean/std)` | Required for valid transfer from ResNet-50; applied last so statistics match the [0,1] tensor range |

In [36]:
# ── 6.3  Define train and val/test transforms + show augmentation effect ───────
# IMAGENET_MEAN and IMAGENET_STD are defined in 6.2 above.

train_transform = T.Compose([
    T.Resize(256),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.2),   # erase random rectangle — helps scratch CNN generalise
])

val_test_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f"train_transform    : {len(train_transform.transforms)} steps")
print(f"val_test_transform : {len(val_test_transform.transforms)} steps  (no stochastic augmentation)")
print()
for i, t in enumerate(train_transform.transforms):
    print(f"  train [{i}]: {t}")

# Show 8 augmented versions of the same image (display without normalisation)
_aug_display = T.Compose([
    T.Resize(256),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
])
_center_crop = T.Compose([T.Resize(256), T.CenterCrop(224)])

_demo_img_c, _demo_cls_c = ds_train[100]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    _img = _center_crop(_demo_img_c) if i == 0 else _aug_display(_demo_img_c)
    ax.imshow(_img)
    ax.set_title("Original (centred)" if i == 0 else f"Augmented #{i}")
    ax.axis("off")
fig.suptitle(f"6.3 — Same image, 7 different augmentations  ({ds_train.classes[_demo_cls_c]})",
             fontsize=13)
plt.tight_layout()
plt.savefig(str(FIGURES / "6c_augmentation_demo.png"), dpi=100, bbox_inches="tight")
plt.show()
print(f"\n✓ 6.3 — transforms defined.")


### 6.4 — Wrap index arrays into `Dataset` objects

The pre-computed `_idx_train`, `_idx_val`, `_idx_test` index arrays (from Section 4.2) are wrapped in a `_Food101Combined` Dataset that maps each combined index back to the correct underlying Food-101 split (`train` or `test`) and applies the appropriate transform on the fly.

In [37]:

# ── 6.4  Wrap pre-computed index arrays into Dataset objects ──────────────────
# train_transform and val_test_transform were defined in 6.3 above.
# The 70/15/15 index split (_idx_train, _idx_val, _idx_test) was created in
# Section 4.2 before EDA. We just apply transforms here — no re-splitting.

_base_train = tv_datasets.Food101(root=FOOD101_ROOT, split="train", download=False,
                                   transform=None)
_base_test  = tv_datasets.Food101(root=FOOD101_ROOT, split="test",  download=False,
                                   transform=None)

class _Food101Combined(torch.utils.data.Dataset):
    """Maps a combined index (0..100,999) back to the correct Food-101 split."""
    def __init__(self, indices, transform):
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        if idx < _n_food_tr:
            img, label = _base_train[idx]
        else:
            img, label = _base_test[idx - _n_food_tr]
        return self.transform(img), label

img_train_ds = _Food101Combined(_idx_train, train_transform)
img_val_ds   = _Food101Combined(_idx_val,   val_test_transform)
img_test_ds  = _Food101Combined(_idx_test,  val_test_transform)

print(f"img_train_ds : {len(img_train_ds):,} samples")
print(f"img_val_ds   : {len(img_val_ds):,} samples")
print(f"img_test_ds  : {len(img_test_ds):,} samples")
print(f"\n✓ 6.4 — Dataset objects created.")


### 6.5 — Build `DataLoader` objects

DataLoaders wrap the three `Dataset` objects with batching, shuffling (training only), and pinned memory for fast GPU transfer. `IMG_BATCH = 64` keeps GPU memory usage manageable for ResNet-50 on a standard Colab T4.

In [38]:

# ── 6.5  DataLoaders ──────────────────────────────────────────────────────────
IMG_BATCH = 64

img_train_loader = DataLoader(img_train_ds, batch_size=IMG_BATCH, shuffle=True,
                               num_workers=2, pin_memory=True)
img_val_loader   = DataLoader(img_val_ds,   batch_size=IMG_BATCH, shuffle=False,
                               num_workers=2, pin_memory=True)
img_test_loader  = DataLoader(img_test_ds,  batch_size=IMG_BATCH, shuffle=False,
                               num_workers=2, pin_memory=True)

_total = len(_idx_train) + len(_idx_val) + len(_idx_test)
print(f"{'Split':<10} {'Samples':>8}  {'%':>6}  {'Batches':>8}")
print("-" * 40)
for _name, _ds, _ldr in [("train", img_train_ds, img_train_loader),
                          ("val",   img_val_ds,   img_val_loader),
                          ("test",  img_test_ds,  img_test_loader)]:
    print(f"{_name:<10} {len(_ds):>8,}  {len(_ds)/_total*100:>5.1f}%  {len(_ldr):>8,}")
print(f"{'total':<10} {_total:>8,}  100.0%")

imgs, labels = next(iter(img_train_loader))
assert imgs.shape == (IMG_BATCH, 3, 224, 224), f"Unexpected shape: {imgs.shape}"
print(f"\nBatch shape : {imgs.shape}  (B, C, H, W)")
print(f"Label range : {labels.min().item()} – {labels.max().item()}")
print(f"\n✓ 6.5 — image DataLoaders ready.")


## Section 7 — Text Preprocessing and DataLoaders

All preprocessing steps run on the training split only to prevent data leakage.
The vocabulary, GloVe matrix, class weights, and Word2Vec model are built exclusively from df_train.

| Sub-section | What |
|---|---|
| **7.1** | Tokenisation strategy (word-level, justified) |
| **7.2** | Vocabulary building and OOV handling |
| **7.3** | Padding and truncation to MAX_SEQ_LEN |
| **7.4** | GloVe embedding matrix initialisation |
| **7.5** | Class weights for imbalanced grape labels |
| **7.6** | Word2Vec fine-tuning on wine reviews |
| **7.7** | Grape centroid embeddings in shared flavour space |
| **7.8** | Text DataLoaders |
| **7.9** | Batch shape verification |

**Correct step order (data leakage prevention)**

`
1. Train / val / test split          <- must come FIRST (Section 4)
2. Tokenise all splits
3. Build vocabulary on TRAINING tokens only
4. Load GloVe -> build embedding matrix from training vocab
5. Compute class weights from TRAINING labels only
6. Fine-tune Word2Vec on TRAINING reviews only          (Section 7.6)
7. Compute grape embeddings from TRAINING reviews only  (Section 7.7)
8. Pad / truncate to MAX_SEQ_LEN
9. Build DataLoaders
`


### 7.1 — Tokenisation strategy

**Strategy chosen: word-level tokenisation** (lowercase → punctuation stripped → whitespace split).

**Justification:**
- Wine tasting notes use a specific but stable vocabulary (*tannins*, *terroir*, *minerality*, *oak*). Word-level tokens map directly to meaningful tasting descriptors; subword (BPE) would unnecessarily fragment terms like *full-bodied* or *well-rounded*.
- GloVe and Word2Vec pre-trained embeddings are word-level, so word tokenisation lets us initialise the embedding layer directly from pre-trained vectors without any adaptation.
- Character-level tokenisation would require a much deeper model to reconstruct word meanings and is not well-suited for an LSTM trained on a limited domain corpus.

In [39]:

import string

# ── 7.1  Extract texts + tokenise ─────────────────────────────────────────────
# Unpack DataFrames into plain lists
txt_train = df_train["review_text"].tolist()
lbl_train = df_train["grape_class"].map(GRAPE_TO_IDX).tolist()
txt_val   = df_val["review_text"].tolist()
lbl_val   = df_val["grape_class"].map(GRAPE_TO_IDX).tolist()
txt_test  = df_test["review_text"].tolist()
lbl_test  = df_test["grape_class"].map(GRAPE_TO_IDX).tolist()

print(f"{'Split':<8} {'Samples':>10}")
print("-" * 20)
print(f"{'train':<8} {len(txt_train):>10,}")
print(f"{'val':<8} {len(txt_val):>10,}")
print(f"{'test':<8} {len(txt_test):>10,}")

# Tokenise: lowercase → strip punctuation → whitespace split
_punct = str.maketrans("", "", string.punctuation)

def tokenise(text):
    return text.lower().translate(_punct).split()

tok_train = [tokenise(t) for t in txt_train]
tok_val   = [tokenise(t) for t in txt_val]
tok_test  = [tokenise(t) for t in txt_test]

# Show sample tokenisation
print("\n--- Sample tokenisation ---")
for i in range(2):
    print(f"\nOriginal : {txt_train[i][:120]}")
    print(f"Tokens   : {tok_train[i][:12]} ...")
print(f"\nAvg tokens/review — train: {sum(len(t) for t in tok_train)/len(tok_train):.1f} "
      f"  val: {sum(len(t) for t in tok_val)/len(tok_val):.1f}")


### 7.2 — Vocabulary and OOV handling

- Vocabulary is built **from training tokens only** (no val/test leakage).
- Tokens appearing fewer than `MIN_FREQ = 3` times in training are discarded to remove noise and reduce the embedding matrix size.
- Two special tokens are reserved: **`<PAD>` (index 0)** and **`<UNK>` (index 1)**. Any token absent from the training vocabulary — whether in val/test or at deployment — is mapped to `<UNK>`.
- OOV rates for train and val are printed below.

In [40]:

# ── 7.2  Build vocabulary on TRAINING tokens only ────────────────────────────
# Val and test see the same vocab but do not influence it (no leakage).
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
MIN_FREQ  = 3   # discard tokens appearing fewer than 3 times in training

_train_counts = Counter(tok for sent in tok_train for tok in sent)
_vocab_words  = [w for w, c in _train_counts.items() if c >= MIN_FREQ]

VOCAB      = {PAD_TOKEN: 0, UNK_TOKEN: 1}
VOCAB.update({w: i + 2 for i, w in enumerate(_vocab_words)})
VOCAB_SIZE = len(VOCAB)

pct_unk_train = sum(1 for s in tok_train for w in s if w not in VOCAB) / \
                max(sum(len(s) for s in tok_train), 1) * 100
pct_unk_val   = sum(1 for s in tok_val   for w in s if w not in VOCAB) / \
                max(sum(len(s) for s in tok_val),   1) * 100

print(f"Vocabulary size     : {VOCAB_SIZE:,}  (MIN_FREQ = {MIN_FREQ})")
print(f"Special tokens      : PAD='{PAD_TOKEN}' (idx {VOCAB[PAD_TOKEN]}),  UNK='{UNK_TOKEN}' (idx {VOCAB[UNK_TOKEN]})")
print(f"OOV rate (train)    : {pct_unk_train:.2f}%  (tokens mapped to <UNK>)")
print(f"OOV rate (val)      : {pct_unk_val:.2f}%")
print(f"\nTop-10 most frequent training tokens:")
for _w, _c in sorted(_train_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {_w:<20} {_c:>6,}")


### 7.3 — Padding and truncation

All sequences are padded or right-truncated to `MAX_SEQ_LEN` (set to the **95th percentile** of training review lengths in Section 5.2.1). This means:
- Exactly **5 % of training reviews** are truncated — only tail words are dropped.
- Shorter reviews are right-padded with `<PAD>` tokens (index 0).
- The BiLSTM uses `pack_padded_sequence` to ignore padding positions during the forward pass.
- The exact `MAX_SEQ_LEN` value and the truncation percentage are printed below.

In [41]:

# ── 7.3  Encode and pad to MAX_SEQ_LEN ────────────────────────────────────────
# MAX_SEQ_LEN was set to the 95th percentile of training lengths in Section 5.2.1.
# Sequences longer than MAX_SEQ_LEN are right-truncated; shorter ones are
# right-padded with PAD_TOKEN (index 0).

def encode_and_pad(tokens, max_len):
    ids = [VOCAB.get(w, VOCAB[UNK_TOKEN]) for w in tokens[:max_len]]
    ids += [VOCAB[PAD_TOKEN]] * (max_len - len(ids))
    return ids

X_train = np.array([encode_and_pad(t, MAX_SEQ_LEN) for t in tok_train], dtype=np.int64)
X_val   = np.array([encode_and_pad(t, MAX_SEQ_LEN) for t in tok_val],   dtype=np.int64)
X_test  = np.array([encode_and_pad(t, MAX_SEQ_LEN) for t in tok_test],  dtype=np.int64)

pct_trunc_train = sum(len(t) > MAX_SEQ_LEN for t in tok_train) / len(tok_train) * 100
pct_trunc_val   = sum(len(t) > MAX_SEQ_LEN for t in tok_val)   / len(tok_val)   * 100

print(f"MAX_SEQ_LEN       = {MAX_SEQ_LEN}  (p95 of training review lengths)")
print(f"X_train shape     : {X_train.shape}  (N × MAX_SEQ_LEN)")
print(f"X_val   shape     : {X_val.shape}")
print(f"X_test  shape     : {X_test.shape}")
print(f"Truncated (train) : {pct_trunc_train:.1f}%  (reviews longer than MAX_SEQ_LEN)")
print(f"Truncated (val)   : {pct_trunc_val:.1f}%")

# Show a sample encoding
print(f"\n--- Sample encoding (first review) ---")
print(f"Tokens (first 8)  : {tok_train[0][:8]}")
print(f"Encoded (first 8) : {X_train[0][:8].tolist()}")


### 7.4 — GloVe embedding matrix

The BiLSTM embedding layer is initialised from **GloVe `glove-wiki-gigaword-100`** (100-d, ~66 MB download via gensim, cached locally after the first run).

| Property | Value |
|---|---|
| Embedding dimension | 100-d |
| Corpus | Wikipedia + Gigaword |
| Coverage | ~85–90 % of training vocabulary tokens |
| Tokens not in GloVe | Initialised to zero; learned from scratch during BiLSTM training |

The embedding matrix has shape `(VOCAB_SIZE × 100)` and is passed to `nn.Embedding.from_pretrained()` in Section 11 with `freeze=False` so vectors are fine-tuned during training.

In [42]:

# ── 7.4  GloVe embedding matrix ───────────────────────────────────────────────
# Download once (~66 MB) via gensim; cached locally afterward.
print("Loading GloVe 100-d  (downloads ~66 MB on first run) ...")
glove_kv  = gensim_api.load("glove-wiki-gigaword-100")
EMBED_DIM = 100

embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
n_found = 0
for word, idx in VOCAB.items():
    if word in glove_kv:
        embedding_matrix[idx] = glove_kv[word]
        n_found += 1

print(f"Embedding matrix shape : {embedding_matrix.shape}  (VOCAB_SIZE × EMBED_DIM)")
print(f"GloVe coverage         : {n_found / max(VOCAB_SIZE, 1) * 100:.1f}%  of vocab tokens found in GloVe")
print("✓ 7.4 — GloVe embedding matrix ready.")


### 7.5 — Class weights

Class weights correct for label imbalance across the 15 grape classes in WineSensed. Common varieties (Cabernet Sauvignon, Chardonnay) dominate; rarer ones (Viognier, Chenin Blanc) need upweighting.

- Computed via `sklearn compute_class_weight("balanced", ...)` on **training labels only** — no leakage.
- Passed to `nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))` inside the BiLSTM training loop.
- Tensor kept on CPU here and moved to the training device when used.

In [43]:

from sklearn.utils.class_weight import compute_class_weight

# ── 7.5  Class weights (TRAINING labels only) ─────────────────────────────────
_cw = compute_class_weight("balanced", classes=np.arange(len(GRAPE_CLASSES)), y=lbl_train)
# Keep on CPU — move to DEVICE only when passed to criterion inside a training cell.
CLASS_WEIGHTS = torch.tensor(_cw, dtype=torch.float32)

print(f"Class weights shape : {CLASS_WEIGHTS.shape}  ({len(GRAPE_CLASSES)} classes)")
print(f"  min = {CLASS_WEIGHTS.min():.3f}   max = {CLASS_WEIGHTS.max():.3f}\n")
print(f"  {'Grape':<25} {'Weight':>8}")
print(f"  {'-'*35}")
for grape, w in zip(GRAPE_CLASSES, CLASS_WEIGHTS):
    print(f"  {grape:<25} {w:>8.3f}")
print("\n✓ 7.5 — CLASS_WEIGHTS ready.")


### 7.6 — Word2Vec fine-tuning

We start from the **Google News Word2Vec model** (300-d, 3 million vocabulary) — it already understands everyday food language: *tomato*, *fatty*, *smoky*, *creamy*.

We then fine-tune it on the **824k WineSensed training reviews** so wine-specific words (*Sangiovese*, *cassis*, *tannic*, *terroir*, *mineral*) are pulled into the same vector space as the food keywords. This cross-vocabulary alignment is what makes cosine similarity between food keywords and grape names meaningful.

| Sub-step | What |
|---|---|
| 7.6.1 | Install gensim + load Google News base model (~1.6 GB, cached after first run) |
| 7.6.2 | Build wine sentence corpus from `tok_train` |
| 7.6.3 | Extend vocab + fine-tune 5 epochs (~10–15 min CPU) |
| 7.6.4 | Save fine-tuned model → `weights/word2vec_wine.model` + Drive |

> **Note:** `~/gensim-data/` is large (~1.6 GB). Do not commit it to git.


#### 7.6.1 — Install gensim & define paths

Install [gensim](https://radimrehurek.com/gensim/) if not already present, import the required classes, and define local + Drive output paths for the fine-tuned model.

In [ ]:
import subprocess, sys

try:
    import gensim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gensim>=4.3"])
    import gensim

import gensim.downloader as gensim_api
from gensim.models import Word2Vec

W2V_MODEL_PATH = WEIGHTS / "word2vec_wine.model"
W2V_DRIVE_PATH = Path(WEIGHTS_DIR) / "word2vec_wine.model"

print(f"gensim {gensim.__version__}")
print(f"  local  → {W2V_MODEL_PATH}")
print(f"  Drive  → {W2V_DRIVE_PATH}")


#### 7.6.2 — Build wine sentence corpus

`tok_train` is a list of token-string sequences produced in Section 7.1.  
We drop `<pad>` and `<unk>` tokens and feed the cleaned sentences to Word2Vec.

In [ ]:
_skip = {PAD_TOKEN, UNK_TOKEN}
wine_sentences = [
    [tok for tok in seq if tok not in _skip]
    for seq in tok_train if seq
]
print(f"Corpus  : {len(wine_sentences):,} sentences")
print(f"Avg len : {sum(len(s) for s in wine_sentences)/len(wine_sentences):.1f} tokens/sentence")


#### 7.6.3 — Load Google News base model & fine-tune

We download the **Google News Word2Vec** base (300-d, ~1.6 GB, cached after first run), initialise the vocabulary with pre-trained vectors, and fine-tune for **5 epochs** on wine reviews.  
> Expect ~10–15 min on CPU; ~2 min on GPU.

In [ ]:
import glob, shutil

print("Loading Google News base model (~1.6 GB, cached after first run)...")
_base_kv = gensim_api.load("word2vec-google-news-300")  # KeyedVectors

# Build a trainable Word2Vec wrapper
w2v_model = Word2Vec(vector_size=300, window=5, min_count=2, workers=4, seed=SEED)
w2v_model.build_vocab(wine_sentences)

# Expand vocab with Google News words, then copy pre-trained vectors
w2v_model.build_vocab([list(_base_kv.key_to_index.keys())], update=True)
# vectors_lockf defaults to all-ones (fully trainable) — no manual override needed
for _w in _base_kv.key_to_index:
    if _w in w2v_model.wv:
        w2v_model.wv[_w] = _base_kv[_w]

print(f"Vocab size : {len(w2v_model.wv):,} words")
print(f"Fine-tuning Word2Vec  |  epochs=5  |  device: CPU")
print("-" * 60)

N_EPOCHS    = 5
history_w2v = {"loss": []}
_best_loss  = float("inf")
_prev_loss  = 0.0

def _sync_w2v_to_drive(ckpt_path):
    """Copy all gensim files for a given .model path to Drive."""
    if not IN_COLAB:
        return
    for _f in glob.glob(str(ckpt_path) + "*"):
        shutil.copy(_f, Path(WEIGHTS_DIR) / Path(_f).name)

for epoch in range(1, N_EPOCHS + 1):
    w2v_model.train(
        wine_sentences,
        total_examples=len(wine_sentences),
        epochs=1,
        compute_loss=True,
    )
    _total      = w2v_model.get_latest_training_loss()
    _epoch_loss = _total - _prev_loss
    _prev_loss  = _total
    history_w2v["loss"].append(_epoch_loss)

    # Save epoch checkpoint (resume-safe after Colab disconnect)
    _ckpt_path = WEIGHTS / f"word2vec_wine_ep{epoch:02d}.model"
    w2v_model.save(str(_ckpt_path))
    _sync_w2v_to_drive(_ckpt_path)

    # Keep a separate "best" copy
    _marker = ""
    if _epoch_loss < _best_loss:
        _best_loss = _epoch_loss
        w2v_model.save(str(W2V_MODEL_PATH))
        _sync_w2v_to_drive(W2V_MODEL_PATH)
        _marker = "  ← best"

    print(f"Epoch {epoch:>2}/{N_EPOCHS}  loss={_epoch_loss:.4f}{_marker}")

print(f"\n✓ Fine-tuning complete  |  final vocab: {len(w2v_model.wv):,} words")
print(f"  Best loss : {_best_loss:.4f}  |  word2vec_wine.model = best checkpoint")


#### 7.6.4 — Save fine-tuned model

Persist the model locally and, if running in Colab, sync to Google Drive so it survives session restarts.

In [ ]:
import shutil, glob

# Save gensim model (writes .model + auxiliary .npy files)
w2v_model.save(str(W2V_MODEL_PATH))

# Collect ALL files written by gensim for this model
_w2v_files = glob.glob(str(W2V_MODEL_PATH) + "*")

if IN_COLAB:
    W2V_DRIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    for _src in _w2v_files:
        _dst = Path(WEIGHTS_DIR) / Path(_src).name
        shutil.copy(_src, _dst)
    print(f"✓ Saved {len(_w2v_files)} file(s) to Drive:")
    for _src in sorted(_w2v_files):
        print(f"    {Path(_src).name}")
else:
    print(f"✓ Saved {len(_w2v_files)} file(s) locally:")
    for _src in sorted(_w2v_files):
        print(f"    {Path(_src).name}")


### 7.7 — Grape centroid embeddings

Each grape variety is represented as a **centroid vector** in the 300-d Word2Vec space:
the average of all Word2Vec vectors for tokens appearing in training reviews for that grape.

This centroid is the bridge between food flavour keywords and grape recommendations.
When a food query arrives (e.g., pizza -> 	omato, cheesy, savory, herby),
its keyword vectors are averaged and compared against all 15 grape centroids via cosine similarity.

| Sub-step | What |
|---|---|
| **7.7.1** | Define paths and load fine-tuned Word2Vec model |
| **7.7.2** | Compute per-grape centroid vectors (shape: 15 x 300) |
| **7.7.3** | Save grape embeddings locally and to Drive |
| **7.7.4** | Sanity check: centroid similarity heatmap + food flavor probes |


#### 7.7.1 — Define paths & load fine-tuned model

Define output paths and load the Word2Vec model produced in 7.6.4.  
If `w2v_model` is already in scope from 7.6.3, the load step is skipped.

In [ ]:
import numpy as np
from gensim.models import Word2Vec

GRAPE_EMB_PATH       = WEIGHTS / "grape_embeddings.npy"
GRAPE_EMB_DRIVE_PATH = Path(WEIGHTS_DIR) / "grape_embeddings.npy"

if "w2v_model" not in dir():
    _src = W2V_DRIVE_PATH if (IN_COLAB and W2V_DRIVE_PATH.exists()) else W2V_MODEL_PATH
    w2v_model = Word2Vec.load(str(_src))
    print(f"  → Loaded from {_src}")
else:
    print("  → w2v_model already in scope")

_wv = w2v_model.wv
print(f"  Vocab size : {len(_wv):,}")
print(f"  Paths      : local={GRAPE_EMB_PATH}")


#### 7.7.2 — Compute per-grape centroid vectors

For each of the 15 grape classes, collect all training review tokens and average their Word2Vec vectors.  
Result: `grape_embeddings` — shape `(15, 300)`, one centroid per grape in the shared flavor space.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Step 1: fit TF-IDF on all training reviews to get global IDF scores ──
# High IDF = rare/distinctive word  |  Low IDF = common wine vocabulary
_all_reviews = df_train["review_text"].fillna("").astype(str).tolist()
_tfidf = TfidfVectorizer(max_features=50_000, sublinear_tf=True)
_tfidf.fit(_all_reviews)
_idf_map = dict(zip(_tfidf.get_feature_names_out(), _tfidf.idf_))
print(f"TF-IDF vocab size : {len(_idf_map):,}")

# ── Step 2: build TF-IDF-weighted Word2Vec centroid per grape ────────────
grape_embeddings = np.zeros((len(GRAPE_CLASSES), 300), dtype=np.float32)

print(f"\nBuilding TF-IDF weighted centroids for {len(GRAPE_CLASSES)} grape classes...")
print("-" * 65)

for g_idx, grape in enumerate(GRAPE_CLASSES):
    _reviews = df_train.loc[df_train["grape_class"] == grape, "review_text"].dropna().tolist()
    _vecs, _weights = [], []
    for _rev in _reviews:
        for _word in str(_rev).lower().split():
            if _word in _wv and _word in _idf_map:
                _vecs.append(_wv[_word])
                _weights.append(_idf_map[_word])  # higher IDF = more distinctive
    if _vecs:
        _w = np.array(_weights, dtype=np.float32)
        _w /= _w.sum()
        grape_embeddings[g_idx] = np.average(_vecs, axis=0, weights=_w)
        print(f"  {grape:<28} {len(_reviews):>6,} reviews  {len(_vecs):>8,} tokens")
    else:
        print(f"  WARNING: no vectors found for {grape}")

print(f"\n✓ TF-IDF weighted centroids  |  shape: {grape_embeddings.shape}")


#### 7.7.3 — Save grape embeddings

Persist the `(15, 300)` matrix locally and sync to Drive if running in Colab.

In [ ]:
np.save(GRAPE_EMB_PATH, grape_embeddings)

if IN_COLAB:
    import shutil
    shutil.copy(GRAPE_EMB_PATH, GRAPE_EMB_DRIVE_PATH)
    print(f"✓ Saved to Drive : {GRAPE_EMB_DRIVE_PATH}")
else:
    print(f"✓ Saved locally  : {GRAPE_EMB_PATH}")

_size_kb = GRAPE_EMB_PATH.stat().st_size / 1024
print(f"  File size    : {_size_kb:.1f} KB")
print(f"  Shape        : {np.load(GRAPE_EMB_PATH).shape}")


#### 7.7.4 — Sanity check: grape centroid similarity + food probe

Two complementary checks:

1. **Centroid similarity heatmap** — cosine similarity between all 15 grape centroids. Good embeddings show clearly separated grapes with only stylistically related pairs (e.g., Shiraz/Primitivo) clustering together.
2. **Food flavor probes** — 8 random `food_flavor_table` entries queried against grape centroids using their `classic` keywords. This tests the actual downstream pipeline (food → flavor keywords → grape centroid matching).

In [ ]:
import random
from numpy.linalg import norm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

# ─────────────────────────────────────────────────────────────────────────────
# Part 1 — Grape centroid similarity heatmap
# ─────────────────────────────────────────────────────────────────────────────
_sim_matrix = cosine_similarity(grape_embeddings)   # (15, 15)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    _sim_matrix,
    xticklabels=GRAPE_CLASSES,
    yticklabels=GRAPE_CLASSES,
    annot=True, fmt=".2f", cmap="YlOrRd",
    vmin=0.85, vmax=1.0,          # centroids live in a tight cone — zoom in
    linewidths=0.4, ax=ax,
    annot_kws={"size": 7},
)
ax.set_title("Grape Centroid Cosine Similarity\n(TF-IDF weighted Word2Vec, 300-d)",
             fontsize=13, pad=12)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("figures/grape_centroid_heatmap.png", dpi=150)
plt.show()
print("Heatmap saved to figures/grape_centroid_heatmap.png")

# ─────────────────────────────────────────────────────────────────────────────
# Part 2 — Food flavor probes (the actual pipeline)
# ─────────────────────────────────────────────────────────────────────────────
def _nearest_grape(words):
    _vecs = [_wv[w] for w in words if w in _wv]
    if not _vecs:
        return "no vector"
    _q    = np.mean(_vecs, axis=0)
    _sims = grape_embeddings @ _q / (
        norm(grape_embeddings, axis=1) * norm(_q) + 1e-8)
    return GRAPE_CLASSES[int(np.argmax(_sims))]

# load food_flavor_table if not already in scope
try:
    food_flavor_table
except NameError:
    import json as _json
    _fft_path = WEIGHTS.parent / "data" / "food_flavor_table.json"
    food_flavor_table = {k: v for k, v in
                         _json.load(open(_fft_path)).items()
                         if k != "_meta"}

random.seed(42)
_foods = random.sample([k for k in food_flavor_table if k != "_meta"], min(8, len(food_flavor_table) - 1))

col1, col2, col3 = 26, 38, 22
print(f"\n{'Food':{col1}} {'Classic keywords':{col3}} Nearest grape (classic)")
print("-" * 90)
for _food in _foods:
    _kws   = food_flavor_table[_food].get("classic", [])
    _grape = _nearest_grape(_kws)
    _kw_str = ", ".join(_kws[:4])
    print(f"{_food.replace('_', ' '):{col1}} {_kw_str:{col3}} {_grape}")

print("\n\u2713 7.7 complete  |  grape_embeddings ready for Section 13 (recommend).")


### 7.8 — Text DataLoaders

A `ReviewDataset` wraps the integer-encoded sequences (`X_train/val/test`) and class labels so PyTorch's `DataLoader` can yield shuffled mini-batches.

| Loader | Shuffle | Purpose |
|---|---|---|
| `txt_train_loader` | Yes | BiLSTM training |
| `txt_val_loader` | No | Per-epoch validation |
| `txt_test_loader` | No | Final evaluation (used once) |

Batch size `TXT_BATCH = 64` matches the image pipeline (`IMG_BATCH = 64`) so the joint model can zip both loaders.

In [44]:

# ── 7.8  Text DataLoaders ─────────────────────────────────────────────────────
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):          return len(self.X)
    def __getitem__(self, i):   return self.X[i], self.y[i]

TXT_BATCH = 64
txt_train_loader = DataLoader(ReviewDataset(X_train, lbl_train), batch_size=TXT_BATCH, shuffle=True)
txt_val_loader   = DataLoader(ReviewDataset(X_val,   lbl_val),   batch_size=TXT_BATCH, shuffle=False)
txt_test_loader  = DataLoader(ReviewDataset(X_test,  lbl_test),  batch_size=TXT_BATCH, shuffle=False)

print(f"TXT_BATCH        : {TXT_BATCH}")
print(f"Train batches    : {len(txt_train_loader)}")
print(f"Val   batches    : {len(txt_val_loader)}")
print(f"Test  batches    : {len(txt_test_loader)}")
print("✓ 7.8 — text DataLoaders ready.")


### 7.9 — Batch verification

Confirm that the DataLoaders return the expected tensor shapes and dtypes before proceeding to model training.

In [45]:

# ── 7.9  Batch verification ───────────────────────────────────────────────────
xb, yb = next(iter(txt_train_loader))
print(f"Text batch  xb.shape : {xb.shape}   dtype={xb.dtype}   (B × MAX_SEQ_LEN)")
print(f"Label batch yb.shape : {yb.shape}   dtype={yb.dtype}")
assert xb.shape == (TXT_BATCH, MAX_SEQ_LEN), f"Unexpected shape: {xb.shape}"
assert yb.shape == (TXT_BATCH,)
print("\n✓ Section 7 complete — text preprocessing and DataLoaders ready.")


## Section 8 — CNN: Custom Architecture (Trained from Scratch)

**Requires GPU — run in Google Colab.**

A VGG-style CNN trained entirely from random initialisation on Food-101. This serves as the
baseline image model; its test accuracy is compared directly against the transfer-learning
approach in Section 9.

| Sub-section | What |
|---|---|
| **8.1** | Model architecture (ConvBlock + CNNScratch) and forward-pass sanity check |
| **8.2** | Shared training utilities (	rain_one_epoch, evaluate, criterion / optimiser / scheduler) |
| **8.3** | Training loop with per-epoch checkpoints and early stopping |
| **8.3.1** | Resume training (same session, mid-run interruption) |
| **8.3.2** | Restore from checkpoint (after kernel restart) |
| **8.4** | Learning curves (loss + accuracy) |
| **8.5** | Final test evaluation: accuracy, macro F1, classification report |
| **8.5.1** | Test evaluation visualisations: per-class bars, confusion pairs, heatmap |
| **8.6** | Prediction visualiser: image + top-10 confidence bars |

**Architecture**

Four VGG-style convolutional blocks (two convs per block) followed by a three-layer classifier head:

`
Block 1: Conv2d(3→64)    x2 → BN → ReLU → MaxPool2d(2) → Dropout2d(0.1)  [224→112]
Block 2: Conv2d(64→128)   x2 → BN → ReLU → MaxPool2d(2) → Dropout2d(0.1)  [112→56]
Block 3: Conv2d(128→256)  x2 → BN → ReLU → MaxPool2d(2) → Dropout2d(0.1)  [ 56→28]
Block 4: Conv2d(256→512)  x2 → BN → ReLU → MaxPool2d(2) → Dropout2d(0.1)  [ 28→14]
         AdaptiveAvgPool2d(1x1) → Flatten(512)
Head:    FC(512→1024) → ReLU → Dropout(0.5)
         FC(1024→512) → ReLU → Dropout(0.3)
         FC(512→101)
`

**Training protocol**

| Hyperparameter | Value |
|---|---|
| Epochs (max) | 20 |
| Early stopping patience | 3 (on val loss) |
| Optimiser | Adam, lr = 1e-3 |
| LR scheduler | ReduceLROnPlateau (factor=0.5, patience=2) |
| Loss | CrossEntropyLoss |
| Checkpoint | Saved every epoch (incl. history) — safe to resume after Colab disconnect |


In [46]:
# ── 8.1  Model architecture ───────────────────────────────────────────────────
# VGG-style double-conv blocks: two convolutions at each spatial scale
# before pooling — the biggest single improvement for a 101-class problem.
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),  # 2nd conv same scale
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),   # light spatial dropout after each pool
        )
    def forward(self, x): return self.block(x)

class CNNScratch(nn.Module):
    def __init__(self, n_classes=101):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   64),    # 224 → 112
            ConvBlock(64,  128),   # 112 →  56
            ConvBlock(128, 256),   #  56 →  28
            ConvBlock(256, 512),   #  28 →  14   ← new 4th block
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

    def encode(self, x):
        """Return 512-d feature vector (used by joint model)."""
        return self.features(x).squeeze(-1).squeeze(-1)

cnn_scratch = CNNScratch().to(DEVICE)
total_params = sum(p.numel() for p in cnn_scratch.parameters())
train_params = sum(p.numel() for p in cnn_scratch.parameters() if p.requires_grad)
print(f"CNN scratch — total params : {total_params:,}")
print(f"              trainable    : {train_params:,}")

# Sanity-check forward pass
_dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
assert cnn_scratch(_dummy).shape == (2, 101)
print("Forward pass  : OK → shape (2, 101)")
print("✓ 8.1 — CNNScratch model ready.")


### 8.2 — Training utilities

Two shared helper functions used by both CNN-from-scratch (Section 8) and ResNet-50 (Section 9):

| Function | Role |
|---|---|
| `train_one_epoch(model, loader, criterion, optimizer)` | One full pass over the training set; returns mean loss and accuracy |
| `evaluate(model, loader, criterion)` | No-grad validation/test pass; returns mean loss and accuracy |

`criterion_cnn = CrossEntropyLoss()` is used for both models. The optimiser and scheduler are model-specific and defined below.

In [51]:

# ── 8.2  Training utilities ───────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

criterion_cnn     = nn.CrossEntropyLoss()
optimizer_scratch = Adam(cnn_scratch.parameters(), lr=1e-3)
scheduler_scratch = ReduceLROnPlateau(optimizer_scratch, factor=0.5, patience=2)

print("✓ 8.2 — training utilities ready.")
print(f"  criterion  : CrossEntropyLoss")
print(f"  optimiser  : Adam  lr=1e-3")
print(f"  scheduler  : ReduceLROnPlateau  factor=0.5  patience=2")


### 8.3 — Training loop with checkpoints and early stopping

- Trains for up to **20 epochs**; stops early if val loss does not improve for **3 consecutive epochs**.
- A full checkpoint (`model_state` + `optimizer_state` + epoch + val loss) is saved after **every epoch** — training can be safely resumed after a Colab disconnect by reloading the latest checkpoint.
- The best model weights (lowest val loss) are also saved to `cnn_scratch.pt` and used in 8.5 for the final test evaluation.

In [ ]:
# ── 8.3  Training loop ────────────────────────────────────────────────────────
MAX_EPOCHS    = 20
PATIENCE      = 3
best_val_loss = float("inf")
patience_ctr  = 0
history_scratch = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

print(f"Training started — CNN from scratch  |  max {MAX_EPOCHS} epochs  |  device: {DEVICE}")
print("-" * 75)

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(cnn_scratch, img_train_loader, criterion_cnn, optimizer_scratch)
    vl_loss, vl_acc = evaluate(cnn_scratch, img_val_loader, criterion_cnn)
    scheduler_scratch.step(vl_loss)

    history_scratch["train_loss"].append(tr_loss)
    history_scratch["val_loss"].append(vl_loss)
    history_scratch["train_acc"].append(tr_acc)
    history_scratch["val_acc"].append(vl_acc)

    print(f"Epoch {epoch:>2}/{MAX_EPOCHS}  "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.3f}")

    # Save full checkpoint every epoch (resume-safe after Colab disconnect)
    save_checkpoint({
        "epoch":           epoch,
        "model_state":     cnn_scratch.state_dict(),
        "optimizer_state": optimizer_scratch.state_dict(),
        "val_loss":        vl_loss,
        "history":         history_scratch,
    }, f"cnn_scratch_ckpt_ep{epoch:02d}.pt")

    # Keep separate best-model weights
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_ctr  = 0
        save_checkpoint(cnn_scratch.state_dict(), "cnn_scratch_best.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch}  (patience={PATIENCE}).")
            break

print(f"\nTraining complete.  Best model saved as: cnn_scratch_best.pt")


### 8.3.1 — Resume training from checkpoint

If the Colab session is still active and only the training loop cell was interrupted, reload the latest epoch checkpoint and continue from where training stopped.


In [ ]:
# ── 8.3-RESUME  Continue training from checkpoint ────────────────────────────
# Run this cell INSTEAD of re-running the full training loop above.
# Make sure cells 8.1 and 8.2 (model + optimizer definitions) have been run first.

RESUME_EPOCH = 11                         # last completed epoch

# ── 1. Load checkpoint ───────────────────────────────────────────────────────
_ckpt = load_checkpoint(f"cnn_scratch_ckpt_ep{RESUME_EPOCH:02d}.pt")
cnn_scratch.load_state_dict(_ckpt["model_state"])
optimizer_scratch.load_state_dict(_ckpt["optimizer_state"])
print(f"Loaded checkpoint: epoch {_ckpt['epoch']}, val_loss={_ckpt['val_loss']:.4f}")

# ── 2. Restore history from checkpoint ───────────────────────────────────────
history_scratch = _ckpt["history"]
print(f"History restored: {len(history_scratch['train_loss'])} epochs")

# ── 3. Restore early-stopping state ─────────────────────────────────────────
best_val_loss = min(history_scratch["val_loss"])
patience_ctr  = 0                                  # last saved epoch was the best

# ── 4. Continue training from next epoch ─────────────────────────────────────
MAX_EPOCHS = 20
PATIENCE   = 3

print(f"\nResuming — CNN from scratch  |  epochs {RESUME_EPOCH+1}–{MAX_EPOCHS}  |  device: {DEVICE}")
print("-" * 75)

for epoch in range(RESUME_EPOCH + 1, MAX_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(cnn_scratch, img_train_loader, criterion_cnn, optimizer_scratch)
    vl_loss, vl_acc = evaluate(cnn_scratch, img_val_loader, criterion_cnn)
    scheduler_scratch.step(vl_loss)

    history_scratch["train_loss"].append(tr_loss)
    history_scratch["val_loss"].append(vl_loss)
    history_scratch["train_acc"].append(tr_acc)
    history_scratch["val_acc"].append(vl_acc)

    print(f"Epoch {epoch:>2}/{MAX_EPOCHS}  "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.3f}")

    save_checkpoint({
        "epoch":           epoch,
        "model_state":     cnn_scratch.state_dict(),
        "optimizer_state": optimizer_scratch.state_dict(),
        "val_loss":        vl_loss,
        "history":         history_scratch,
    }, f"cnn_scratch_ckpt_ep{epoch:02d}.pt")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_ctr  = 0
        save_checkpoint(cnn_scratch.state_dict(), "cnn_scratch_best.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch}  (patience={PATIENCE}).")
            break

print(f"\nTraining complete.  Best model saved as: cnn_scratch_best.pt")


### 8.3.2 — Restore training state from checkpoint

After a kernel restart (e.g. Colab disconnect), training results and helper functions are lost from memory but checkpoints are still saved on Google Drive. This cell restores **everything** needed by Sections 8.4–8.6 and Section 9:

- `evaluate()` and `train_one_epoch()` functions  
- `criterion_cnn` (CrossEntropyLoss)  
- `history_scratch` (training curves)  
- `cnn_scratch` best weights

In [52]:
# ── 8.3b  Restore all Section 8 state from saved checkpoints ─────────────────
# Use this cell after a kernel restart to reload training history, model weights,
# helper functions, and loss criterion from Drive — without re-running training.
import glob, os

# ── 1. Training utilities (evaluate, train_one_epoch, criterion) ──────────────
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

criterion_cnn = nn.CrossEntropyLoss()

# ── 2. Restore history_scratch from last checkpoint ───────────────────────────
_scratch_ckpts = sorted(glob.glob(str(WEIGHTS / "cnn_scratch_ckpt_ep*.pt")))
if not _scratch_ckpts:
    raise FileNotFoundError(
        "No cnn_scratch checkpoint found in weights folder — run Section 8.3 first."
    )

_last_ckpt = load_checkpoint(os.path.basename(_scratch_ckpts[-1]))
history_scratch = _last_ckpt["history"]

# ── 3. Load best model weights into cnn_scratch ──────────────────────────────
cnn_scratch.load_state_dict(load_checkpoint("cnn_scratch_best.pt"))

print(f"✓ 8.3b — All Section 8 state restored:")
print(f"  Functions : evaluate, train_one_epoch")
print(f"  Criterion : criterion_cnn (CrossEntropyLoss)")
print(f"  History   : {len(history_scratch['train_loss'])} epochs from {os.path.basename(_scratch_ckpts[-1])}")
print(f"  Model     : cnn_scratch loaded with best weights (cnn_scratch_best.pt)")
print(f"  Best val loss: {min(history_scratch['val_loss']):.4f}")
print(f"  Best val acc:  {max(history_scratch['val_acc']):.4f}")

### 8.4 — Learning curves

Plot training and validation loss and accuracy over epochs. Overfitting shows up as a widening gap between train and val curves. The saved figure is used in the final report.

In [53]:

# ── 8.4  Learning curves ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_scratch["train_loss"], label="train", marker="o", markersize=3)
ax1.plot(history_scratch["val_loss"],   label="val",   marker="o", markersize=3)
ax1.set_title("CNN Scratch — Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(history_scratch["train_acc"], label="train", marker="o", markersize=3)
ax2.plot(history_scratch["val_acc"],   label="val",   marker="o", markersize=3)
ax2.set_title("CNN Scratch — Accuracy")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
save_figure(fig, "cnn_scratch_curves.png")
plt.show()


### 8.5 — Final test evaluation

The **test set is used exactly once** here. Loads the best checkpoint (`cnn_scratch.pt`) and computes:

| Metric | Description |
|---|---|
| Test accuracy | Overall top-1 accuracy across all 101 classes |
| Macro F1 | Unweighted mean F1 across all classes — not biased by class frequency |
| Classification report | Per-class precision, recall, F1, support |

Results will be compared against ResNet-50 (Section 9) in the discussion.

In [54]:
from sklearn.metrics import classification_report, f1_score

# ── 8.5  Final test evaluation (test set used once) ───────────────────────────
cnn_scratch.load_state_dict(load_checkpoint("cnn_scratch_best.pt"))
_, test_acc = evaluate(cnn_scratch, img_test_loader, criterion_cnn)

all_preds, all_labels = [], []
cnn_scratch.eval()
with torch.no_grad():
    for imgs, labels in img_test_loader:
        preds = cnn_scratch(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

macro_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"CNN Scratch — Test accuracy : {test_acc:.4f}")
print(f"              Macro F1      : {macro_f1:.4f}\n")
print(classification_report(all_labels, all_preds,
                             target_names=ds_train.classes,
                             zero_division=0))
print(f"✓ Section 8 complete — best weights: cnn_scratch_best.pt")


### 8.5.1 — Test evaluation: visualisations

Three charts complementing the classification report in 8.5:

1. **Top-20 best / worst classes** — per-class accuracy bars
2. **Top-15 most confused pairs** — bar chart of the highest off-diagonal confusion counts
3. **Focused confusion heatmap** — only the classes involved in the top-15 confused pairs, row-normalised so each cell is a recall fraction


In [55]:
# ── 8.5b  Test evaluation visualisations — CNN Scratch ───────────────────────
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

_CLASS_NAMES = ds_train.classes   # 101 Food-101 class names

# Build the full confusion matrix once
sc_cm = confusion_matrix(all_labels, all_preds, labels=list(range(101)))

# ── 1. Per-class accuracy bars ────────────────────────────────────────────────
sc_per_class_acc = sc_cm.diagonal() / sc_cm.sum(axis=1).clip(min=1)

sc_sorted_idx  = np.argsort(sc_per_class_acc)
sc_worst20_idx = sc_sorted_idx[:20]
sc_best20_idx  = sc_sorted_idx[-20:][::-1]

fig, (ax_best, ax_worst) = plt.subplots(1, 2, figsize=(18, 7))

ax_best.barh(
    [_CLASS_NAMES[i].replace("_", " ") for i in sc_best20_idx[::-1]],
    sc_per_class_acc[sc_best20_idx[::-1]] * 100, color="#2ca02c")
ax_best.set_xlim(0, 100)
ax_best.set_xlabel("Accuracy (%)")
ax_best.set_title("Top-20 best recognised classes — CNN Scratch")
for i, v in enumerate(sc_per_class_acc[sc_best20_idx[::-1]] * 100):
    ax_best.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=8)

ax_worst.barh(
    [_CLASS_NAMES[i].replace("_", " ") for i in sc_worst20_idx[::-1]],
    sc_per_class_acc[sc_worst20_idx[::-1]] * 100, color="#d62728")
ax_worst.set_xlim(0, 100)
ax_worst.set_xlabel("Accuracy (%)")
ax_worst.set_title("Top-20 worst recognised classes — CNN Scratch")
for i, v in enumerate(sc_per_class_acc[sc_worst20_idx[::-1]] * 100):
    ax_worst.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=8)

plt.tight_layout()
save_figure(fig, "cnn_scratch_per_class_accuracy.png")
plt.show()

# ── 2. Top-15 most confused pairs — bar chart ─────────────────────────────────
sc_cm_no_diag = sc_cm.copy()
np.fill_diagonal(sc_cm_no_diag, 0)

TOP_PAIRS     = 15
sc_flat_idx   = np.argsort(sc_cm_no_diag.ravel())[-(TOP_PAIRS):][::-1]
sc_true_idx   = sc_flat_idx // 101
sc_pred_idx   = sc_flat_idx % 101
sc_counts     = sc_cm_no_diag.ravel()[sc_flat_idx]
sc_pair_labels = [
    f"{_CLASS_NAMES[t].replace('_', ' ')} →\n{_CLASS_NAMES[p].replace('_', ' ')}"
    for t, p in zip(sc_true_idx, sc_pred_idx)
]

fig2, ax2 = plt.subplots(figsize=(12, 6))
ax2.barh(sc_pair_labels[::-1], sc_counts[::-1], color="#ff7f0e")
ax2.set_xlabel("Number of test images misclassified")
ax2.set_title(f"Top-{TOP_PAIRS} most confused class pairs — CNN Scratch\n(true class → predicted class)")
for i, v in enumerate(sc_counts[::-1]):
    ax2.text(v + 0.2, i, str(int(v)), va="center", fontsize=9)
plt.tight_layout()
save_figure(fig2, "cnn_scratch_confused_pairs.png")
plt.show()

# ── 3. Focused confusion heatmap — classes from top confused pairs only ────────
# Collect unique class indices that appear in the top pairs
sc_focus_idx = sorted(set(sc_true_idx.tolist() + sc_pred_idx.tolist()))
sc_focus_labels = [_CLASS_NAMES[i].replace("_", " ") for i in sc_focus_idx]

# Slice the full CM to keep only those rows/columns, then row-normalise
sc_cm_focus = sc_cm[np.ix_(sc_focus_idx, sc_focus_idx)]
sc_cm_focus_norm = (sc_cm_focus.astype(float)
                    / sc_cm_focus.sum(axis=1, keepdims=True).clip(min=1))

n_focus = len(sc_focus_idx)
fig3, ax3 = plt.subplots(figsize=(max(10, n_focus * 0.7), max(8, n_focus * 0.6)))
sns.heatmap(
    sc_cm_focus_norm,
    ax=ax3,
    annot=True, fmt=".2f",
    cmap="Blues",
    linewidths=0.5,
    xticklabels=sc_focus_labels,
    yticklabels=sc_focus_labels,
    cbar_kws={"label": "Fraction of true class (recall)"},
    vmin=0, vmax=1,
)
ax3.set_xlabel("Predicted class", fontsize=11)
ax3.set_ylabel("True class", fontsize=11)
ax3.set_title(
    f"Confusion heatmap — top-confused classes — CNN Scratch\n"
    f"(classes appearing in the top-{TOP_PAIRS} most confused pairs)",
    fontsize=12, pad=10)
plt.setp(ax3.get_xticklabels(), rotation=45, ha="right", fontsize=9)
plt.setp(ax3.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
save_figure(fig3, "cnn_scratch_confusion_matrix.png")
plt.show()

print("✓ 8.5b — visualisations complete.")


### 8.6 — Prediction visualiser

For each sample: the original food image is shown on the left with its true label; the right panel shows a **top-10 softmax confidence bar chart**.

- **Green bar** — model's top-1 prediction is correct (`✓ Correct`)
- **Red bar** — model's top-1 prediction is wrong (`✗ Wrong`)
- Dashed line at 50 % marks the decision boundary reference
- Caption shows: sample index · true class · predicted class · confidence %

`N_SAMPLES` controls how many panels are shown. The function `plot_predictions()` can be reused for ResNet-50 in Section 9.

In [56]:

# ── 8.6  Prediction visualiser ────────────────────────────────────────────────
import random
from torchvision.transforms.functional import to_pil_image

N_SAMPLES   = 6          # number of panels to display
TOP_K       = 10         # top-K classes to show in the bar chart
CLASS_NAMES = ds_train.classes   # list of 101 Food-101 class names

def plot_predictions(model, dataset, n=N_SAMPLES, top_k=TOP_K, seed=42,
                     save_filename=None):
    """
    Show n panels: food image (left) + top-k confidence bar chart (right).
    Green bar = correct prediction, red bar = wrong.
    Pass save_filename="my_plot.png" to save the figure.
    """
    model.eval()
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), n)

    ncols = 2
    fig, axes = plt.subplots(n, ncols, figsize=(11, 4 * n),
                             gridspec_kw={"width_ratios": [1, 2]})

    with torch.no_grad():
        for row, idx in enumerate(indices):
            img_tensor, true_label = dataset[idx]

            # ── forward pass ─────────────────────────────────────────────────
            logits = model(img_tensor.unsqueeze(0).to(DEVICE))          # (1, 101)
            probs  = torch.softmax(logits, dim=1).squeeze().cpu()       # (101,)
            top_probs, top_ids = probs.topk(top_k)                      # (top_k,)

            pred_label   = top_ids[0].item()
            confidence   = top_probs[0].item() * 100
            is_correct   = (pred_label == true_label)

            # ── left panel: food image ────────────────────────────────────────
            ax_img = axes[row, 0]
            # Denormalise: ImageNet mean/std → [0, 1]
            img_show = img_tensor.clone()
            for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
                img_show[c] = img_show[c] * s + m
            img_show = img_show.clamp(0, 1)
            ax_img.imshow(img_show.permute(1, 2, 0).numpy())
            ax_img.set_title(f"True label: {CLASS_NAMES[true_label].replace('_', ' ')}",
                             fontsize=10)
            ax_img.axis("off")

            # ── right panel: confidence bar chart ─────────────────────────────
            ax_bar = axes[row, 1]
            bar_labels = [CLASS_NAMES[i.item()].replace("_", " ") for i in top_ids]
            bar_values = [p.item() * 100 for p in top_probs]

            bar_color  = "#2ca02c" if is_correct else "#d62728"  # green / red
            bar_colors = [bar_color if i == 0 else "#aec7e8" for i in range(top_k)]

            bars = ax_bar.barh(bar_labels[::-1], bar_values[::-1],
                               color=bar_colors[::-1])
            ax_bar.axvline(50, color="gray", linestyle="--", linewidth=0.8)
            ax_bar.set_xlim(0, 100)
            ax_bar.set_xlabel("Confidence (%)")

            tick_mark = "✓ Correct" if is_correct else "✗ Wrong"
            ax_bar.set_title(
                f"Prediction: {CLASS_NAMES[pred_label].replace('_', ' ')} "
                f"({confidence:.1f}%)\n{tick_mark}",
                fontsize=10,
                color=bar_color,
            )

            # value labels on bars
            for bar, val in zip(bars[::-1], bar_values):
                if val > 1:
                    ax_bar.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                                f"{val:.1f}%", va="center", fontsize=8)

            # ── caption ───────────────────────────────────────────────────────
            caption = (f"Sample index: {idx}  |  "
                       f"True: {CLASS_NAMES[true_label].replace('_', ' ')}  |  "
                       f"Predicted: {CLASS_NAMES[pred_label].replace('_', ' ')}  |  "
                       f"Confidence: {confidence:.1f}%")
            fig.text(0.01, 1 - (row + 1) / n + 0.01 / n,
                     caption, fontsize=7.5, color="#444444",
                     transform=fig.transFigure)

    plt.tight_layout(rect=[0, 0.0, 1, 1])
    if save_filename:
        save_figure(fig, save_filename)
    plt.show()


# ── Run on the test set with the best CNN-scratch weights ─────────────────────
cnn_scratch.load_state_dict(load_checkpoint("cnn_scratch_best.pt"))
plot_predictions(
    model=cnn_scratch,
    dataset=img_test_ds,
    n=N_SAMPLES,
    seed=42,
    save_filename="cnn_scratch_predictions.png",
)
print("✓ 8.6 — prediction visualiser complete.")


## Section 9 — CNN: ResNet-50 (Transfer Learning)

**Requires GPU — run in Google Colab.**

ResNet-50 pre-trained on ImageNet is fine-tuned on Food-101 using a two-phase strategy.
Reuses 	rain_one_epoch, evaluate, and criterion_cnn defined in Section 8.2.

| Sub-section | What |
|---|---|
| **9.1** | Build model — replace final FC layer for 101 classes, freeze backbone |
| **9.2** | Phase A — train head only (5 epochs, LR 1e-3) |
| **9.3** | Phase B — unfreeze layer4 (10 epochs, LR 1e-4) |
| **9.3.1** | Resume Phase B (same session, mid-run interruption) |
| **9.3.2** | Restore from checkpoint (after kernel restart) |
| **9.4** | Learning curves (Phase A + B combined) |
| **9.5** | Final test evaluation: accuracy, macro F1, classification report |
| **9.5.1** | Test evaluation visualisations: per-class bars, confusion pairs, heatmap |
| **9.6** | Prediction visualiser |
| **9.7** | CNN Scratch vs. ResNet-50 discussion |

**Two-phase fine-tuning rationale**

| Phase | Layers trained | Epochs | LR | Why |
|---|---|---|---|---|
| A | Classification head only (frozen backbone) | 5 | 1e-3 | Finds a good head initialisation before touching backbone weights |
| B | Head + layer4 unfrozen | 10 | 1e-4 | Adapts high-level features to food domain without corrupting low-level ImageNet representations |

Checkpoints saved every epoch; best model (lowest Phase B val loss) saved to cnn_resnet50_best.pt.


In [57]:
# ── 9.1  Build ResNet-50 ──────────────────────────────────────────────────────
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze all layers
for p in resnet50.parameters():
    p.requires_grad = False

# Replace classification head → 101 classes
resnet50.fc = nn.Linear(2048, 101)
resnet50    = resnet50.to(DEVICE)

frozen  = sum(p.numel() for p in resnet50.parameters() if not p.requires_grad)
trained = sum(p.numel() for p in resnet50.parameters() if p.requires_grad)
print(f"ResNet-50 — frozen: {frozen:,}  trainable: {trained:,}")

assert resnet50(torch.randn(2, 3, 224, 224).to(DEVICE)).shape == (2, 101)
print("Forward pass: OK → shape (2, 101)")


### 9.2 — Phase A: train head only (epochs 1–5, LR 1e-3)

The backbone is fully frozen. Only the replacement FC layer is trained. This prevents noisy early gradients from corrupting pre-trained ImageNet features.


In [58]:
# ── 9.2  Phase A — frozen backbone, head only ─────────────────────────────────
# Prerequisites check — catches missing definitions early with clear guidance
_missing = [name for name in
    ["train_one_epoch", "evaluate", "criterion_cnn",
     "img_train_loader", "img_val_loader"]
    if name not in dir() and name not in vars()]
if _missing:
    raise RuntimeError(
        f"\n❌ Missing from kernel: {_missing}\n"
        "   After reconnecting, re-run these sections IN ORDER before Section 9.2:\n"
        "   1. Section 1  (1.0 → 1.3) — imports, paths, Drive mount, helpers\n"
        "   2. Section 2  — Food-101 raw data loading (ds_train, ds_test)\n"
        "   3. Section 4.2 — index arrays (_idx_train, _idx_val, _idx_test)\n"
        "   4. Section 6  — image transforms + DataLoaders\n"
        "   5. Section 8.1 — CNNScratch architecture + criterion_cnn\n"
        "   6. Section 8.2 — train_one_epoch, evaluate\n"
        "   7. Section 9.1 — rebuild ResNet-50\n"
        "   Then re-run this cell."
    )

opt_rn   = Adam(resnet50.fc.parameters(), lr=1e-3)
sch_rn   = ReduceLROnPlateau(opt_rn, factor=0.5, patience=2)
hist_rn  = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_rn_a_loss = float("inf")

print("── Phase A (frozen backbone, head only) ──")
for epoch in range(1, 6):
    tr_loss, tr_acc = train_one_epoch(resnet50, img_train_loader, criterion_cnn, opt_rn)
    vl_loss, vl_acc = evaluate(resnet50, img_val_loader, criterion_cnn)
    sch_rn.step(vl_loss)
    hist_rn["train_loss"].append(tr_loss); hist_rn["val_loss"].append(vl_loss)
    hist_rn["train_acc"].append(tr_acc);  hist_rn["val_acc"].append(vl_acc)
    save_checkpoint(
        {"epoch": epoch, "model_state": resnet50.state_dict(),
         "optimizer_state": opt_rn.state_dict(), "val_loss": vl_loss,
         "history": hist_rn},
        f"resnet50_phaseA_ckpt_ep{epoch:02d}.pt"
    )
    if vl_loss < best_rn_a_loss:
        best_rn_a_loss = vl_loss
        save_checkpoint(resnet50.state_dict(), "resnet50_phaseA_best.pt")
    print(f"  Ep {epoch} | train {tr_loss:.4f}/{tr_acc:.3f} | val {vl_loss:.4f}/{vl_acc:.3f}")

print("\n✓ Phase A complete — best Phase A weights saved as resnet50_phaseA_best.pt")


### 9.3 — Phase B: unfreeze layer4 (epochs 6–15, LR 1e-4)

Unfreezes ResNet-50's final convolutional block (layer4) and continues training at a 10× lower learning rate. Low LR is essential here — layer4 already contains useful high-level features; we adapt rather than overwrite them.

Run this cell when continuing in the same session immediately after Phase A. If the session was interrupted, use **9.3b** to restore from checkpoint first.


In [59]:
# ── 9.3  Phase B — unfreeze layer4 ───────────────────────────────────────────
# Run this cell only if Phase A finished in the SAME session.
# After a disconnect → use 9.3-RESUME below instead.

for p in resnet50.layer4.parameters():
    p.requires_grad = True

opt_rn_b     = Adam(filter(lambda p: p.requires_grad, resnet50.parameters()), lr=1e-4)
sch_rn_b     = ReduceLROnPlateau(opt_rn_b, factor=0.5, patience=2)
best_rn_loss = float("inf")

print("── Phase B (layer4 + head unfrozen) ──")
for epoch in range(6, 16):
    tr_loss, tr_acc = train_one_epoch(resnet50, img_train_loader, criterion_cnn, opt_rn_b)
    vl_loss, vl_acc = evaluate(resnet50, img_val_loader, criterion_cnn)
    sch_rn_b.step(vl_loss)
    hist_rn["train_loss"].append(tr_loss); hist_rn["val_loss"].append(vl_loss)
    hist_rn["train_acc"].append(tr_acc);  hist_rn["val_acc"].append(vl_acc)
    save_checkpoint(
        {"epoch": epoch, "model_state": resnet50.state_dict(),
         "optimizer_state": opt_rn_b.state_dict(), "val_loss": vl_loss,
         "history": hist_rn},
        f"resnet50_phaseB_ckpt_ep{epoch:02d}.pt"
    )
    print(f"  Ep {epoch} | train {tr_loss:.4f}/{tr_acc:.3f} | val {vl_loss:.4f}/{vl_acc:.3f}")
    if vl_loss < best_rn_loss:
        best_rn_loss = vl_loss
        save_checkpoint(resnet50.state_dict(), "cnn_resnet50_best.pt")

print("\n✓ Phase B complete — best model saved as cnn_resnet50_best.pt")


### 9.3.1 — Resume Phase B from checkpoint

If Phase B was interrupted mid-run, reload the latest Phase B epoch checkpoint and continue from where training stopped (session must still be active).


In [60]:
# ── 9.3-RESUME  Restore and continue Phase B after a disconnect ───────────────
# After reconnecting: re-run Sections 1.0 → 1.3 first, then run THIS cell.
# DO NOT run the 9.3 cell above — it assumes Phase A model is still in memory.

# ── Step 1: Rebuild model architecture ────────────────────────────────────────
resnet50 = models.resnet50(weights=None)
resnet50.fc = nn.Linear(2048, 101)
for p in resnet50.layer4.parameters():
    p.requires_grad = True
resnet50 = resnet50.to(DEVICE)

# ── Step 2: Find the latest Phase B checkpoint (fall back to Phase A best) ────
resume_from   = None
start_epoch   = 6
best_rn_loss  = float("inf")
ckpt          = None

for ep in range(15, 5, -1):
    try:
        ckpt = load_checkpoint(f"resnet50_phaseB_ckpt_ep{ep:02d}.pt")
        resume_from = ep
        break
    except FileNotFoundError:
        pass

if resume_from is None:
    print("  No Phase B checkpoint found — loading Phase A best to start Phase B")
    # Try to restore Phase A history from its last checkpoint
    try:
        _ckpt_a = load_checkpoint("resnet50_phaseA_ckpt_ep05.pt")
        hist_rn = _ckpt_a["history"]
        print(f"  Phase A history restored: {len(hist_rn['train_loss'])} epochs")
    except (FileNotFoundError, KeyError):
        hist_rn = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        print("  Phase A history not available — starting history fresh")
    resnet50.load_state_dict(load_checkpoint("resnet50_phaseA_best.pt"))
    start_epoch  = 6
    best_rn_loss = float("inf")
else:
    resnet50.load_state_dict(ckpt["model_state"])
    start_epoch  = ckpt["epoch"] + 1
    best_rn_loss = ckpt["val_loss"]
    hist_rn      = ckpt["history"]
    print(f"  Resuming Phase B from epoch {start_epoch} "
          f"(last val loss: {best_rn_loss:.4f}, "
          f"history: {len(hist_rn['train_loss'])} epochs)")

# ── Step 3: Resume Phase B ────────────────────────────────────────────────────
opt_rn_b = Adam(filter(lambda p: p.requires_grad, resnet50.parameters()), lr=1e-4)
sch_rn_b = ReduceLROnPlateau(opt_rn_b, factor=0.5, patience=2)
if ckpt is not None and "optimizer_state" in ckpt:
    opt_rn_b.load_state_dict(ckpt["optimizer_state"])

print(f"\n── Phase B resumed (epochs {start_epoch}–15) ──")
for epoch in range(start_epoch, 16):
    tr_loss, tr_acc = train_one_epoch(resnet50, img_train_loader, criterion_cnn, opt_rn_b)
    vl_loss, vl_acc = evaluate(resnet50, img_val_loader, criterion_cnn)
    sch_rn_b.step(vl_loss)
    hist_rn["train_loss"].append(tr_loss); hist_rn["val_loss"].append(vl_loss)
    hist_rn["train_acc"].append(tr_acc);  hist_rn["val_acc"].append(vl_acc)
    save_checkpoint(
        {"epoch": epoch, "model_state": resnet50.state_dict(),
         "optimizer_state": opt_rn_b.state_dict(), "val_loss": vl_loss,
         "history": hist_rn},
        f"resnet50_phaseB_ckpt_ep{epoch:02d}.pt"
    )
    print(f"  Ep {epoch} | train {tr_loss:.4f}/{tr_acc:.3f} | val {vl_loss:.4f}/{vl_acc:.3f}")
    if vl_loss < best_rn_loss:
        best_rn_loss = vl_loss
        save_checkpoint(resnet50.state_dict(), "cnn_resnet50_best.pt")

print("\n✓ Phase B complete — best model saved as cnn_resnet50_best.pt")


### 9.3.2 — Restore training state from checkpoint

After a kernel restart (e.g. Colab disconnect), training results and helper functions are lost from memory but checkpoints are still saved on Google Drive. This cell restores **everything** needed by Sections 9.4–9.5:

- `evaluate()` and `train_one_epoch()` functions
- `criterion_cnn` (CrossEntropyLoss)
- `hist_rn` (training curves — Phase A + Phase B combined)
- `resnet50` best weights (`cnn_resnet50_best.pt`)

> **When to use:** Run this cell instead of 9.3 / 9.3-RESUME when you only need to produce learning curves, evaluation, or visualisations from a previously completed training run.

In [ ]:
# ── 9.3b  Restore all Section 9 state from saved checkpoints ─────────────────
# Use this cell after a kernel restart to reload training history, model weights,
# helper functions, and loss criterion from Drive — without re-running training.
import glob, os

# ── 1. Training utilities (evaluate, train_one_epoch, criterion) ─────────────
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

criterion_cnn = nn.CrossEntropyLoss()

# ── 2. Restore hist_rn — Phase B checkpoint first, fall back to Phase A ───────
_rn_b_ckpts = sorted(glob.glob(str(WEIGHTS / 'resnet50_phaseB_ckpt_ep*.pt')))
_rn_a_ckpts = sorted(glob.glob(str(WEIGHTS / 'resnet50_phaseA_ckpt_ep*.pt')))

if _rn_b_ckpts:
    _last_ckpt_rn = load_checkpoint(os.path.basename(_rn_b_ckpts[-1]))
    _source = f'Phase B  ({os.path.basename(_rn_b_ckpts[-1])})'
elif _rn_a_ckpts:
    _last_ckpt_rn = load_checkpoint(os.path.basename(_rn_a_ckpts[-1]))
    _source = f'Phase A  ({os.path.basename(_rn_a_ckpts[-1])})'
else:
    raise FileNotFoundError(
        'No ResNet-50 checkpoint found in weights folder — run Section 9.2 / 9.3 first.'
    )

hist_rn = _last_ckpt_rn['history']

# ── 3. Rebuild ResNet-50 architecture and load best weights ────────────────
resnet50 = models.resnet50(weights=None)
resnet50.fc = nn.Linear(2048, 101)
resnet50 = resnet50.to(DEVICE)
resnet50.load_state_dict(load_checkpoint('cnn_resnet50_best.pt'))

print('✓ 9.3b — All Section 9 state restored:')
print(f'  Functions : evaluate, train_one_epoch')
print(f'  Criterion : criterion_cnn (CrossEntropyLoss)')
print(f'  History   : {len(hist_rn["train_loss"])} epochs  ← {_source}')
print(f'  Model     : resnet50 loaded with best weights (cnn_resnet50_best.pt)')
print(f'  Best val loss: {min(hist_rn["val_loss"]):.4f}')
print(f'  Best val acc:  {max(hist_rn["val_acc"]):.4f}')


### 9.4 — Learning curves

Plot combined Phase A + Phase B training and validation loss/accuracy. The phase boundary (epoch 5) is marked with a vertical dashed line to show where backbone unfreezing occurs.


In [61]:
# ── 9.4  Learning curves ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_rn["train_loss"], label="train"); ax1.plot(hist_rn["val_loss"], label="val")
# Phase A→B boundary only visible if hist_rn contains both phases
if len(hist_rn["train_loss"]) > 5:
    ax1.axvline(4.5, color="gray", linestyle=":", label="Phase A→B")
ax1.set_title("ResNet-50 — Loss"); ax1.legend()
ax2.plot(hist_rn["train_acc"],  label="train"); ax2.plot(hist_rn["val_acc"],  label="val")
if len(hist_rn["train_acc"]) > 5:
    ax2.axvline(4.5, color="gray", linestyle=":", label="Phase A→B")
ax2.set_title("ResNet-50 — Accuracy"); ax2.legend()
plt.tight_layout()
save_figure(fig, "resnet50_curves.png")
plt.show()


### 9.5 — Final test evaluation

The **test set is used exactly once** here. Loads cnn_resnet50_best.pt (best Phase B val loss) and computes test accuracy, macro F1, and a full per-class classification report. Results are compared against Section 8 in 9.7.


In [62]:
# ── 9.5  Test evaluation ──────────────────────────────────────────────────────
resnet50.load_state_dict(load_checkpoint("cnn_resnet50_best.pt"))
_, rn_test_acc = evaluate(resnet50, img_test_loader, criterion_cnn)

rn_preds, rn_labels = [], []
resnet50.eval()
with torch.no_grad():
    for imgs, labels in img_test_loader:
        preds = resnet50(imgs.to(DEVICE)).argmax(1).cpu()
        rn_preds.extend(preds.tolist())
        rn_labels.extend(labels.tolist())

rn_f1 = f1_score(rn_labels, rn_preds, average="macro")
print(f"\nResNet-50   — Test accuracy: {rn_test_acc:.4f}  Macro F1: {rn_f1:.4f}")

# Section 8.5 results — only available if Section 8.5 was run in this session
if "test_acc" in dir() or "test_acc" in vars():
    print(f"CNN Scratch — Test accuracy: {test_acc:.4f}  Macro F1: {macro_f1:.4f}")
else:
    print("CNN Scratch — results not in memory (re-run Section 8.5 to compare)")
print("\n✓ Section 9 complete — best ResNet-50 weights: cnn_resnet50_best.pt")


### 9.5.1 — Test evaluation: visualisations

Three charts complementing the classification report in 9.5:

1. **Top-20 best / worst classes** — per-class accuracy bars
2. **Top-15 most confused pairs** — bar chart of highest off-diagonal confusion counts
3. **Focused confusion heatmap** — only the classes involved in the top-15 confused pairs, row-normalised so each cell is a recall fraction


In [63]:
# ── 9.5b  Test evaluation visualisations — ResNet-50 ─────────────────────────
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

CLASS_NAMES = ds_train.classes   # 101 Food-101 class names

# Build the full confusion matrix once
cm = confusion_matrix(rn_labels, rn_preds, labels=list(range(101)))

# ── 1. Per-class accuracy bars ────────────────────────────────────────────────
per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)

sorted_idx   = np.argsort(per_class_acc)
worst20_idx  = sorted_idx[:20]
best20_idx   = sorted_idx[-20:][::-1]

fig, (ax_best, ax_worst) = plt.subplots(1, 2, figsize=(18, 7))

ax_best.barh(
    [CLASS_NAMES[i].replace("_", " ") for i in best20_idx[::-1]],
    per_class_acc[best20_idx[::-1]] * 100, color="#2ca02c")
ax_best.set_xlim(0, 100)
ax_best.set_xlabel("Accuracy (%)")
ax_best.set_title("Top-20 best recognised classes — ResNet-50")
for i, v in enumerate(per_class_acc[best20_idx[::-1]] * 100):
    ax_best.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=8)

ax_worst.barh(
    [CLASS_NAMES[i].replace("_", " ") for i in worst20_idx[::-1]],
    per_class_acc[worst20_idx[::-1]] * 100, color="#d62728")
ax_worst.set_xlim(0, 100)
ax_worst.set_xlabel("Accuracy (%)")
ax_worst.set_title("Top-20 worst recognised classes — ResNet-50")
for i, v in enumerate(per_class_acc[worst20_idx[::-1]] * 100):
    ax_worst.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=8)

plt.tight_layout()
save_figure(fig, "resnet50_per_class_accuracy.png")
plt.show()

# ── 2. Top-15 most confused pairs — bar chart ─────────────────────────────────
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

TOP_PAIRS   = 15
flat_idx    = np.argsort(cm_no_diag.ravel())[-TOP_PAIRS:][::-1]
true_idx    = flat_idx // 101
pred_idx    = flat_idx % 101
counts      = cm_no_diag.ravel()[flat_idx]
pair_labels = [
    f"{CLASS_NAMES[t].replace('_', ' ')} →\n{CLASS_NAMES[p].replace('_', ' ')}"
    for t, p in zip(true_idx, pred_idx)
]

fig2, ax2 = plt.subplots(figsize=(12, 6))
ax2.barh(pair_labels[::-1], counts[::-1], color="#ff7f0e")
ax2.set_xlabel("Number of test images misclassified")
ax2.set_title(f"Top-{TOP_PAIRS} most confused class pairs — ResNet-50\n(true class → predicted class)")
for i, v in enumerate(counts[::-1]):
    ax2.text(v + 0.2, i, str(int(v)), va="center", fontsize=9)
plt.tight_layout()
save_figure(fig2, "resnet50_confused_pairs.png")
plt.show()

# ── 3. Focused confusion heatmap — classes from top confused pairs only ────────
# Collect unique class indices that appear in the top pairs
focus_idx    = sorted(set(true_idx.tolist() + pred_idx.tolist()))
focus_labels = [CLASS_NAMES[i].replace("_", " ") for i in focus_idx]

# Slice the full CM to keep only those rows/columns, then row-normalise
cm_focus = cm[np.ix_(focus_idx, focus_idx)]
cm_focus_norm = (cm_focus.astype(float)
                 / cm_focus.sum(axis=1, keepdims=True).clip(min=1))

n_focus = len(focus_idx)
fig3, ax3 = plt.subplots(figsize=(max(10, n_focus * 0.7), max(8, n_focus * 0.6)))
sns.heatmap(
    cm_focus_norm,
    ax=ax3,
    annot=True, fmt=".2f",
    cmap="Blues",
    linewidths=0.5,
    xticklabels=focus_labels,
    yticklabels=focus_labels,
    cbar_kws={"label": "Fraction of true class (recall)"},
    vmin=0, vmax=1,
)
ax3.set_xlabel("Predicted class", fontsize=11)
ax3.set_ylabel("True class", fontsize=11)
ax3.set_title(
    f"Confusion heatmap — top-confused classes — ResNet-50\n"
    f"(classes appearing in the top-{TOP_PAIRS} most confused pairs)",
    fontsize=12, pad=10)
plt.setp(ax3.get_xticklabels(), rotation=45, ha="right", fontsize=9)
plt.setp(ax3.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
save_figure(fig3, "resnet50_confusion_matrix.png")
plt.show()

print("✓ 9.5b — visualisations complete.")


### 9.6 — Prediction visualiser

Same layout as Section 8.6: food image alongside a top-10 softmax confidence bar chart. Green bar = correct top-1 prediction, red bar = wrong. Reuses plot_predictions() defined in Section 8.6.


In [64]:
# ── 9.6  Prediction visualiser — ResNet-50 ────────────────────────────────────
import random

# Define constants / function if Section 8.6 was not run in this session
if "N_SAMPLES" not in dir() and "N_SAMPLES" not in vars():
    N_SAMPLES = 6
if "TOP_K" not in dir() and "TOP_K" not in vars():
    TOP_K = 10
if "CLASS_NAMES" not in dir() and "CLASS_NAMES" not in vars():
    CLASS_NAMES = ds_train.classes

if "plot_predictions" not in dir() and "plot_predictions" not in vars():
    def plot_predictions(model, dataset, n=N_SAMPLES, top_k=TOP_K, seed=42,
                         save_filename=None):
        """
        Show n panels: food image (left) + top-k confidence bar chart (right).
        Green bar = correct prediction, red bar = wrong.
        Pass save_filename='my_plot.png' to save the figure.
        """
        model.eval()
        rng = random.Random(seed)
        indices = rng.sample(range(len(dataset)), n)

        fig, axes = plt.subplots(n, 2, figsize=(11, 4 * n),
                                 gridspec_kw={"width_ratios": [1, 2]})

        with torch.no_grad():
            for row, idx in enumerate(indices):
                img_tensor, true_label = dataset[idx]

                logits = model(img_tensor.unsqueeze(0).to(DEVICE))
                probs  = torch.softmax(logits, dim=1).squeeze().cpu()
                top_probs, top_ids = probs.topk(top_k)

                pred_label = top_ids[0].item()
                confidence = top_probs[0].item() * 100
                is_correct = (pred_label == true_label)

                # left panel: food image
                ax_img = axes[row, 0]
                img_show = img_tensor.clone()
                for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
                    img_show[c] = img_show[c] * s + m
                img_show = img_show.clamp(0, 1)
                ax_img.imshow(img_show.permute(1, 2, 0).numpy())
                ax_img.set_title(
                    f"True label: {CLASS_NAMES[true_label].replace('_', ' ')}",
                    fontsize=10)
                ax_img.axis("off")

                # right panel: confidence bar chart
                ax_bar = axes[row, 1]
                bar_labels = [CLASS_NAMES[i.item()].replace("_", " ") for i in top_ids]
                bar_values = [p.item() * 100 for p in top_probs]
                bar_color  = "#2ca02c" if is_correct else "#d62728"
                bar_colors = [bar_color if i == 0 else "#aec7e8" for i in range(top_k)]

                bars = ax_bar.barh(bar_labels[::-1], bar_values[::-1],
                                   color=bar_colors[::-1])
                ax_bar.axvline(50, color="gray", linestyle="--", linewidth=0.8)
                ax_bar.set_xlim(0, 100)
                ax_bar.set_xlabel("Confidence (%)")
                tick_mark = "✓ Correct" if is_correct else "✗ Wrong"
                ax_bar.set_title(
                    f"Prediction: {CLASS_NAMES[pred_label].replace('_', ' ')} "
                    f"({confidence:.1f}%)\n{tick_mark}",
                    fontsize=10, color=bar_color)

                for bar, val in zip(bars[::-1], bar_values):
                    if val > 1:
                        ax_bar.text(val + 0.5,
                                    bar.get_y() + bar.get_height() / 2,
                                    f"{val:.1f}%", va="center", fontsize=8)

                caption = (f"Sample index: {idx}  |  "
                           f"True: {CLASS_NAMES[true_label].replace('_', ' ')}  |  "
                           f"Predicted: {CLASS_NAMES[pred_label].replace('_', ' ')}  |  "
                           f"Confidence: {confidence:.1f}%")
                fig.text(0.01, 1 - (row + 1) / n + 0.01 / n,
                         caption, fontsize=7.5, color="#444444",
                         transform=fig.transFigure)

        plt.tight_layout(rect=[0, 0.0, 1, 1])
        if save_filename:
            save_figure(fig, save_filename)
        plt.show()

resnet50.load_state_dict(load_checkpoint("cnn_resnet50_best.pt"))
plot_predictions(
    model=resnet50,
    dataset=img_test_ds,
    n=N_SAMPLES,
    seed=99,               # different seed → different samples than Section 8.6
    save_filename="resnet50_predictions.png",
)
print("✓ 9.6 — ResNet-50 prediction visualiser complete.")


### 9.7 — CNN Scratch vs. ResNet-50: Discussion

| Metric | CNN Scratch | ResNet-50 |
|---|---|---|
| Test Accuracy | 47.43% | **78.42%** |
| Macro F1 | 0.462 | **0.784** |
| Trainable parameters (final) | ~8.9 M (all layers) | ~2.1 M (head + layer4) |
| Training phases | Single phase, 20 epochs | Phase A (head, 5 ep) + Phase B (layer4 unfrozen, 10 ep) |

#### Why ResNet-50 outperforms by such a large margin

**1. ImageNet pre-training — a 14-million-image head start.**  
ResNet-50 was trained on 14 million ImageNet images before it saw a single food photo. It already recognises textures (crispy batter, glossy sauce, flaky pastry), shapes (round, layered, irregular), and lighting patterns. The CNN scratch model builds all of these representations from 75,750 training images alone — an order of magnitude less data for a far harder initialisation problem.

**2. Representational depth — 50 layers vs. 4 blocks.**  
ResNet-50 has 50 convolutional layers and residual skip connections that allow gradients to flow cleanly through the entire network. The scratch CNN has 4 double-conv blocks, giving it roughly 8 conv layers. With 101 classes to separate, many of them visually similar (spaghetti bolognese vs. spaghetti carbonara; chocolate cake vs. red velvet cake), shallow architectures cannot build fine-grained discriminative representations in the final layers.

**3. Two-phase fine-tuning — stable feature extraction before specialisation.**  
Phase A froze the backbone and trained only the classification head for 5 epochs. This prevented the pre-trained features from being corrupted by noisy early gradients. Phase B then unfroze only layer4 — the final ResNet block — and fine-tuned at a 10x lower learning rate (1e-4 vs 1e-3). This controlled unfreezing preserved low-level features (edges, textures) while adapting high-level features (food-specific patterns) to the target domain.

**4. Domain similarity — ImageNet contains food.**  
ImageNet includes food categories (bananas, pizza, strawberries, ice cream). The pre-trained feature maps are not just generically useful — they are specifically well-calibrated for the visual properties of food photography: colour saturation, plated presentation, overhead angles. The scratch CNN must discover all of this without any prior.

#### Failure mode analysis

Both models share the same top failure pairs, confirming the difficulty is inherent to the data rather than architecture-specific:

- **Steak ↔ Filet mignon** — identical presentation; label distinction is portion cut, not visual appearance
- **Spaghetti bolognese — Spaghetti carbonara** — colour difference (red vs. cream sauce) often lost under harsh lighting
- **Chocolate cake ↔ Red velvet cake** — distinguishable only by cross-section; plated whole-cake photos are nearly identical

ResNet-50 recovers faster from these ambiguous pairs because its deeper feature hierarchy captures subtle cues (sauce sheen, garnish colour, crust texture) that the scratch model cannot resolve at its depth.

#### Conclusion

For a 101-class food classification task, transfer learning from ImageNet is not a marginal improvement — it is the decisive factor. ResNet-50 achieves **78.4% test accuracy** against the scratch model’s 47.4%, a gap of **31 percentage points**. The scratch architecture is a valid proof-of-concept and demonstrates that the training pipeline works end-to-end, but ResNet-50 is the model used in the Wine Peer pairing pipeline.

## Section 10 — CNN Explainability: Grad-CAM

Grad-CAM (Gradient-weighted Class Activation Mapping) visualises which spatial regions of a food image drove the ResNet-50 classification decision. Understanding where the model looks confirms it is using food-relevant features (textures, plating shapes) rather than background artefacts.

Six examples are shown: two correct predictions, two wrong predictions, and two where the correct class is in the top-5 but not top-1.

| Sub-section | What |
|---|---|
| **10.1** | Restore prerequisites after kernel restart |
| **10.2** | Grad-CAM visualisation — 6 annotated examples |


### 10.1 — Restore prerequisites after kernel restart

If the Colab session was interrupted, this cell rebuilds `resnet50` (loading `cnn_resnet50_best.pt`) and reconstructs `img_test_loader` from the saved index arrays. Skip this cell if all prior cells are still in memory.


In [ ]:
# ── 10.2  Prerequisites — run after Colab reconnect ──────────────────────────
# Run this cell ONLY if you reconnected to Colab and kernel variables were lost.
# If the session is still active (no disconnect), skip to 10.1 directly.
#
# After this cell completes, run 10.1 (Grad-CAM cell) below.

import os, sys
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision.datasets import Food101
from torch.utils.data import DataLoader

# ── DEVICE ────────────────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
_cuda_ok = False
if torch.cuda.is_available():
    try:
        torch.zeros(1).cuda()
        _cuda_ok = True
    except Exception:
        pass
DEVICE = torch.device("cuda" if _cuda_ok else "cpu")
print(f"Device        : {DEVICE}")

# ── Paths ─────────────────────────────────────────────────────────────────────
if IN_COLAB:
    WEIGHTS        = Path("/content/weights")
    FIGURES        = Path("/content/figures")
    _DRIVE_WEIGHTS = Path("/content/drive/MyDrive/wine-dine/weights")
    FOOD101_ROOT   = Path("/content/data")
else:
    WEIGHTS        = Path("weights")
    FIGURES        = Path("figures")
    _DRIVE_WEIGHTS = WEIGHTS
    FOOD101_ROOT   = Path("wine-dine/data")

WEIGHTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# ── Rebuild ResNet-50 architecture ───────────────────────────────────────────
resnet50 = models.resnet50(weights=None)
resnet50.fc = nn.Linear(2048, 101)
resnet50 = resnet50.to(DEVICE)

# ── Load best weights (VM first, Drive fallback) ──────────────────────────────
_fname  = "cnn_resnet50_best.pt"
_loaded = False
for _wdir in [WEIGHTS, _DRIVE_WEIGHTS]:
    _candidate = _wdir / _fname
    if _candidate.exists():
        resnet50.load_state_dict(
            torch.load(_candidate, map_location=DEVICE, weights_only=True))
        resnet50.eval()
        print(f"Weights loaded: {_candidate}")
        _loaded = True
        break
if not _loaded:
    raise FileNotFoundError(
        f"'{_fname}' not found.\n"
        f"  Checked: {WEIGHTS / _fname}\n"
        f"           {_DRIVE_WEIGHTS / _fname}\n"
        "  → Mount Drive (re-run Section 1.3) or retrain (Section 9)."
    )

# ── ImageNet constants + val/test transform ───────────────────────────────────
IMAGENET_MEAN      = [0.485, 0.456, 0.406]
IMAGENET_STD       = [0.229, 0.224, 0.225]
val_test_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Rebuild test DataLoader ───────────────────────────────────────────────────
IMG_BATCH = 64
_ds_test        = Food101(root=FOOD101_ROOT, split="test", transform=val_test_transform)
img_test_loader = DataLoader(_ds_test, batch_size=IMG_BATCH, shuffle=False,
                             num_workers=2, pin_memory=True)
CLASS_NAMES     = _ds_test.classes

print(f"Test loader  : {len(_ds_test):,} images  |  batch size: {IMG_BATCH}")
print(f"CLASS_NAMES  : {len(CLASS_NAMES)} classes")
print("✓ 10.2 complete — proceed to 10.1 (Grad-CAM cell below).")


### 10.2 — Grad-CAM visualisation

Hooks into the final convolutional layer (`layer4[-1]`) to capture activations and gradients, then overlays the resulting heatmap on the original image. Each panel shows the food photo, the Grad-CAM overlay, and the model top-1 prediction with confidence.


In [65]:
# ── 10  Grad-CAM explainability ───────────────────────────────────────────────
# requires: pip install torchcam
from torchcam.methods import GradCAM
from torchcam.utils  import overlay_mask
from torchvision.transforms.functional import to_pil_image

# Load best ResNet-50 weights
resnet50.load_state_dict(torch.load(WEIGHTS / "cnn_resnet50_best.pt", map_location=DEVICE))
resnet50.eval()

cam_extractor = GradCAM(resnet50, target_layer="layer4")

# ── Collect 6 representative test examples ────────────────────────────────────
# 2 correct top-1 | 2 wrong top-1 (not in top-5) | 2 top-5 hits (wrong top-1)
examples = {"correct": [], "wrong": [], "top5_correct": []}
for imgs, labels in img_test_loader:
    imgs   = imgs.to(DEVICE)
    logits = resnet50(imgs)
    top5   = logits.topk(5, dim=1).indices.cpu()
    top1   = logits.argmax(1).cpu()
    for i in range(len(labels)):
        lbl = labels[i].item()
        if top1[i] == lbl and len(examples["correct"]) < 2:
            examples["correct"].append((imgs[i].detach(), lbl, top1[i].item()))
        elif top1[i] != lbl and lbl not in top5[i].tolist() and len(examples["wrong"]) < 2:
            examples["wrong"].append((imgs[i].detach(), lbl, top1[i].item()))
        elif top1[i] != lbl and lbl in top5[i].tolist() and len(examples["top5_correct"]) < 2:
            examples["top5_correct"].append((imgs[i].detach(), lbl, top1[i].item()))
    if all(len(v) >= 2 for v in examples.values()):
        break

all_examples = (
    [("correct ✓",  *e) for e in examples["correct"]] +
    [("wrong ✗",    *e) for e in examples["wrong"]] +
    [("top-5 hit",  *e) for e in examples["top5_correct"]]
)

# ── Plot: row 0 = original image, row 1 = Grad-CAM overlay ───────────────────
fig, axes = plt.subplots(2, 6, figsize=(22, 8))
_mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
_std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

for col, (case, img_t, true_lbl, pred_lbl) in enumerate(all_examples):
    out = resnet50(img_t.unsqueeze(0))
    cam = cam_extractor(pred_lbl, out)[0]          # (1, H, W)

    # Un-normalise for display
    orig    = (img_t.cpu() * _std + _mean).clamp(0, 1)
    overlay = overlay_mask(to_pil_image(orig), to_pil_image(cam.squeeze(0)), alpha=0.5)

    title_color = "green" if case.startswith("correct") else ("orange" if case.startswith("top") else "red")
    axes[0, col].imshow(to_pil_image(orig))
    axes[0, col].set_title(
        f"True: {CLASS_NAMES[true_lbl]}\nPred: {CLASS_NAMES[pred_lbl]}\n[{case}]",
        fontsize=7, color=title_color
    )
    axes[0, col].axis("off")
    axes[1, col].imshow(overlay)
    axes[1, col].set_title("Grad-CAM", fontsize=7)
    axes[1, col].axis("off")

plt.suptitle("Section 10 — Grad-CAM: what ResNet-50 looks at when classifying food", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "10_gradcam.png", dpi=100, bbox_inches="tight")
plt.show()
cam_extractor.remove_hooks()
print("✓ Section 10 — Grad-CAM complete.  Figure saved → figures/10_gradcam.png")


## Section 11 — RNN / LSTM Branch (Text Model)

This section trains **three text classification models** on the **WineSensed** wine-review dataset to predict **grape variety** (15 classes) from tasting-note text, progressing from a simple recurrent baseline to a pre-trained Transformer encoder.

### Sub-sections

| Sub-section | Model | Content |
|---|---|---|
| **11.1** | — | Text `Dataset` class + `DataLoader` objects |
| **11.2** | LSTM Baseline | Variation 1 — Unidirectional 2-layer LSTM |
| **11.3** | — | Shared training utilities (`train_text_epoch`, `eval_text`) |
| **11.4** | LSTM Baseline | Training loop with per-epoch Drive checkpoints |
| **11.5** | LSTM Baseline | Test evaluation — accuracy, Macro F1, curves, confusion matrix |
| **11.6** | BiLSTM + Attention | Variation 2 — Bidirectional LSTM + Bahdanau attention |
| **11.7** | BiLSTM + Attention | Training loop with per-epoch Drive checkpoints |
| **11.8** | BiLSTM + Attention | Test evaluation — accuracy, Macro F1, curves, confusion matrix |
| **11.9** | — | Side-by-side comparison: LSTM vs BiLSTM + Attention |
| **11.10** | DistilBERT | **Bonus (+3 pts)** — Install `transformers`, build `WineBertDataset` |
| **11.11** | DistilBERT | Fine-tune linear head on frozen DistilBERT backbone |
| **11.12** | DistilBERT | Test evaluation + **3-model comparison** (LSTM / BiLSTM / DistilBERT) |

### Model overview

| Model | Architecture | Embeddings | Trainable params |
|---|---|---|---|
| **LSTM Baseline** | 2-layer unidirectional LSTM → final hidden state → FC | GloVe 100-d, fine-tuned | ~3–5 M |
| **BiLSTM + Attention** | 2-layer BiLSTM + Bahdanau attention → context vector → FC | GloVe 100-d, fine-tuned | ~5–8 M |
| **DistilBERT (frozen)** | Frozen DistilBERT backbone → [CLS] token → linear head | Contextual (BERT sub-word) | ~15 K (head only) |

### Design choices (LSTM / BiLSTM)

| Choice | Rationale |
|---|---|
| GloVe 100-d frozen → fine-tuned | Wine descriptors (tannins, terroir, minerality) are in GloVe’s Wikipedia corpus; fine-tuning adapts them to the domain |
| `pack_padded_sequence` | Ignores `<PAD>` positions during LSTM forward pass — correct gradient flow |
| Gradient clipping (max-norm 5) | Standard RNN stability measure |
| Weighted CrossEntropy | Corrects for class imbalance across 15 grape varieties (`CLASS_WEIGHTS` from Section 7.5) |
| Early stopping (patience = 4) | Prevents overfitting on the relatively small WineSensed corpus |
| Per-epoch Drive checkpoints | Full state saved each epoch — training can resume after a Colab disconnect |

### Design choices (DistilBERT)

| Choice | Rationale |
|---|---|
| Frozen backbone | Avoids catastrophic forgetting on the small wine dataset; only the 15-class head is updated |
| [CLS] token pooling | Standard BERT practice — the [CLS] position is pre-trained to summarise the full sequence |
| Sub-word tokenisation | Handles domain-specific words (e.g. `terroir` → `terr`, `##oir`) without OOV issues |
| max_len = 128 | Covers >95% of wine review lengths; GPU-memory-friendly |

**Prerequisite cells:** 1.2 (imports), 4.1 (splits), 7.1–7.5 (text preprocessing — `txt_train`, `lbl_train`, `VOCAB`, `embedding_matrix`, `CLASS_WEIGHTS`)


### 11.1 — Text Dataset and DataLoaders

The ReviewDataset class wraps the integer-encoded sequences and class labels so PyTorch DataLoaders can yield shuffled mini-batches. This cell also builds 	xt_train_loader, 	xt_val_loader, and 	xt_test_loader that are shared by all three text models.


In [ ]:
# ── 11.1  Text Dataset and DataLoaders ───────────────────────────────────────
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class WineTextDataset(torch.utils.data.Dataset):
    """Dataset for padded token-id sequences + sequence lengths + labels."""
    def __init__(self, sequences, labels):
        self.X       = torch.tensor(sequences, dtype=torch.long)
        self.y       = torch.tensor(labels,    dtype=torch.long)
        self.lengths = (self.X != 0).sum(dim=1).clamp(min=1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.lengths[i], self.y[i]


TXT_BATCH = 64

txt_train_ds = WineTextDataset(X_train, lbl_train)
txt_val_ds   = WineTextDataset(X_val,   lbl_val)
txt_test_ds  = WineTextDataset(X_test,  lbl_test)

txt_train_loader = DataLoader(txt_train_ds, batch_size=TXT_BATCH, shuffle=True,
                               num_workers=0, pin_memory=False)
txt_val_loader   = DataLoader(txt_val_ds,   batch_size=TXT_BATCH, shuffle=False,
                               num_workers=0, pin_memory=False)
txt_test_loader  = DataLoader(txt_test_ds,  batch_size=TXT_BATCH, shuffle=False,
                               num_workers=0, pin_memory=False)

print(f"{'Split':<10} {'Samples':>8}  {'Batches':>8}")
print("-" * 30)
for _name, _ds, _ldr in [("train", txt_train_ds, txt_train_loader),
                          ("val",   txt_val_ds,   txt_val_loader),
                          ("test",  txt_test_ds,  txt_test_loader)]:
    print(f"{_name:<10} {len(_ds):>8,}  {len(_ldr):>8,}")

_xb_t, _lb_t, _yb_t = next(iter(txt_train_loader))
print(f"\nBatch shapes:")
print(f"  sequences : {_xb_t.shape}   (B x MAX_SEQ_LEN={MAX_SEQ_LEN})")
print(f"  lengths   : {_lb_t.shape}   min={_lb_t.min()}, max={_lb_t.max()}")
print(f"  labels    : {_yb_t.shape}   classes 0-{_yb_t.max()}")
print(f"\n✓ 11.1 — Text DataLoaders ready.")

### 11.2 — Variation 1: Unidirectional LSTM Baseline

**Architecture:**
```
Input (B × L)  →  Embedding (GloVe 100-d, fine-tuned)
               →  Dropout(0.4)
               →  LSTM (100-d → 256-d, 2 layers, unidirectional)
               →  Last hidden state of final layer  (B × 256)
               →  Dropout(0.4)
               →  Linear(256 → 15)
```

- Reads each wine review **left-to-right only**; final hidden state summarises the full sequence.
- `pack_padded_sequence` ensures `<PAD>` positions do not contribute to gradients.
- This is the **baseline** — deliberately simple so improvements from Variation 2 are clearly attributable to bidirectionality and attention.

In [ ]:
# ── 11.2  Unidirectional LSTM Baseline ───────────────────────────────────────
class LSTMBaseline(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes,
                 n_layers=2, dropout=0.4, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(
                torch.tensor(embedding_matrix, dtype=torch.float32))
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, lengths):
        emb    = self.drop(self.embedding(x))
        packed = pack_padded_sequence(emb, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        out = self.drop(hidden[-1])
        return self.fc(out)

    def encode(self, x, lengths):
        emb    = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        return hidden[-1]


HIDDEN_DIM  = 256
N_LAYERS    = 2
DROPOUT_RNN = 0.4

lstm_model = LSTMBaseline(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    n_classes=len(GRAPE_CLASSES), n_layers=N_LAYERS, dropout=DROPOUT_RNN,
    embedding_matrix=embedding_matrix,
).to(DEVICE)

_total = sum(p.numel() for p in lstm_model.parameters())
print(f"LSTMBaseline — total params : {_total:,}")
_out_lstm = lstm_model(_xb_t.to(DEVICE), _lb_t.to(DEVICE))
assert _out_lstm.shape == (TXT_BATCH, len(GRAPE_CLASSES))
print(f"Forward pass  : OK → {_out_lstm.shape}")
print("✓ 11.2 — LSTMBaseline ready.")

### 11.3 — Shared training utilities

Two helper functions shared by the LSTM Baseline (11.4) and BiLSTM + Attention (11.7):

| Function | Role |
|---|---|
| 	rain_text_epoch(model, loader, criterion, optimizer) | One full training pass; returns mean loss and accuracy |
| eval_text(model, loader, criterion) | No-grad validation/test pass; returns mean loss and accuracy |


In [ ]:
# ── 11.3  Shared text training utilities ─────────────────────────────────────
def train_text_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for seqs, lengths, labels in loader:
        seqs, lengths, labels = seqs.to(DEVICE), lengths.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(seqs, lengths)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

@torch.no_grad()
def eval_text(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for seqs, lengths, labels in loader:
        seqs, lengths, labels = seqs.to(DEVICE), lengths.to(DEVICE), labels.to(DEVICE)
        logits      = model(seqs, lengths)
        total_loss += criterion(logits, labels).item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

criterion_txt = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
print("✓ 11.3 — training utilities ready.")
print(f"  criterion_txt : CrossEntropyLoss(weight=CLASS_WEIGHTS)  — {len(GRAPE_CLASSES)} classes")

### 11.4 — Training the LSTM Baseline

| Setting | Value |
|---|---|
| Optimiser | Adam, lr = 1e-3, weight_decay = 1e-5 |
| LR scheduler | ReduceLROnPlateau × 0.5, patience 2 |
| Max epochs | 30 |
| Early stopping | patience = 4 |
| Gradient clipping | max-norm 5 |

Best weights saved to `weights/lstm_best.pt`.

In [ ]:
# ── 11.4  Train the LSTM Baseline ────────────────────────────────────────────
TXT_EPOCHS   = 30
TXT_PATIENCE = 4

opt_lstm   = Adam(lstm_model.parameters(), lr=1e-3, weight_decay=1e-5)
sched_lstm = ReduceLROnPlateau(opt_lstm, factor=0.5, patience=2)

history_lstm       = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss_lstm = float("inf")
no_improve_lstm    = 0
best_ckpt_lstm     = WEIGHTS / "lstm_best.pt"

print(f"Training LSTMBaseline  (max {TXT_EPOCHS} epochs, patience={TXT_PATIENCE})")
print(f"{'Epoch':>5}  {'tr_loss':>8}  {'tr_acc':>7}  {'vl_loss':>8}  {'vl_acc':>7}  {'lr':>9}")
print("-" * 57)

for epoch in range(1, TXT_EPOCHS + 1):
    tr_loss, tr_acc = train_text_epoch(lstm_model, txt_train_loader, criterion_txt, opt_lstm)
    vl_loss, vl_acc = eval_text(lstm_model, txt_val_loader, criterion_txt)
    history_lstm["train_loss"].append(tr_loss)
    history_lstm["val_loss"].append(vl_loss)
    history_lstm["train_acc"].append(tr_acc)
    history_lstm["val_acc"].append(vl_acc)
    sched_lstm.step(vl_loss)
    lr_now = opt_lstm.param_groups[0]["lr"]
    if vl_loss < best_val_loss_lstm:
        best_val_loss_lstm = vl_loss
        no_improve_lstm    = 0
        save_checkpoint(lstm_model.state_dict(), "lstm_best.pt")
        marker = " ✓"
    else:
        no_improve_lstm += 1
        marker = ""
    # ── per-epoch full checkpoint (enables resume after disconnect) ────────────
    save_checkpoint(
        {"epoch": epoch, "model_state": lstm_model.state_dict(),
         "optimizer_state": opt_lstm.state_dict(), "val_loss": vl_loss,
         "history": history_lstm},
        f"lstm_ckpt_ep{epoch:02d}.pt"
    )
    print(f"{epoch:>5}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  {vl_loss:>8.4f}  {vl_acc:>7.4f}  {lr_now:>9.2e}{marker}")
    if no_improve_lstm >= TXT_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest val loss : {best_val_loss_lstm:.4f}  →  {best_ckpt_lstm}")
print("✓ 11.4 — LSTM Baseline training complete.")


### 11.5 — LSTM Baseline: test evaluation

The **test set is used exactly once** here. Loads the best LSTM checkpoint and computes test accuracy, Macro F1, per-class classification report, learning curves, and confusion matrix.


In [ ]:
# ── 11.5  LSTM Baseline — test evaluation, curves, confusion matrix ───────────
lstm_model.load_state_dict(torch.load(best_ckpt_lstm, map_location=DEVICE))
lstm_model.eval()

all_preds_lstm, all_labels_lstm = [], []
with torch.no_grad():
    for seqs, lengths, labels in txt_test_loader:
        logits = lstm_model(seqs.to(DEVICE), lengths.to(DEVICE))
        all_preds_lstm.extend(logits.argmax(1).cpu().tolist())
        all_labels_lstm.extend(labels.tolist())

lstm_test_acc = sum(p == l for p, l in zip(all_preds_lstm, all_labels_lstm)) / len(all_labels_lstm)
lstm_f1       = f1_score(all_labels_lstm, all_preds_lstm, average="macro")
print(f"LSTM Baseline — Test accuracy : {lstm_test_acc:.4f}  ({lstm_test_acc*100:.2f}%)")
print(f"                Macro F1      : {lstm_f1:.4f}")
print()
print(classification_report(all_labels_lstm, all_preds_lstm, target_names=GRAPE_CLASSES, digits=3))

_epochs_lstm = range(1, len(history_lstm["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(_epochs_lstm, history_lstm["train_loss"], label="Train", lw=1.8)
axes[0].plot(_epochs_lstm, history_lstm["val_loss"],   label="Val",   lw=1.8)
axes[0].set_title("LSTM Baseline — Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(_epochs_lstm, history_lstm["train_acc"], label="Train", lw=1.8)
axes[1].plot(_epochs_lstm, history_lstm["val_acc"],   label="Val",   lw=1.8)
axes[1].set_title("LSTM Baseline — Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.suptitle(f"11.5 — LSTM Baseline  |  Test Acc: {lstm_test_acc*100:.2f}%  Macro-F1: {lstm_f1:.4f}", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "12_lstm_curves.png", dpi=100, bbox_inches="tight")
plt.show()

lstm_cm = confusion_matrix(all_labels_lstm, all_preds_lstm)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(lstm_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=GRAPE_CLASSES, yticklabels=GRAPE_CLASSES, ax=ax, linewidths=0.3)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"LSTM Baseline — Confusion Matrix  (Test Acc: {lstm_test_acc*100:.2f}%)")
plt.xticks(rotation=45, ha="right", fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "12_lstm_cm.png", dpi=100, bbox_inches="tight")
plt.show()
print("✓ 11.5 — LSTM Baseline test evaluation complete.")

### 11.6 — Variation 2: Bidirectional LSTM + Bahdanau Attention

**Architecture:**
```
Input (B × L)  →  Embedding (GloVe 100-d, fine-tuned)
               →  Dropout(0.4)
               →  BiLSTM (100-d → 256-d × 2, 2 layers)
               →  Bahdanau attention over all L hidden states  →  B × 512
               →  Dropout(0.4)
               →  Linear(512 → 15)
```

**Bahdanau (additive) attention:**

$$\text{score}(h_t) = \mathbf{v}^\top \tanh(\mathbf{W} h_t) \qquad
\alpha_t = \text{softmax}(\text{score}) \qquad
\mathbf{c} = \sum_t \alpha_t h_t$$

Attention lets the model up-weight grape-discriminative descriptor words (e.g. *tannins* for Cabernet, *tropical fruit* for Viognier) rather than compressing the whole sequence into a single vector.

In [ ]:
# ── 11.6  Bidirectional LSTM + Bahdanau Attention ────────────────────────────
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
        self.v    = nn.Linear(hidden_dim,     1,           bias=False)

    def forward(self, hidden_states, mask=None):
        energy  = torch.tanh(self.attn(hidden_states))
        scores  = self.v(energy).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        context = (weights.unsqueeze(-1) * hidden_states).sum(dim=1)
        return context, weights


class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes,
                 n_layers=2, dropout=0.4, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(
                torch.tensor(embedding_matrix, dtype=torch.float32))
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.attention = BahdanauAttention(hidden_dim)
        self.drop      = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, n_classes)

    def forward(self, x, lengths):
        emb    = self.drop(self.embedding(x))
        packed = pack_padded_sequence(emb, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=x.shape[1])
        mask    = (x != 0)
        context, _ = self.attention(output, mask)
        return self.fc(self.drop(context))

    def encode(self, x, lengths):
        emb    = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=x.shape[1])
        context, _ = self.attention(output, (x != 0))
        return context


bilstm_model = BiLSTMAttention(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    n_classes=len(GRAPE_CLASSES), n_layers=N_LAYERS, dropout=DROPOUT_RNN,
    embedding_matrix=embedding_matrix,
).to(DEVICE)

_total_bi = sum(p.numel() for p in bilstm_model.parameters())
print(f"BiLSTMAttention — total params : {_total_bi:,}")
_out_bi = bilstm_model(_xb_t.to(DEVICE), _lb_t.to(DEVICE))
assert _out_bi.shape == (TXT_BATCH, len(GRAPE_CLASSES))
print(f"Forward pass  : OK → {_out_bi.shape}")
print("✓ 11.6 — BiLSTMAttention ready.")

### 11.7 — Training BiLSTM + Attention

Same training protocol as the LSTM Baseline (11.4) — Adam, ReduceLROnPlateau, early stopping (patience 4), gradient clipping, per-epoch Drive checkpoints. Best weights saved to weights/bilstm_best.pt.


In [ ]:
# ── 11.7  Train BiLSTM + Attention ───────────────────────────────────────────
opt_bilstm   = Adam(bilstm_model.parameters(), lr=1e-3, weight_decay=1e-5)
sched_bilstm = ReduceLROnPlateau(opt_bilstm, factor=0.5, patience=2)

history_bilstm       = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss_bilstm = float("inf")
no_improve_bilstm    = 0
best_ckpt_bilstm     = WEIGHTS / "bilstm_best.pt"

print(f"Training BiLSTMAttention  (max {TXT_EPOCHS} epochs, patience={TXT_PATIENCE})")
print(f"{'Epoch':>5}  {'tr_loss':>8}  {'tr_acc':>7}  {'vl_loss':>8}  {'vl_acc':>7}  {'lr':>9}")
print("-" * 57)

for epoch in range(1, TXT_EPOCHS + 1):
    tr_loss, tr_acc = train_text_epoch(bilstm_model, txt_train_loader, criterion_txt, opt_bilstm)
    vl_loss, vl_acc = eval_text(bilstm_model, txt_val_loader, criterion_txt)
    history_bilstm["train_loss"].append(tr_loss)
    history_bilstm["val_loss"].append(vl_loss)
    history_bilstm["train_acc"].append(tr_acc)
    history_bilstm["val_acc"].append(vl_acc)
    sched_bilstm.step(vl_loss)
    lr_now = opt_bilstm.param_groups[0]["lr"]
    if vl_loss < best_val_loss_bilstm:
        best_val_loss_bilstm = vl_loss
        no_improve_bilstm    = 0
        save_checkpoint(bilstm_model.state_dict(), "bilstm_best.pt")
        marker = " ✓"
    else:
        no_improve_bilstm += 1
        marker = ""
    # ── per-epoch full checkpoint (enables resume after disconnect) ────────────
    save_checkpoint(
        {"epoch": epoch, "model_state": bilstm_model.state_dict(),
         "optimizer_state": opt_bilstm.state_dict(), "val_loss": vl_loss,
         "history": history_bilstm},
        f"bilstm_ckpt_ep{epoch:02d}.pt"
    )
    print(f"{epoch:>5}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  {vl_loss:>8.4f}  {vl_acc:>7.4f}  {lr_now:>9.2e}{marker}")
    if no_improve_bilstm >= TXT_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest val loss : {best_val_loss_bilstm:.4f}  →  {best_ckpt_bilstm}")
print("✓ 11.7 — BiLSTM + Attention training complete.")


### 11.8 — BiLSTM + Attention: test evaluation

Same evaluation format as 11.5. Loads the best BiLSTM checkpoint and computes test accuracy, Macro F1, per-class classification report, learning curves, and confusion matrix.


In [ ]:
# ── 11.8  BiLSTM + Attention — test evaluation, curves, confusion matrix ──────
bilstm_model.load_state_dict(torch.load(best_ckpt_bilstm, map_location=DEVICE))
bilstm_model.eval()

all_preds_bilstm, all_labels_bilstm = [], []
with torch.no_grad():
    for seqs, lengths, labels in txt_test_loader:
        logits = bilstm_model(seqs.to(DEVICE), lengths.to(DEVICE))
        all_preds_bilstm.extend(logits.argmax(1).cpu().tolist())
        all_labels_bilstm.extend(labels.tolist())

bilstm_test_acc = sum(p == l for p, l in zip(all_preds_bilstm, all_labels_bilstm)) / len(all_labels_bilstm)
bilstm_f1       = f1_score(all_labels_bilstm, all_preds_bilstm, average="macro")
print(f"BiLSTM+Attention — Test accuracy : {bilstm_test_acc:.4f}  ({bilstm_test_acc*100:.2f}%)")
print(f"                   Macro F1      : {bilstm_f1:.4f}")
print()
print(classification_report(all_labels_bilstm, all_preds_bilstm, target_names=GRAPE_CLASSES, digits=3))

_epochs_bi = range(1, len(history_bilstm["train_loss"]) + 1)
fig, axes  = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(_epochs_bi, history_bilstm["train_loss"], label="Train", lw=1.8)
axes[0].plot(_epochs_bi, history_bilstm["val_loss"],   label="Val",   lw=1.8)
axes[0].set_title("BiLSTM + Attention — Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(_epochs_bi, history_bilstm["train_acc"], label="Train", lw=1.8)
axes[1].plot(_epochs_bi, history_bilstm["val_acc"],   label="Val",   lw=1.8)
axes[1].set_title("BiLSTM + Attention — Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.suptitle(f"11.8 — BiLSTM + Attention  |  Test Acc: {bilstm_test_acc*100:.2f}%  Macro-F1: {bilstm_f1:.4f}", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "12_bilstm_curves.png", dpi=100, bbox_inches="tight")
plt.show()

bilstm_cm = confusion_matrix(all_labels_bilstm, all_preds_bilstm)
fig, ax   = plt.subplots(figsize=(13, 11))
sns.heatmap(bilstm_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=GRAPE_CLASSES, yticklabels=GRAPE_CLASSES, ax=ax, linewidths=0.3)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"BiLSTM + Attention — Confusion Matrix  (Test Acc: {bilstm_test_acc*100:.2f}%)")
plt.xticks(rotation=45, ha="right", fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "12_bilstm_cm.png", dpi=100, bbox_inches="tight")
plt.show()
print("✓ 11.8 — BiLSTM + Attention test evaluation complete.")

### 11.9 — Comparison: LSTM Baseline vs. BiLSTM + Attention

Produces:
1. **4-panel training curves** — both models side-by-side
2. **Per-class accuracy bar chart** — which grape varieties each model handles better
3. **Summary table** — test accuracy, Macro F1, parameter count

In [ ]:
# ── 11.9  Side-by-side comparison ────────────────────────────────────────────
_lstm_params   = sum(p.numel() for p in lstm_model.parameters())
_bilstm_params = sum(p.numel() for p in bilstm_model.parameters())

print("=" * 65)
print(f"{'Model':<28} {'Test Acc':>10}  {'Macro F1':>10}  {'Params':>10}")
print("-" * 65)
print(f"{'LSTM Baseline':<28} {lstm_test_acc*100:>9.2f}%  {lstm_f1:>10.4f}  {_lstm_params:>10,}")
print(f"{'BiLSTM + Attention':<28} {bilstm_test_acc*100:>9.2f}%  {bilstm_f1:>10.4f}  {_bilstm_params:>10,}")
print("=" * 65)

e1 = range(1, len(history_lstm["train_loss"]) + 1)
e2 = range(1, len(history_bilstm["train_loss"]) + 1)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0,0].plot(e1, history_lstm["train_loss"], label="Train", lw=1.8)
axes[0,0].plot(e1, history_lstm["val_loss"],   label="Val",   lw=1.8)
axes[0,0].set_title("LSTM Baseline — Loss"); axes[0,0].legend()
axes[0,1].plot(e1, history_lstm["train_acc"], label="Train", lw=1.8)
axes[0,1].plot(e1, history_lstm["val_acc"],   label="Val",   lw=1.8)
axes[0,1].set_title("LSTM Baseline — Accuracy"); axes[0,1].legend()
axes[1,0].plot(e2, history_bilstm["train_loss"], label="Train", lw=1.8)
axes[1,0].plot(e2, history_bilstm["val_loss"],   label="Val",   lw=1.8)
axes[1,0].set_title("BiLSTM + Attention — Loss"); axes[1,0].legend()
axes[1,1].plot(e2, history_bilstm["train_acc"], label="Train", lw=1.8)
axes[1,1].plot(e2, history_bilstm["val_acc"],   label="Val",   lw=1.8)
axes[1,1].set_title("BiLSTM + Attention — Accuracy"); axes[1,1].legend()
plt.suptitle("11.9 — LSTM vs BiLSTM + Attention: Training Curves", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "12_rnn_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

def per_class_acc(preds, labels, n_classes):
    counts = [0]*n_classes; correct_c = [0]*n_classes
    for p, l in zip(preds, labels):
        counts[l]  += 1
        correct_c[l] += int(p == l)
    return [correct_c[i]/counts[i] if counts[i] else 0 for i in range(n_classes)]

lstm_pca   = per_class_acc(all_preds_lstm,   all_labels_lstm,   len(GRAPE_CLASSES))
bilstm_pca = per_class_acc(all_preds_bilstm, all_labels_bilstm, len(GRAPE_CLASSES))
x = range(len(GRAPE_CLASSES)); w = 0.38
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar([i - w/2 for i in x], lstm_pca,   w, label="LSTM Baseline",     alpha=0.85, color="#4C72B0")
ax.bar([i + w/2 for i in x], bilstm_pca, w, label="BiLSTM + Attention", alpha=0.85, color="#DD8452")
ax.set_xticks(list(x)); ax.set_xticklabels(GRAPE_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Per-class Accuracy"); ax.set_ylim(0, 1.05)
ax.axhline(lstm_test_acc,   color="#4C72B0", lw=1.2, ls="--", alpha=0.6, label="LSTM overall")
ax.axhline(bilstm_test_acc, color="#DD8452", lw=1.2, ls="--", alpha=0.6, label="BiLSTM overall")
ax.set_title("11.9 — Per-class Accuracy: LSTM vs BiLSTM + Attention"); ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "12_rnn_per_class.png", dpi=100, bbox_inches="tight")
plt.show()

print("Figures: 12_rnn_comparison.png  12_rnn_per_class.png")
print("✓ Section 11 complete.  Next: Section 12 — Business Integration.")

### 11.10 — DistilBERT: build DataLoader (Bonus +3 pts)

Installs 	ransformers and tokenises the wine reviews using the DistilBERT sub-word tokeniser (distilbert-base-uncased, max_len = 128). Wraps them in WineBertDataset and builds train / val / test DataLoaders. Sub-word tokenisation handles domain-specific terms (	erroir, minerality) without OOV issues.


In [ ]:
# ── 11.10  Build DistilBERT DataLoader ───────────────────────────────────
BERT_MODEL_NAME = 'distilbert-base-uncased'
BERT_MAX_LEN    = 128   # covers >95% of wine review tokens
BERT_BATCH      = 32    # smaller than LSTM — transformer is heavier

bert_tokenizer = DistilBertTokenizerFast.from_pretrained(BERT_MODEL_NAME)

class WineBertDataset(Dataset):
    """Tokenises raw review strings on the fly using DistilBERT tokenizer."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            'input_ids':      self.encodings['input_ids'][i],
            'attention_mask': self.encodings['attention_mask'][i],
            'label':          self.labels[i],
        }


# txt_train / txt_val / txt_test = raw review strings (Section 7.5)
# lbl_train / lbl_val / lbl_test = integer grape labels
bert_train_ds = WineBertDataset(txt_train, lbl_train, bert_tokenizer, BERT_MAX_LEN)
bert_val_ds   = WineBertDataset(txt_val,   lbl_val,   bert_tokenizer, BERT_MAX_LEN)
bert_test_ds  = WineBertDataset(txt_test,  lbl_test,  bert_tokenizer, BERT_MAX_LEN)

bert_train_loader = DataLoader(bert_train_ds, batch_size=BERT_BATCH, shuffle=True,  num_workers=0)
bert_val_loader   = DataLoader(bert_val_ds,   batch_size=BERT_BATCH, shuffle=False, num_workers=0)
bert_test_loader  = DataLoader(bert_test_ds,  batch_size=BERT_BATCH, shuffle=False, num_workers=0)

print(f'DistilBERT tokenizer : {BERT_MODEL_NAME}  |  max_len={BERT_MAX_LEN}')
print(f"{'Split':<8}  {'Samples':>8}  {'Batches':>8}")
for _name, _ds, _ldr in [('train', bert_train_ds, bert_train_loader),
                          ('val',   bert_val_ds,   bert_val_loader),
                          ('test',  bert_test_ds,  bert_test_loader)]:
    print(f'{_name:<8}  {len(_ds):>8,}  {len(_ldr):>8,}')
print('\n✓ 11.10 — WineBertDataset and DataLoaders ready.')


### 11.11 — DistilBERT: fine-tune linear head

The DistilBERT backbone is frozen; only the 15-class linear head trained on top of the [CLS] token is updated. This avoids catastrophic forgetting on the small WineSensed corpus while still benefiting from contextual sub-word representations.


In [ ]:
# ── 11.11  Fine-tune linear head on frozen DistilBERT ─────────────────────────
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

class DistilBertClassifier(nn.Module):
    """
    Frozen DistilBERT backbone + trainable linear head.
    Uses the [CLS] token representation (index 0) as the sentence embedding.
    """
    def __init__(self, model_name, n_classes, dropout=0.3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(model_name)
        # Freeze all backbone parameters
        for p in self.bert.parameters():
            p.requires_grad = False
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        out   = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls   = out.last_hidden_state[:, 0, :]   # [CLS] token
        return self.fc(self.drop(cls))

    def encode(self, input_ids, attention_mask):
        """Return 768-d CLS embedding (used by joint model)."""
        with torch.no_grad():
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]


bert_model = DistilBertClassifier(
    model_name=BERT_MODEL_NAME,
    n_classes=len(GRAPE_CLASSES),
).to(DEVICE)

_total_bert  = sum(p.numel() for p in bert_model.parameters())
_train_bert  = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f'DistilBertClassifier — total params   : {_total_bert:,}')
print(f'                       trainable head : {_train_bert:,}')

# ── Training ──────────────────────────────────────────────────────────────────
BERT_EPOCHS   = 10
BERT_PATIENCE = 3

opt_bert   = Adam(bert_model.fc.parameters(), lr=1e-3, weight_decay=1e-5)
sched_bert = ReduceLROnPlateau(opt_bert, factor=0.5, patience=2)

history_bert       = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss_bert = float('inf')
no_improve_bert    = 0

def train_bert_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for batch in loader:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(lbls)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / n, correct / n

@torch.no_grad()
def eval_bert(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for batch in loader:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['label'].to(DEVICE)
        logits      = model(ids, mask)
        total_loss += criterion(logits, lbls).item() * len(lbls)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / n, correct / n


print(f'Training DistilBertClassifier  (max {BERT_EPOCHS} epochs, patience={BERT_PATIENCE})')
print(f"{'Epoch':>5}  {'tr_loss':>8}  {'tr_acc':>7}  {'vl_loss':>8}  {'vl_acc':>7}  {'lr':>9}")
print('-' * 57)

for epoch in range(1, BERT_EPOCHS + 1):
    tr_loss, tr_acc = train_bert_epoch(bert_model, bert_train_loader, criterion_txt, opt_bert)
    vl_loss, vl_acc = eval_bert(bert_model, bert_val_loader, criterion_txt)
    history_bert['train_loss'].append(tr_loss)
    history_bert['val_loss'].append(vl_loss)
    history_bert['train_acc'].append(tr_acc)
    history_bert['val_acc'].append(vl_acc)
    sched_bert.step(vl_loss)
    lr_now = opt_bert.param_groups[0]['lr']
    if vl_loss < best_val_loss_bert:
        best_val_loss_bert = vl_loss
        no_improve_bert    = 0
        save_checkpoint(bert_model.state_dict(), 'distilbert_best.pt')
        marker = ' \u2713'
    else:
        no_improve_bert += 1
        marker = ''
    save_checkpoint(
        {'epoch': epoch, 'model_state': bert_model.state_dict(),
         'optimizer_state': opt_bert.state_dict(), 'val_loss': vl_loss,
         'history': history_bert},
        f'distilbert_ckpt_ep{epoch:02d}.pt'
    )
    print(f'{epoch:>5}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  {vl_loss:>8.4f}  {vl_acc:>7.4f}  {lr_now:>9.2e}{marker}')
    if no_improve_bert >= BERT_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val loss : {best_val_loss_bert:.4f}')
print('\u2713 11.11 \u2014 DistilBERT head training complete.')


### 11.12 — DistilBERT: test evaluation and 3-model comparison

Loads the best DistilBERT checkpoint and produces a final 3-model comparison table (LSTM Baseline / BiLSTM + Attention / DistilBERT) showing test accuracy, Macro F1, trainable parameter count, and training time.


In [ ]:
# ── 11.12  DistilBERT test evaluation + 3-model comparison ───────────────────
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load best weights
bert_model.load_state_dict(load_checkpoint('distilbert_best.pt'))
bert_model.eval()

all_preds_bert, all_labels_bert = [], []
with torch.no_grad():
    for batch in bert_test_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['label']
        logits = bert_model(ids, mask)
        all_preds_bert.extend(logits.argmax(1).cpu().tolist())
        all_labels_bert.extend(lbls.tolist())

bert_test_acc = sum(p == l for p, l in zip(all_preds_bert, all_labels_bert)) / len(all_labels_bert)
bert_f1       = f1_score(all_labels_bert, all_preds_bert, average='macro')
print(f'DistilBERT (frozen) — Test accuracy : {bert_test_acc:.4f}  ({bert_test_acc*100:.2f}%)')
print(f'                      Macro F1      : {bert_f1:.4f}')
print()
print(classification_report(all_labels_bert, all_preds_bert, target_names=GRAPE_CLASSES, digits=3))

# ── Learning curves ───────────────────────────────────────────────────────────
_epochs_bert = range(1, len(history_bert['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(_epochs_bert, history_bert['train_loss'], label='Train', lw=1.8)
axes[0].plot(_epochs_bert, history_bert['val_loss'],   label='Val',   lw=1.8)
axes[0].set_title('DistilBERT — Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(_epochs_bert, history_bert['train_acc'], label='Train', lw=1.8)
axes[1].plot(_epochs_bert, history_bert['val_acc'],   label='Val',   lw=1.8)
axes[1].set_title('DistilBERT — Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.suptitle(f'11.12 — DistilBERT  |  Test Acc: {bert_test_acc*100:.2f}%  Macro-F1: {bert_f1:.4f}', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / '13_distilbert_curves.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Confusion matrix ──────────────────────────────────────────────────────────
bert_cm = confusion_matrix(all_labels_bert, all_preds_bert)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(bert_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GRAPE_CLASSES, yticklabels=GRAPE_CLASSES, ax=ax, linewidths=0.3)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'DistilBERT — Confusion Matrix  (Test Acc: {bert_test_acc*100:.2f}%)')
plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / '13_distilbert_cm.png', dpi=100, bbox_inches='tight')
plt.show()

# ── 3-model comparison table ─────────────────────────────────────────────────
_lstm_params   = sum(p.numel() for p in lstm_model.parameters())
_bilstm_params = sum(p.numel() for p in bilstm_model.parameters())
_bert_params   = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)

print('\n' + '=' * 70)
print(f"{'Model':<30} {'Test Acc':>10}  {'Macro F1':>10}  {'Train Params':>14}")
print('-' * 70)
print(f"{'LSTM Baseline':<30} {lstm_test_acc*100:>9.2f}%  {lstm_f1:>10.4f}  {_lstm_params:>14,}")
print(f"{'BiLSTM + Attention':<30} {bilstm_test_acc*100:>9.2f}%  {bilstm_f1:>10.4f}  {_bilstm_params:>14,}")
print(f"{'DistilBERT (frozen head)':<30} {bert_test_acc*100:>9.2f}%  {bert_f1:>10.4f}  {_bert_params:>14,}")
print('=' * 70)

# ── Per-class accuracy comparison bar chart ───────────────────────────────────
def _per_class_acc(preds, labels, n):
    cnt = [0]*n; cor = [0]*n
    for p, l in zip(preds, labels):
        cnt[l] += 1; cor[l] += int(p == l)
    return [cor[i]/cnt[i] if cnt[i] else 0 for i in range(n)]

bert_pca = _per_class_acc(all_preds_bert, all_labels_bert, len(GRAPE_CLASSES))
x = range(len(GRAPE_CLASSES)); w = 0.27
fig, ax = plt.subplots(figsize=(18, 6))
ax.bar([i - w for i in x],    lstm_pca,   w, label='LSTM Baseline',       alpha=0.85, color='#4C72B0')
ax.bar([i      for i in x],   bilstm_pca, w, label='BiLSTM + Attention',   alpha=0.85, color='#DD8452')
ax.bar([i + w  for i in x],   bert_pca,   w, label='DistilBERT (frozen)',  alpha=0.85, color='#55A868')
ax.set_xticks(list(x)); ax.set_xticklabels(GRAPE_CLASSES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Per-class Accuracy'); ax.set_ylim(0, 1.05)
ax.axhline(lstm_test_acc,   color='#4C72B0', lw=1.0, ls='--', alpha=0.5)
ax.axhline(bilstm_test_acc, color='#DD8452', lw=1.0, ls='--', alpha=0.5)
ax.axhline(bert_test_acc,   color='#55A868', lw=1.0, ls='--', alpha=0.5)
ax.set_title('11.12 — Per-class Accuracy: LSTM vs BiLSTM vs DistilBERT'); ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / '13_text_model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('Figures: 13_distilbert_curves.png  13_distilbert_cm.png  13_text_model_comparison.png')
print('\u2713 11.12 — DistilBERT evaluation and 3-model comparison complete.')


## Section 12 — Save All Results and Weights

Saves all five trained models' weight files, per-class metrics, predictions, and confusion matrices to local disk. When running on Colab, everything is also synced to Google Drive so results survive session restarts.

| Model | Weight file |
|---|---|
| CNN from scratch | cnn_scratch_best.pt |
| ResNet-50 (transfer learning) | cnn_resnet50_best.pt |
| LSTM Baseline | lstm_best.pt |
| BiLSTM + Attention | ilstm_best.pt |
| DistilBERT (frozen head) | distilbert_best.pt |


In [ ]:
# ── 12  Save ALL results ─────────────────────────────────────────────────
import glob, shutil, os
import torch

snapshot = {}

# CNN Scratch
snapshot['cnn_scratch'] = {
    'test_acc': globals().get('test_acc'),
    'macro_f1': globals().get('macro_f1'),
    'history': globals().get('history_scratch'),
    'all_preds': globals().get('all_preds'),
    'all_labels': globals().get('all_labels'),
    'confusion_matrix': globals().get('sc_cm'),
    'weights_file': 'cnn_scratch_best.pt',
}

# ResNet-50
snapshot['resnet50'] = {
    'test_acc': globals().get('rn_test_acc'),
    'macro_f1': globals().get('rn_f1'),
    'history': globals().get('hist_rn'),
    'all_preds': globals().get('rn_preds'),
    'all_labels': globals().get('rn_labels'),
    'confusion_matrix': globals().get('cm'),
    'weights_file': 'cnn_resnet50_best.pt',
}

# LSTM
snapshot['lstm'] = {
    'test_acc': globals().get('lstm_test_acc'),
    'macro_f1': globals().get('lstm_f1'),
    'history': globals().get('history_lstm'),
    'all_preds': globals().get('all_preds_lstm'),
    'all_labels': globals().get('all_labels_lstm'),
    'confusion_matrix': globals().get('lstm_cm'),
    'weights_file': 'lstm_best.pt',
}

# BiLSTM + Attention
snapshot['bilstm'] = {
    'test_acc': globals().get('bilstm_test_acc'),
    'macro_f1': globals().get('bilstm_f1'),
    'history': globals().get('history_bilstm'),
    'all_preds': globals().get('all_preds_bilstm'),
    'all_labels': globals().get('all_labels_bilstm'),
    'confusion_matrix': globals().get('bilstm_cm'),
    'weights_file': 'bilstm_best.pt',
}

# DistilBERT (frozen head)
snapshot['distilbert'] = {
    'test_acc': globals().get('bert_test_acc'),
    'macro_f1': globals().get('bert_f1'),
    'history': globals().get('history_bert'),
    'all_preds': globals().get('all_preds_bert'),
    'all_labels': globals().get('all_labels_bert'),
    'confusion_matrix': globals().get('bert_cm'),
    'weights_file': 'distilbert_best.pt',
}

# Text artefacts
snapshot['text'] = {
    'VOCAB': globals().get('VOCAB'),
    'VOCAB_SIZE': globals().get('VOCAB_SIZE'),
    'MAX_SEQ_LEN': globals().get('MAX_SEQ_LEN'),
    'embedding_matrix': globals().get('embedding_matrix'),
    'GRAPE_CLASSES': globals().get('GRAPE_CLASSES'),
    'GRAPE_TO_IDX': globals().get('GRAPE_TO_IDX'),
    'CLASS_WEIGHTS': globals().get('CLASS_WEIGHTS'),
}

snapshot['splits'] = {
    'SEED': SEED,
    'train_size': globals().get('train_size'),
    'test_size': globals().get('test_size'),
}

# Sanitise snapshot — convert tensors/arrays to lists, drop any module/unpicklable objects
import pickle as _pkl, numpy as _np

def _sanitize(obj, _depth=0):
    if _depth > 12:
        return None
    if obj is None or isinstance(obj, (bool, int, float, str)):
        return obj
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, _np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: _sanitize(v, _depth+1) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_sanitize(v, _depth+1) for v in obj]
    try:
        _pkl.dumps(obj)   # test if picklable
        return obj
    except Exception:
        return None       # silently drop modules, lambdas, etc.

safe_snapshot = _sanitize(snapshot)

# 1. Save snapshot locally
snap_local = os.path.join(str(LOCAL_WEIGHTS), 'results_snapshot.pt')
torch.save(safe_snapshot, snap_local)
print(f'✓ results_snapshot.pt saved  ({os.path.getsize(snap_local)/1e6:.1f} MB)')

# 2. Copy snapshot to Drive
if IN_COLAB:
    shutil.copy2(snap_local, os.path.join(WEIGHTS_DIR, 'results_snapshot.pt'))
    print(f'✓ Snapshot → Drive: {WEIGHTS_DIR}')

# 3. Sync weight files
weight_files = ['cnn_scratch_best.pt', 'cnn_resnet50_best.pt',
                'lstm_best.pt', 'bilstm_best.pt', 'distilbert_best.pt']
print('\nWeight files:')
for wf in weight_files:
    src = os.path.join(str(LOCAL_WEIGHTS), wf)
    if os.path.exists(src):
        mb = os.path.getsize(src) / 1e6
        if IN_COLAB:
            shutil.copy2(src, os.path.join(WEIGHTS_DIR, wf))
            print(f'  ✓ {wf:<35} {mb:.1f} MB → Drive')
        else:
            print(f'  ✓ {wf:<35} {mb:.1f} MB (local)')
    else:
        print(f'  ✗ {wf:<35} MISSING')

# 4. Sync figures
local_pngs = sorted(glob.glob(os.path.join(str(LOCAL_FIGURES), '*.png')))
if IN_COLAB:
    synced = 0
    for src in local_pngs:
        dest = os.path.join(FIGURES_DIR, os.path.basename(src))
        if not os.path.exists(dest) or os.path.getsize(dest) != os.path.getsize(src):
            shutil.copy2(src, dest)
            synced += 1
    print(f'\nFigures: {synced} synced to Drive, {len(local_pngs)-synced} already up-to-date.')
else:
    print(f'\n{len(local_pngs)} figure(s) saved locally.')

# 5. Print final summary table
print('\n' + '=' * 72)
print(f"{'Model':<30} {'Test Acc':>10}  {'Macro F1':>10}  Weight file")
print('-' * 72)
_rows = [
    ('CNN from scratch',         snapshot['cnn_scratch']['test_acc'],  snapshot['cnn_scratch']['macro_f1'],  'cnn_scratch_best.pt'),
    ('ResNet-50 transfer',       snapshot['resnet50']['test_acc'],     snapshot['resnet50']['macro_f1'],     'cnn_resnet50_best.pt'),
    ('LSTM baseline',            snapshot['lstm']['test_acc'],         snapshot['lstm']['macro_f1'],         'lstm_best.pt'),
    ('BiLSTM + Attention',       snapshot['bilstm']['test_acc'],       snapshot['bilstm']['macro_f1'],       'bilstm_best.pt'),
    ('DistilBERT (frozen head)', snapshot['distilbert']['test_acc'],   snapshot['distilbert']['macro_f1'],   'distilbert_best.pt'),
]
for name, acc, f1, wf in _rows:
    _acc = f'{acc*100:.2f}%' if acc is not None else 'n/a'
    _f1  = f'{f1:.4f}'       if f1  is not None else 'n/a'
    print(f'  {name:<28} {_acc:>10}  {_f1:>10}  {wf}')
print('=' * 72)
print('\n✓ Section 12 complete — all results saved.')


## Section 13 — Business Integration: Wine Recommendation Pipeline

This section implements the full end-to-end `recommend()` pipeline that connects all trained models:

```
Food image  ->  CNN (ResNet-50)
                   | food label
            food_flavor_table  ->  classic flavor keywords only
                   | keywords (IDF-weighted, wine-vocab filtered)
                   |
            +------+-------------------------------+
            |                                      |
          W2V rank-1                            W2V top-5 candidates
          cosine sim                       (pick most distant from Safe Bet)
            |                                      |
        Safe Bet grape                       Hidden Gem grape
            |
        grape centroid
        most distant from Safe Bet
            |
        Bold Move grape
            |
        (all three grapes)
            |
        BiLSTM  ->  hidden-state centroid  ->  most representative review (test set)
        df_wine ->  highest-rated bottle for each grape
            |
        Recommendation card  (Safe Bet / Bold Move / Hidden Gem)
```

**Three pairing mechanisms, three different confidence metrics:**

| Pairing | Mechanism | Score |
|---|---|---|
| **Safe Bet** | W2V rank-1: grape whose centroid best matches the food's classic flavor keywords | flavor match % |
| **Bold Move** | Grape centroid most distant from Safe Bet in embedding space (geometry, no keywords) | contrast % |
| **Hidden Gem** | From W2V top-5 flavor matches, pick the one furthest from Safe Bet centroid | flavor match % |

> ### From this point on, only the **test set** is used
>
> All model training is complete. Sections 8, 9, and 11 were trained and evaluated
> exclusively on the train/val splits. Section 13 is the first place where we query
> the BiLSTM on reviews it has **never seen during training** — the held-out test set.
> This ensures that the tasting notes on recommendation cards are genuine model
> predictions on unseen text, not memorised training examples.

| Sub-section | Content |
|---|---|
| **13.1** | `_WINE_VOCAB`, `_IDF`, `embed_keywords()`, `find_grape()`, `find_contrast_grape()` |
| **13.2** | `get_best_review()` — BiLSTM review retrieval (test set) |
| **13.3** | `get_best_wine()` — Vivino rating lookup |
| **13.4** | `recommend()` — full pipeline |
| **13.5** | 20-example recommendation table |


### 13.1 — Flavor embedding helpers

`embed_keywords(kws)` → mean Word2Vec vector for a keyword list.  
`find_grape(kws)` → grape class name with highest cosine similarity to that vector.


In [ ]:
# ── 13.1  Flavor embedding helpers ──────────────────────────────────────────
import math
from numpy.linalg import norm as _norm

# ── Step 1: build wine vocabulary set ──────────────────────────────────
# Only words that appear in BOTH the W2V model AND the wine review corpus are
# useful query terms. Food-texture words (greasy, starchy, baked) exist in the
# Google News base model but drift to meaningless regions after fine-tuning on
# wine text. We filter them out automatically.
import re as _re
_WINE_VOCAB = set()
for _row in df_train['review_text']:
    for _tok in _re.findall(r"[a-z]+", str(_row).lower()):
        if _tok in _wv:
            _WINE_VOCAB.add(_tok)
print(f"✓ wine vocabulary: {len(_WINE_VOCAB):,} in-corpus W2V words")

# ── Step 2: IDF weights from training corpus ────────────────────────────
# Words that appear in every review (good, wine, the) carry no discriminating
# signal. IDF up-weights rare, grape-specific words (cassis, garrigue, petrichor)
# and down-weights ubiquitous ones. Computed once at startup.
from collections import defaultdict as _dd
_doc_freq = _dd(int)
_N_docs   = len(df_train)
for _row in df_train['review_text']:
    for _tok in set(_re.findall(r"[a-z]+", str(_row).lower())):
        _doc_freq[_tok] += 1
_IDF = {w: math.log(_N_docs / (1 + _doc_freq[w])) for w in _WINE_VOCAB}
print(f"✓ IDF weights computed over {_N_docs:,} training reviews")

# ── Step 3: discriminative TF-IDF grape centroids ──────────────────────────
# Problem: grape_embeddings.npy was built by averaging review vectors per
# grape. Because all wine reviews share the same "generic wine language"
# (tannins, acidity, fruit, finish), every centroid lands near the same point
# (pairwise cosine ~0.99). Simple mean-centering spreads them geometrically
# but the centroids still don't capture what's DISTINCTIVE about each grape.
#
# Fix (Option B): rebuild centroids with discriminative TF-IDF weighting.
# Each word's contribution to a grape's centroid is weighted by:
#   disc = (TF_grape / mean_TF_across_grapes) × IDF
# → cassis / Cabernet, petrol / Riesling, garrigue / Grenache get high weight.
# → tannins / all grapes gets near-zero weight.
# Then mean-centering removes any residual shared direction.
#
# _grape_emb_c is used for ALL similarity calculations in Section 13.

from collections import Counter as _Counter

def _make_discriminative_centroids(n_top=80, alpha=2.0):
    """Discriminative TF-IDF weighted grape centroids from training reviews.

    alpha  — exponent on the TF ratio.  alpha=1 is linear (mild discrimination).
             alpha=2 squares it: a word 3x overrepresented in Barbera gets 9x
             the weight vs. 3x with alpha=1.  This sharpens centroids toward
             truly grape-specific vocabulary (petrol→Riesling, garrigue→Grenache).
    n_top  — how many highest-scoring words to use per grape.
             Smaller = more discriminative, less cross-grape contamination.
    """
    grape_tokens = {}
    for g in GRAPE_CLASSES:
        mask  = df_train['grape_class'] == g
        texts = df_train.loc[mask, 'review_text'].dropna().tolist()
        toks  = [t for text in texts
                   for t in _re.findall(r"[a-z]+", text.lower())
                   if t in _WINE_VOCAB]
        grape_tokens[g] = toks

    grape_tf = {}
    for g, toks in grape_tokens.items():
        cnt   = _Counter(toks)
        total = sum(cnt.values()) or 1
        grape_tf[g] = {w: c / total for w, c in cnt.items()}

    all_words = set(w for tf in grape_tf.values() for w in tf)
    avg_tf = {w: sum(grape_tf[g].get(w, 0.0) for g in GRAPE_CLASSES) / len(GRAPE_CLASSES)
              for w in all_words}

    new_emb = np.zeros((len(GRAPE_CLASSES), _wv.vector_size), dtype=np.float32)
    for i, g in enumerate(GRAPE_CLASSES):
        scores = {}
        for w, tf in grape_tf[g].items():
            if w not in _wv:
                continue
            # Squared ratio: only truly grape-specific words dominate
            scores[w] = ((tf / (avg_tf[w] + 1e-9)) ** alpha) * _IDF.get(w, 1.0)
        if not scores:
            new_emb[i] = grape_embeddings[i]
            continue
        top_words = sorted(scores.items(), key=lambda x: -x[1])[:n_top]
        vec, wsum = np.zeros(_wv.vector_size, dtype=np.float32), 0.0
        for w, sc in top_words:
            vec  += sc * _wv[w]
            wsum += sc
        new_emb[i] = vec / (wsum + 1e-8)
    return new_emb

_grape_disc  = _make_discriminative_centroids()
_grape_mean  = _grape_disc.mean(axis=0)
_grape_emb_c = _grape_disc - _grape_mean          # discriminative + centred

import numpy as _np2
_mask      = ~_np2.eye(15, dtype=bool)
_norms_raw = np.linalg.norm(grape_embeddings, axis=1, keepdims=True)
_cos_raw   = (grape_embeddings @ grape_embeddings.T) / (_norms_raw * _norms_raw.T + 1e-8)
_norms_d   = np.linalg.norm(_grape_emb_c, axis=1, keepdims=True)
_cos_d     = (_grape_emb_c @ _grape_emb_c.T)   / (_norms_d   * _norms_d.T   + 1e-8)
print(f"✓ Discriminative + mean-centered grape embeddings ready")
print(f"  Pairwise cosine BEFORE (raw average embeddings) : {_cos_raw[_mask].mean():.4f}")
print(f"  Pairwise cosine AFTER  (disc TF-IDF + centred)  : {_cos_d[_mask].mean():.4f}")
print(f"  (lower = more spread out = more grape variety in recommendations)")

# ── expand_keywords(): food → wine translation layer ─────────────────────────
# STEP 1 — The translator.
#
# Food flavor keywords are written in food language: cheesy, baked, fatty,
# starchy, tomato. Most of these words don't appear in wine reviews, so the
# _WINE_VOCAB filter silently drops them and the query collapses.
#
# Solution: for each food keyword, find its nearest W2V neighbours that ARE
# in the wine vocabulary. The W2V model itself does the translation:
#   cheesy → buttery, creamy, rich, lactic, oaky
#   baked  → toasty, roasted, warm, caramelised, spiced
#   tomato → cherry, raspberry, redcurrant, plum, cassis
#
# The food JSON stays in food language (natural, maintainable).
# This function is the explicit bridge between the two vocabularies.

def expand_keywords(keywords, topn=15, max_total=50):
    """
    Translate food-world keywords into wine-world vocabulary.

    For each keyword (whether in wine vocab or not):
      - Keep it if it is already in _WINE_VOCAB
      - Find its topn nearest W2V neighbours that ARE in _WINE_VOCAB
    Returns a deduplicated list of up to max_total wine-vocab words.

    topn=15 (was 8): wider neighbourhood captures more diverse wine concepts.
    max_total=50 (was 30): richer query vector — different foods reach different
    grape-specific vocabulary regions instead of all converging on the same
    generic fruit/acid cluster.
    """
    seen     = set()
    expanded = []

    for kw in keywords:
        # keep original seed word if already in wine vocab
        if kw in _WINE_VOCAB and kw not in seen:
            expanded.append(kw)
            seen.add(kw)

        # find wine-vocab neighbours (works even if kw not in _WINE_VOCAB —
        # Google News base vocab covers most food words)
        if kw in _wv:
            try:
                # over-fetch then filter to wine vocab
                neighbours = _wv.most_similar(kw, topn=topn * 4)
                for w, _ in neighbours:
                    if w in _WINE_VOCAB and w not in seen:
                        expanded.append(w)
                        seen.add(w)
                        if len(expanded) >= max_total:
                            return expanded
            except KeyError:
                pass

    return expanded


# ── embed_keywords() ───────────────────────────────────────────────────────
# STEP 2 — Updated to call expand_keywords() first.
# Everything else (IDF weighting, food-label anchor) is unchanged.

def embed_keywords(keywords, food_label=None):
    """
    IDF-weighted mean W2V vector for a list of keywords.

    Pipeline:
      1. expand_keywords() — translate food words to wine-vocab neighbours
      2. IDF-weight the expanded set (rare grape words count more)
      3. Weighted mean → 300-d query vector
    Falls back to food label anchor if expansion yields fewer than 2 words.
    Returns None only if nothing at all can be embedded.
    """
    # Step 1: expand food keywords into wine vocabulary
    valid = expand_keywords(keywords)

    # anchor fallback: use food name if expansion yields too few results
    if len(valid) < 2 and food_label and food_label in _wv:
        valid = valid + [food_label]

    if not valid:
        return None

    _vecs    = np.array([_wv[w] for w in valid], dtype=np.float32)
    _weights = np.array([_IDF.get(w, 1.0) for w in valid], dtype=np.float32)
    _weights /= _weights.sum() + 1e-8
    return (_vecs * _weights[:, None]).sum(axis=0)

# ── find_grape(): Safe Bet (rank=1) or Hidden Gem (rank=2) ────────────────
def find_grape(keywords, food_label=None, rank=1):
    """
    Embed keywords and return the Nth-ranked grape by cosine similarity.
      rank=1 -> Safe Bet  (closest match to food flavor profile)
      rank=2 -> candidate pool for Hidden Gem (see recommend())
    Uses mean-centered grape embeddings (_grape_emb_c) for discriminating sims.

    Score: (cosine_similarity + 1) / 2 × 100  maps sim ∈ [-1, 1] → [0, 100]%.
    After discriminative centering, rank-1 sim is typically 0.1–0.6,
    giving display scores of 55–80% that vary meaningfully across different foods.
    Rank-1 is NOT forced to 95% — the score reflects genuine semantic distance.
    """
    q = embed_keywords(keywords, food_label=food_label)
    if q is None:
        return GRAPE_CLASSES[0], 0.0
    sims    = _grape_emb_c @ q / (_norm(_grape_emb_c, axis=1) * _norm(q) + 1e-8)
    ranked  = np.argsort(sims)[::-1]                # descending
    idx     = ranked[min(rank - 1, len(ranked) - 1)]
    display = round((float(sims[idx]) + 1.0) / 2.0 * 100.0, 1)    # [-1,1]→[0,100]
    return GRAPE_CLASSES[idx], display

# ── find_contrast_grape(): Bold Move ──────────────────────────────────
def find_contrast_grape(safe_bet_grape, hidden_gem_grape):
    """
    Return the grape centroid MOST DISTANT from the Safe Bet grape in W2V
    space, excluding grapes already chosen (Safe Bet + Hidden Gem).
    Uses mean-centered embeddings (_grape_emb_c).

    Score: (1 - sim) / 2 * 100  maps sim ∈ [-1, 1] cleanly to [0, 100]%:
      sim =  1.0  (same direction)  →   0% contrast
      sim =  0.0  (orthogonal)      →  50% contrast
      sim = -1.0  (opposite)        → 100% contrast
    Raw (1-sim)*100 breaks after centering because sim can be < -1 numerically.
    """
    sb_idx   = GRAPE_CLASSES.index(safe_bet_grape)
    sb_vec   = _grape_emb_c[sb_idx]
    exclude  = {safe_bet_grape, hidden_gem_grape}

    best_grape, best_sim = None, 2.0   # track raw sim; lower = more contrast
    for i, g in enumerate(GRAPE_CLASSES):
        if g in exclude:
            continue
        sim = float(_grape_emb_c[i] @ sb_vec /
                    (_norm(_grape_emb_c[i]) * _norm(sb_vec) + 1e-8))
        if sim < best_sim:
            best_sim   = sim
            best_grape = g

    contrast_pct = round((1.0 - best_sim) / 2.0 * 100, 1)   # [0, 100]
    return best_grape, contrast_pct

# ── smoke-test: per-word translation + pairing result ────────────────────────
# STEP 3 — Shows the food→wine translation word by word,
# then the final grape recommendations for pizza.
_pizza_kws = food_flavor_table.get('pizza', {}).get('classic', [])

print("✓ Query expansion ready")
print()
print("  Food keyword  →  wine-vocab neighbours")
print("  " + "-" * 52)
for _kw in _pizza_kws:
    if _kw in _wv:
        _nbrs = [w for w, _ in _wv.most_similar(_kw, topn=32) if w in _WINE_VOCAB][:5]
    else:
        _nbrs = []
    _in_wine = "✓" if _kw in _WINE_VOCAB else "✗"
    print(f"  {_in_wine} {_kw:<14}→  {', '.join(_nbrs) if _nbrs else '(no wine-vocab neighbours)'}")
print()

_expanded = expand_keywords(_pizza_kws)
print(f"  Merged expanded query ({len(_expanded)} wine-vocab words):")
print(f"  {_expanded}")
print()

_g1, _p1 = find_grape(_pizza_kws, food_label='pizza', rank=1)
_g2, _p2 = find_grape(_pizza_kws, food_label='pizza', rank=2)
_gc, _pc = find_contrast_grape(_g1, _g2)
print(f"  Safe Bet    : {_g1}  ({_p1}%)")
print(f"  Hidden Gem  : {_g2}  ({_p2}%)")
print(f"  Bold Move   : {_gc}  ({_pc}% contrast)")


### 13.2 — BiLSTM review retrieval

`get_best_review(grape)` samples up to 500 test-set reviews for the **target grape only**,
runs them through the BiLSTM, collects each review's final hidden state (256-d), computes
the **centroid** of those hidden states, and returns the review whose hidden state is
closest to that centroid (cosine similarity).

**Why hidden-state centroid?**

| Approach | Problem |
|---|---|
| argmax softmax on mixed sample | 100% scores — winner-takes-all artefact |
| median softmax on mixed sample | 0.7% scores — drowned out by other grapes in the sample |
| softmax on target-grape-only sample | Still uncalibrated; argmax still inflated |
| **BiLSTM hidden-state centroid** | Picks the review that best represents the *average BiLSTM encoding* of this grape's tasting language — genuinely central, no classifier scoring artifact |

The score shown on the recommendation card is **not** a BiLSTM softmax value:
- **Safe Bet / Hidden Gem**: Word2Vec cosine similarity ×100 (flavor match %)
- **Bold Move**: (1 − cosine similarity to Safe Bet centroid) ×100 (contrast %)


In [ ]:
# ── 13.2  BiLSTM review retrieval (test set, most representative by hidden state) ─
import torch

def get_best_review(grape_name, sample_n=500):
    """
    Return the most representative real Vivino tasting note for grape_name.

    Strategy: sample test reviews for the TARGET GRAPE ONLY, run them through
    the BiLSTM, collect the final hidden state for each, compute the mean
    (centroid), and return the review whose hidden state is closest to that
    centroid (cosine similarity).

    This picks the review that best represents the average BiLSTM encoding
    of this grape's tasting language — genuinely central, not a statistical
    artifact of softmax scoring.
    """
    import numpy as _np
    from numpy.linalg import norm as _lnorm

    # filter test set to target grape
    _grape_mask = df_test['grape_class'] == grape_name
    _df_grape   = df_test[_grape_mask].reset_index(drop=True)
    _X_grape    = X_test[df_test.index[_grape_mask].to_numpy()]

    # if not enough reviews, return the first one
    if len(_df_grape) == 0:
        return df_test.loc[0, 'review_text']

    # sample up to sample_n
    _n   = min(sample_n, len(_df_grape))
    _idx = _np.random.choice(len(_df_grape), size=_n, replace=False)
    _df_s = _df_grape.iloc[_idx].reset_index(drop=True)
    _X_s  = _X_grape[_idx]

    _seqs = torch.tensor(_X_s, dtype=torch.long)
    _lens = (_seqs != 0).sum(dim=1).clamp(min=1)
    _ds   = torch.utils.data.TensorDataset(_seqs, _lens)
    _dl   = torch.utils.data.DataLoader(_ds, batch_size=128, shuffle=False)

    bilstm_model.eval()
    _hiddens = []
    with torch.no_grad():
        for (_batch, _batch_lens) in _dl:
            _batch      = _batch.to(DEVICE)
            _batch_lens = _batch_lens.to(DEVICE)
            # encode() = attention-weighted context vector (no classifier head)
            _h = bilstm_model.encode(_batch, _batch_lens)       # [B, 256]
            _hiddens.append(_h.cpu().numpy())

    _H       = _np.concatenate(_hiddens, axis=0)           # [N, 256]
    _centroid = _H.mean(axis=0)                             # [256]
    # cosine similarity of each review hidden state to the centroid
    _norms  = _lnorm(_H, axis=1, keepdims=True).clip(min=1e-8)
    _cosims = (_H / _norms) @ (_centroid / (_lnorm(_centroid) + 1e-8))
    best_i  = int(_np.argmax(_cosims))
    return _df_s.loc[best_i, 'review_text']

print('✓ get_best_review() ready (target-grape test set, BiLSTM hidden-state centroid)')
_rev = get_best_review(GRAPE_CLASSES[0])
print(f'  test grape : {GRAPE_CLASSES[0]}')
print(f'  review     : {_rev[:140]}...')


### 13.3 — Vivino wine lookup

`get_best_wine(grape)` returns the wine name and Vivino approval percentage for the
highest-rated wine of that grape class in `df_wine`.


In [ ]:
# -- 13.3  Vivino wine lookup -------------------------------------------------

def get_best_wine(grape_name):
    """
    Return (wine_label, rating_pct) for the highest-rated wine of grape_name
    in df_wine_mapped.  rating_pct is already 0-100.
    """
    _sub = df_wine_mapped[
        df_wine_mapped['grape_class'] == grape_name
    ].dropna(subset=['rating_pct'])
    if _sub.empty:
        return 'Unknown wine', 0.0
    best       = _sub.loc[_sub['rating_pct'].idxmax()]
    rating_pct = round(float(best['rating_pct']), 1)
    wine_name  = str(best['wine_label'])
    return wine_name, rating_pct

print('ok get_best_wine() ready')
_wine, _pct = get_best_wine(GRAPE_CLASSES[0])
print(f'  test grape : {GRAPE_CLASSES[0]}')
print(f'  best wine  : {_wine}  ({_pct}%)')


### 13.4 — `recommend()` — full pipeline

Takes a Food-101 class name and returns three wine pairings, one per keyword set:

| Internal key | Card label | Pairing intent |
|---|---|---|
| `classic` | **Safe Bet** | Amplifies the food dominant flavours — grapes with matching character |
| `contrast` | **Bold Move** | Cuts through richness / refreshes — grapes that contrast the dish |
| `safe_bet` | **Hidden Gem** | Crowd-pleasing, approachable — easy-drinking grapes that won't clash |


In [ ]:
# ── 13.4  recommend() — full pipeline ──────────────────────────────────────────

def recommend(food_label):
    """
    Full Wine Peer pipeline. Three intents, three different mechanisms:

    Safe Bet   (classic)  -- W2V rank-1: grape whose tasting vocabulary
                             best matches the food's flavor keywords (IDF-
                             weighted, wine-vocab filtered). Reliable,
                             textbook pairing.

    Bold Move  (contrast) -- Grape centroid most DISTANT from Safe Bet in
                             the 300-d W2V embedding space. Stylistically
                             opposite -- the sommelier surprise. Score is
                             (1 - cosine_sim_to_safe_bet) * 100.

    Hidden Gem (safe_bet) -- W2V top-5 candidates by flavor similarity,
                             then pick the one with the greatest centroid
                             distance from Safe Bet. Real flavor affinity
                             with the food (in the top 5) but not the
                             obvious first pick. The discovery.

    All three grapes are guaranteed distinct by construction.
    """
    food_label = food_label.lower().replace(" ", "_")
    if food_label not in food_flavor_table:
        return {"error": f"'{food_label}' not in food_flavor_table"}

    classic_kws = food_flavor_table[food_label].get("classic", [])

    # ── Safe Bet: rank-1 W2V match ───────────────────────────────────────
    sb_grape, sb_match = find_grape(classic_kws, food_label=food_label, rank=1)

    # ── Hidden Gem: top-5 W2V candidates, most distant from Safe Bet ──────────
    # Genuine flavor affinity (top-5 similarity) but not the obvious pick.
    # Uses mean-centered embeddings (_grape_emb_c) throughout for consistency.
    from numpy.linalg import norm as _n2
    sb_vec   = _grape_emb_c[GRAPE_CLASSES.index(sb_grape)]
    q        = embed_keywords(classic_kws, food_label=food_label)
    if q is not None:
        food_sims = _grape_emb_c @ q / (
            np.linalg.norm(_grape_emb_c, axis=1) * np.linalg.norm(q) + 1e-8)
        top5_idx  = np.argsort(food_sims)[::-1][:5]   # top-5 by food similarity
        # among top-5, pick the one furthest from Safe Bet centroid
        best_hg, best_dist = None, -1.0
        for i in top5_idx:
            g = GRAPE_CLASSES[i]
            if g == sb_grape:
                continue
            dist = 1.0 - float(_grape_emb_c[i] @ sb_vec /
                               (_n2(_grape_emb_c[i]) * _n2(sb_vec) + 1e-8))
            if dist > best_dist:
                best_dist, best_hg = dist, g
        hg_grape     = best_hg or GRAPE_CLASSES[1]
        hg_idx       = GRAPE_CLASSES.index(hg_grape)
        hg_match     = round((float(food_sims[hg_idx]) + 1.0) / 2.0 * 100.0, 1)  # [-1,1]→[0,100]
    else:
        # no embeddable keywords at all: fall back to rank-2
        hg_grape, hg_match = find_grape(classic_kws, food_label=food_label, rank=2)

    # ── Bold Move: grape centroid most distant from Safe Bet ────────────────
    bm_grape, bm_contrast = find_contrast_grape(sb_grape, hg_grape)

    # ── assemble result ───────────────────────────────────────────────────
    result = {"food": food_label, "pairings": {}}
    for intent, grape, score, score_key in [
        ("classic",  sb_grape, sb_match,    "flavor_match"),
        ("contrast", bm_grape, bm_contrast, "contrast_pct"),
        ("safe_bet", hg_grape, hg_match,    "flavor_match"),
    ]:
        wine, rating_pct = get_best_wine(grape)
        review           = get_best_review(grape)
        result["pairings"][intent] = {
            "grape"      : grape,
            "wine"       : wine,
            "rating_pct" : rating_pct,
            "review"     : review[:200],
            score_key    : score,
        }

    return result


# ── quick demo ─────────────────────────────────────────────────────────────
_demo = recommend("pizza")
print(f"Food : {_demo['food']}\n")
_labels = {"classic": "Safe Bet", "contrast": "Bold Move", "safe_bet": "Hidden Gem"}
_score_key = {"classic": "flavor_match", "contrast": "contrast_pct", "safe_bet": "flavor_match"}
_score_lbl = {"classic": "flavor match", "contrast": "contrast",     "safe_bet": "flavor match"}
for intent, data in _demo["pairings"].items():
    sk = _score_key[intent]
    print(f"[{_labels[intent]}]")
    print(f"  Grape  : {data['grape']}")
    print(f"  Score  : {data.get(sk, '?')}%  ({_score_lbl[intent]})")
    print(f"  Wine   : {data['wine']}  ({data['rating_pct']}%)")
    print(f"  Review : {data['review'][:110]}...")
    print()


### 13.5 — 20-example recommendation table

Evaluation across 20 diverse Food-101 classes. Each row shows the food, the three
recommended grapes, and the confidence score for each pairing — demonstrating
the integrated CNN + Word2Vec + BiLSTM pipeline.

**Score semantics by pairing type:**

| Column | Mechanism | Score meaning |
|---|---|---|
| Safe Bet match% | W2V rank-1 cosine sim vs. classic flavor keywords | Higher = better flavor match |
| Bold Move contrast% | 1 − cosine sim to Safe Bet centroid | Higher = more stylistically opposite |
| Hidden Gem match% | W2V flavor sim (top-5 candidate, furthest from Safe Bet) | Higher = stronger flavor affinity |

Bold Move scores are geometrically determined — they reflect the grape embedding space,
not the food's flavor keywords. Safe Bet and Hidden Gem scores are both genuine
cosine similarities to the food's keyword vector; Hidden Gem is always the runner-up
that departs most from the Safe Bet.

> **Rubric §5.6:** *"Present a summary table showing at least 10–20 examples where both
> models' predictions are displayed side-by-side with the combined business recommendation."*


In [ ]:
# ── 13.5  20-example recommendation table ──────────────────────────────────
import pandas as pd

_test_foods = [
    "pizza", "steak", "sushi", "hamburger", "caesar_salad",
    "grilled_salmon", "baby_back_ribs", "lobster_bisque", "chocolate_cake",
    "cheese_plate", "ramen", "tacos", "french_onion_soup", "oysters",
    "pad_thai", "beef_tartare", "apple_pie", "bruschetta", "paella", "hummus"
]

rows = []
for food in _test_foods:
    rec = recommend(food)
    if "error" in rec:
        continue
    p = rec["pairings"]
    rows.append({
        "Food"                    : food.replace("_", " ").title(),
        "Safe Bet grape"          : p["classic"]["grape"],
        "Safe Bet match%"         : p["classic"]["flavor_match"],
        "Bold Move grape"         : p["contrast"]["grape"],
        "Bold Move contrast%"     : p["contrast"]["contrast_pct"],
        "Hidden Gem grape"        : p["safe_bet"]["grape"],
        "Hidden Gem match%"       : p["safe_bet"]["flavor_match"],
    })

df_rec = pd.DataFrame(rows)
print(df_rec.to_string(index=False))
print(f"\n✓ {len(df_rec)} food recommendations generated")
print("\n✓ Section 13 complete — recommend() pipeline ready for deployment (Section 15).")
